# Hash KWS Ensemble — 3 students + smart aggregation

Trains three near-homogeneous HashedNet KWS students that differ only in
pointwise codebook sizes and seed; they share an identical architecture
and a single distilled teacher. After training, the notebook stacks
test logits and runs every aggregator (mean_logits, mean_probs,
conf_weighted, trimmed, majority_vote, temperature_scaled, learned_weights),
plus diagnostics: oracle top-1, pairwise disagreement, per-class
disagreement rate, single/ensemble dispersion.

All artifacts (3 student bundles, 3 firmware exports, ensemble_results.json,
aggregator_params.h) are zipped at the end and copied to Drive.

See `notes/Journal/2026-05-09_hash_ensemble_plan.md` for the locked plan.


In [ ]:
# The next cell writes the minimal runtime files into /content first.
# If Colab is missing packages, run after bootstrap:
# !pip -q install -r /content/diploma_esp32_distributed_nn/code/training/requirements-kws-hash-exact-frontend.txt


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

FORCE_SYNC_RUNTIME_FILES = True
USE_GOOGLE_DRIVE_CACHE = True
CACHE_SPEECHCOMMANDS_ON_DRIVE = False
DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/diploma_kws_cache/hash_ensemble")
ENSEMBLE_VARIANT_NAMES = ["ens_a", "ens_b", "ens_c"]
TEACHER_VARIANT_NAME = "ens_b"  # which recipe owns the teacher checkpoint
SMOKE_MODE = False  # truncates dataset for quick syntax check

if USE_GOOGLE_DRIVE_CACHE and Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped:", exc)
DRIVE_CACHE_ACTIVE = USE_GOOGLE_DRIVE_CACHE and (Path("/content/drive/MyDrive").exists())

BASE_RUNTIME_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = BASE_RUNTIME_DIR / "diploma_esp32_distributed_nn"

TRAINING_ROOT = PROJECT_ROOT / "code" / "training"
SCRIPTS_ROOT = PROJECT_ROOT / "code" / "scripts"
HASHEDNET95_ROOT = TRAINING_ROOT / "hashednet95"
HASH_ENSEMBLE_ROOT = TRAINING_ROOT / "hash_ensemble"
FILE_PAYLOADS = json.loads('{"code/training/requirements-kws-hash.txt": "torch>=2.2\\ntorchaudio>=2.2\\nmatplotlib>=3.8,<4\\n", "code/training/requirements-kws-hash-exact-frontend.txt": "numpy<2\\ntorch>=2.2\\ntorchaudio>=2.2\\nmatplotlib>=3.8,<4\\ntensorflow>=2.15,<3\\n", "code/training/hash_kws_lab/__init__.py": "from .config import ExperimentConfig, make_experiment\\nfrom .recipes import build_recipe_book\\n\\n__all__ = [\\n    \\"ExperimentConfig\\",\\n    \\"build_recipe_book\\",\\n    \\"make_experiment\\",\\n]\\n", "code/training/hash_kws_lab/config.py": "from __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\nVOCABULARY_PRESETS: dict[str, list[str]] = {\\n    \\"kws12\\": [\\n        \\"yes\\",\\n        \\"no\\",\\n        \\"up\\",\\n        \\"down\\",\\n        \\"left\\",\\n        \\"right\\",\\n        \\"on\\",\\n        \\"off\\",\\n        \\"stop\\",\\n        \\"go\\",\\n    ],\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass FeatureConfig:\\n    sample_rate: int = 16_000\\n    clip_samples: int = 16_000\\n    window_ms: int = 30\\n    hop_ms: int = 20\\n    n_mels: int = 40\\n    center: bool = False\\n    frontend_name: str = \\"log_mel\\"\\n    normalize_mode: str = \\"instance\\"\\n    log_offset: float = 1e-6\\n    require_exact_microfrontend: bool = False\\n    lower_band_limit: float = 125.0\\n    upper_band_limit: float = 7_500.0\\n    smoothing_bits: int = 10\\n    even_smoothing: float = 0.025\\n    odd_smoothing: float = 0.06\\n    min_signal_remaining: float = 0.05\\n    enable_pcan: bool = True\\n    pcan_strength: float = 0.95\\n    pcan_offset: float = 80.0\\n    gain_bits: int = 21\\n    enable_log: bool = True\\n    scale_shift: int = 6\\n    exact_value_divisor: float = 665.6\\n    specaugment_prob: float = 0.0\\n    time_mask_max: int = 0\\n    freq_mask_max: int = 0\\n\\n    @property\\n    def window_samples(self) -> int:\\n        return (self.sample_rate * self.window_ms) // 1000\\n\\n    @property\\n    def hop_samples(self) -> int:\\n        return (self.sample_rate * self.hop_ms) // 1000\\n\\n    @property\\n    def frame_count(self) -> int:\\n        usable = max(0, self.clip_samples - self.window_samples)\\n        return 1 + (usable // self.hop_samples)\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int]:\\n        return (self.n_mels, self.frame_count)\\n\\n\\n@dataclass(frozen=True)\\nclass DatasetConfig:\\n    data_dir: str = \\"data\\"\\n    batch_size: int = 256\\n    num_workers: int = 0\\n    persistent_workers: bool = True\\n    prefetch_factor: int = 2\\n    seed: int = 13\\n    unknown_fraction: float = 1.0\\n    silence_fraction: float = 0.12\\n    silence_reference: str = \\"known\\"\\n    train_limit: int | None = None\\n    val_limit: int | None = None\\n    test_limit: int | None = None\\n    time_shift_ms: int = 0\\n    gain_min: float = 1.0\\n    gain_max: float = 1.0\\n    noise_stddev: float = 0.0\\n    cache_features: bool = False\\n    cache_dtype: str = \\"int8\\"\\n\\n\\n@dataclass(frozen=True)\\nclass ModelConfig:\\n    teacher_name: str = \\"\\"\\n    student_name: str = \\"hash_dscnn_deeper\\"\\n    channels: int = 64\\n    teacher_channels: int = 96\\n    num_blocks: int = 4\\n    teacher_num_blocks: int = 5\\n    codebook_size: int = 500\\n    stem_codebook_size: int = 500\\n    depthwise_codebook_size: int = 500\\n    pointwise_codebook_size: int = 500\\n    linear_codebook_size: int = 500\\n    depthwise_codebook_sizes: tuple[int, ...] = ()\\n    pointwise_codebook_sizes: tuple[int, ...] = ()\\n    signed_hash: bool = False\\n    hash_only_pointwise: bool = False\\n    use_residual: bool = False\\n    teacher_dropout: float = 0.10\\n    student_dropout: float = 0.0\\n\\n\\n@dataclass(frozen=True)\\nclass TrainConfig:\\n    seed: int = 13\\n    teacher_epochs: int = 18\\n    student_pretrain_epochs: int = 0\\n    student_epochs: int = 20\\n    student_polish_epochs: int = 0\\n    teacher_lr: float = 1e-3\\n    student_lr: float = 1e-3\\n    polish_lr: float = 2.5e-4\\n    weight_decay: float = 1e-4\\n    optimizer_name: str = \\"adamw\\"\\n    teacher_scheduler_name: str = \\"none\\"\\n    student_scheduler_name: str = \\"none\\"\\n    label_smoothing: float = 0.0\\n    teacher_label_smoothing: float = 0.0\\n    kd_alpha: float = 0.0\\n    kd_alpha_schedule: str = \\"constant\\"\\n    kd_alpha_final: float | None = None\\n    kd_temperature: float = 4.0\\n    kd_temperature_schedule: str = \\"constant\\"\\n    kd_temperature_final: float | None = None\\n    cache_teacher_logits: bool = False\\n    teacher_logits_cache_dir: str = \\"\\"\\n    teacher_logits_cache_dtype: str = \\"float16\\"\\n    teacher_logits_cache_rebuild: bool = False\\n    polish_label_smoothing: float | None = None\\n    grad_clip_norm: float = 0.0\\n    teacher_early_stopping_patience: int = 0\\n    student_early_stopping_patience: int = 0\\n    use_amp: bool = True\\n    use_ema: bool = False\\n    ema_decay: float = 0.999\\n    eval_with_ema: bool = True\\n    top_k: int = 3\\n\\n    @property\\n    def uses_distillation(self) -> bool:\\n        return self.kd_alpha > 0.0\\n\\n\\n@dataclass(frozen=True)\\nclass ExportConfig:\\n    artifacts_dir: str = \\"code/training/hash_artifacts\\"\\n    model_stem: str = \\"hash_kws_student\\"\\n\\n\\n@dataclass(frozen=True)\\nclass ExperimentConfig:\\n    tag: str\\n    vocabulary_preset: str\\n    teacher_reuse_tag: str = \\"\\"\\n    feature: FeatureConfig = field(default_factory=FeatureConfig)\\n    dataset: DatasetConfig = field(default_factory=DatasetConfig)\\n    model: ModelConfig = field(default_factory=ModelConfig)\\n    train: TrainConfig = field(default_factory=TrainConfig)\\n    export: ExportConfig = field(default_factory=ExportConfig)\\n\\n    @property\\n    def commands(self) -> list[str]:\\n        if self.vocabulary_preset not in VOCABULARY_PRESETS:\\n            raise KeyError(f\\"Unknown vocabulary preset: {self.vocabulary_preset}\\")\\n        return list(VOCABULARY_PRESETS[self.vocabulary_preset])\\n\\n    @property\\n    def all_labels(self) -> list[str]:\\n        return [*self.commands, \\"unknown\\", \\"silence\\"]\\n\\n    @property\\n    def label_to_index(self) -> dict[str, int]:\\n        return {label: index for index, label in enumerate(self.all_labels)}\\n\\n    @property\\n    def num_labels(self) -> int:\\n        return len(self.all_labels)\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int]:\\n        return self.feature.feature_shape\\n\\n    @property\\n    def model_input_shape(self) -> tuple[int, int, int]:\\n        mel_bins, frames = self.feature_shape\\n        return (1, mel_bins, frames)\\n\\n    @property\\n    def uses_teacher(self) -> bool:\\n        return bool(self.model.teacher_name)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        payload = asdict(self)\\n        payload[\\"commands\\"] = self.commands\\n        payload[\\"all_labels\\"] = self.all_labels\\n        payload[\\"num_labels\\"] = self.num_labels\\n        payload[\\"feature_shape\\"] = list(self.feature_shape)\\n        payload[\\"model_input_shape\\"] = list(self.model_input_shape)\\n        payload[\\"uses_teacher\\"] = self.uses_teacher\\n        payload[\\"uses_distillation\\"] = self.train.uses_distillation\\n        return payload\\n\\n\\ndef make_experiment(\\n    tag: str = \\"hash_kws12_iterlab_v1\\",\\n    vocabulary_preset: str = \\"kws12\\",\\n) -> ExperimentConfig:\\n    return ExperimentConfig(\\n        tag=tag,\\n        vocabulary_preset=vocabulary_preset,\\n    )\\n\\n\\ndef experiment_from_dict(payload: dict[str, Any]) -> ExperimentConfig:\\n    return ExperimentConfig(\\n        tag=str(payload[\\"tag\\"]),\\n        vocabulary_preset=str(payload[\\"vocabulary_preset\\"]),\\n        teacher_reuse_tag=str(payload.get(\\"teacher_reuse_tag\\", \\"\\")),\\n        feature=FeatureConfig(**payload.get(\\"feature\\", {})),\\n        dataset=DatasetConfig(**payload.get(\\"dataset\\", {})),\\n        model=ModelConfig(**payload.get(\\"model\\", {})),\\n        train=TrainConfig(**payload.get(\\"train\\", {})),\\n        export=ExportConfig(**payload.get(\\"export\\", {})),\\n    )\\n", "code/training/hash_kws_lab/recipes.py": "from __future__ import annotations\\n\\nfrom dataclasses import replace\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef build_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    hash_deeper_usermix_ce = replace(\\n        base,\\n        tag=f\\"{base.tag}_hash_deeper_usermix_ce\\",\\n        dataset=replace(\\n            base.dataset,\\n            unknown_fraction=0.10,\\n            silence_fraction=0.10,\\n            silence_reference=\\"known\\",\\n            time_shift_ms=0,\\n            gain_min=1.0,\\n            gain_max=1.0,\\n            noise_stddev=0.0,\\n        ),\\n        feature=replace(\\n            base.feature,\\n            specaugment_prob=0.0,\\n            time_mask_max=0,\\n            freq_mask_max=0,\\n        ),\\n        model=replace(\\n            base.model,\\n            teacher_name=\\"\\",\\n            student_name=\\"hash_dscnn_deeper\\",\\n            channels=64,\\n            num_blocks=4,\\n            codebook_size=500,\\n            stem_codebook_size=500,\\n            depthwise_codebook_size=500,\\n            pointwise_codebook_size=500,\\n            linear_codebook_size=500,\\n            signed_hash=False,\\n            hash_only_pointwise=False,\\n        ),\\n        train=replace(\\n            base.train,\\n            teacher_epochs=0,\\n            student_pretrain_epochs=0,\\n            student_epochs=20,\\n            student_polish_epochs=0,\\n            student_lr=1e-3,\\n            weight_decay=1e-4,\\n            student_scheduler_name=\\"none\\",\\n            label_smoothing=0.0,\\n            kd_alpha=0.0,\\n            grad_clip_norm=0.0,\\n            student_early_stopping_patience=0,\\n            use_ema=False,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce = replace(\\n        hash_deeper_usermix_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce\\",\\n        dataset=replace(\\n            hash_deeper_usermix_ce.dataset,\\n            unknown_fraction=1.0,\\n            silence_fraction=0.12,\\n            silence_reference=\\"known\\",\\n        ),\\n    )\\n\\n    hash_deeper_fair_augmented = replace(\\n        hash_deeper_fair_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_augmented\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce.dataset,\\n            time_shift_ms=100,\\n            gain_min=0.8,\\n            gain_max=1.2,\\n            noise_stddev=0.004,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_ce.feature,\\n            specaugment_prob=0.35,\\n            time_mask_max=4,\\n            freq_mask_max=3,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce.train,\\n            student_epochs=24,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=0.05,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=6,\\n            use_ema=True,\\n            ema_decay=0.995,\\n            eval_with_ema=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce_exact_microfrontend = replace(\\n        hash_deeper_fair_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce_exact_microfrontend\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce.dataset,\\n            num_workers=2,\\n            cache_features=True,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_ce.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n            specaugment_prob=0.0,\\n            time_mask_max=0,\\n            freq_mask_max=0,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce_exact_microfrontend_tuned = replace(\\n        hash_deeper_fair_ce_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce_exact_microfrontend_tuned\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce_exact_microfrontend.dataset,\\n            batch_size=128,\\n            num_workers=2,\\n            cache_features=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend.train,\\n            student_epochs=28,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=0.02,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=6,\\n            use_ema=False,\\n            eval_with_ema=False,\\n        ),\\n    )\\n\\n    hash_deeper_fair_specaug_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_specaug_exact_microfrontend\\",\\n        feature=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.feature,\\n            specaugment_prob=0.35,\\n            time_mask_max=6,\\n            freq_mask_max=4,\\n        ),\\n    )\\n\\n    hash_deeper_fair_pointwise_budget_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_pointwise_budget_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            stem_codebook_size=192,\\n            depthwise_codebook_size=96,\\n            pointwise_codebook_size=1024,\\n            linear_codebook_size=256,\\n        ),\\n    )\\n\\n    hash_deeper_fair_balanced_budget_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_balanced_budget_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            stem_codebook_size=256,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=864,\\n            linear_codebook_size=520,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_3block_big_pointwise_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_3block_big_pointwise_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            num_blocks=3,\\n            stem_codebook_size=500,\\n            depthwise_codebook_size=500,\\n            pointwise_codebook_size=1000,\\n            linear_codebook_size=384,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_augmented_exact_microfrontend = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_augmented_exact_microfrontend\\",\\n        dataset=replace(\\n            hash_deeper_fair_augmented.dataset,\\n            num_workers=2,\\n            cache_features=False,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_augmented.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed_exact_microfrontend = replace(\\n        hash_deeper_fair_augmented_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_augmented_exact_microfrontend.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed_cached_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed_cached_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_residual_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_residual_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            use_residual=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_residual_specaug_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_residual_specaug_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            use_residual=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_pointwise_only_fair = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_pointwise_only_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            hash_only_pointwise=True,\\n            pointwise_codebook_size=384,\\n            linear_codebook_size=384,\\n        ),\\n    )\\n\\n    dense_teacher_fair = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_dense_teacher_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            teacher_name=\\"\\",\\n            student_name=\\"dense_dscnn_teacher\\",\\n            teacher_channels=96,\\n            teacher_num_blocks=5,\\n            student_dropout=0.10,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_augmented.train,\\n            student_epochs=18,\\n            label_smoothing=0.05,\\n            kd_alpha=0.0,\\n        ),\\n        export=replace(\\n            hash_deeper_fair_augmented.export,\\n            model_stem=\\"dense_hash_teacher\\",\\n        ),\\n    )\\n\\n    hash_deeper_fair_kd = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_kd\\",\\n        teacher_reuse_tag=f\\"{base.tag}_dense_teacher_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            teacher_name=\\"dense_dscnn_teacher\\",\\n            student_name=\\"hash_dscnn_deeper\\",\\n            teacher_channels=96,\\n            teacher_num_blocks=5,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_augmented.train,\\n            teacher_epochs=18,\\n            student_pretrain_epochs=6,\\n            student_epochs=14,\\n            student_polish_epochs=4,\\n            teacher_lr=8e-4,\\n            student_lr=9e-4,\\n            polish_lr=2e-4,\\n            teacher_scheduler_name=\\"cosine\\",\\n            student_scheduler_name=\\"cosine\\",\\n            teacher_label_smoothing=0.05,\\n            label_smoothing=0.02,\\n            kd_alpha=0.60,\\n            kd_temperature=4.0,\\n            teacher_early_stopping_patience=5,\\n            student_early_stopping_patience=6,\\n        ),\\n    )\\n\\n    return {\\n        \\"hash_deeper_usermix_ce\\": hash_deeper_usermix_ce,\\n        \\"hash_deeper_fair_ce\\": hash_deeper_fair_ce,\\n        \\"hash_deeper_fair_augmented\\": hash_deeper_fair_augmented,\\n        \\"hash_deeper_fair_signed\\": hash_deeper_fair_signed,\\n        \\"hash_deeper_fair_ce_exact_microfrontend\\": hash_deeper_fair_ce_exact_microfrontend,\\n        \\"hash_deeper_fair_ce_exact_microfrontend_tuned\\": hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        \\"hash_deeper_fair_specaug_exact_microfrontend\\": hash_deeper_fair_specaug_exact_microfrontend,\\n        \\"hash_deeper_fair_pointwise_budget_exact_microfrontend\\": hash_deeper_fair_pointwise_budget_exact_microfrontend,\\n        \\"hash_deeper_fair_balanced_budget_exact_microfrontend\\": hash_deeper_fair_balanced_budget_exact_microfrontend,\\n        \\"hash_deeper_fair_3block_big_pointwise_exact_microfrontend\\": hash_deeper_fair_3block_big_pointwise_exact_microfrontend,\\n        \\"hash_deeper_fair_augmented_exact_microfrontend\\": hash_deeper_fair_augmented_exact_microfrontend,\\n        \\"hash_deeper_fair_signed_exact_microfrontend\\": hash_deeper_fair_signed_exact_microfrontend,\\n        \\"hash_deeper_fair_signed_cached_exact_microfrontend\\": hash_deeper_fair_signed_cached_exact_microfrontend,\\n        \\"hash_deeper_fair_residual_exact_microfrontend\\": hash_deeper_fair_residual_exact_microfrontend,\\n        \\"hash_deeper_fair_residual_specaug_exact_microfrontend\\": hash_deeper_fair_residual_specaug_exact_microfrontend,\\n        \\"hash_pointwise_only_fair\\": hash_pointwise_only_fair,\\n        \\"dense_teacher_fair\\": dense_teacher_fair,\\n        \\"hash_deeper_fair_kd\\": hash_deeper_fair_kd,\\n    }\\n", "code/training/hash_kws_lab/data.py": "from __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport os\\nimport random\\nfrom collections import Counter\\nfrom dataclasses import asdict\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn.functional as F\\nfrom torch.utils.data import DataLoader, Dataset, Subset\\nfrom tqdm.auto import tqdm\\n\\nfrom .config import ExperimentConfig\\n\\ntry:\\n    import torchaudio\\n    from torchaudio.datasets import SPEECHCOMMANDS\\nexcept ImportError:  # pragma: no cover\\n    torchaudio = None\\n    SPEECHCOMMANDS = None\\n\\ntry:\\n    import tensorflow as tf\\nexcept ImportError:  # pragma: no cover\\n    tf = None\\n\\n\\ndef ensure_torchaudio_available() -> None:\\n    if torchaudio is None or SPEECHCOMMANDS is None:\\n        raise ImportError(\\"torchaudio is required for hash KWS experiments\\")\\n\\n\\ndef ensure_exact_microfrontend_available() -> None:\\n    if tf is None:\\n        raise ImportError(\\n            \\"tensorflow is required for exact microfrontend hash-KWS experiments\\"\\n        )\\n    if _microfrontend_module() is None:\\n        raise ImportError(\\n            \\"tensorflow microfrontend op is unavailable. \\"\\n            \\"Install a TensorFlow build that provides \\"\\n            \\"tensorflow.lite.experimental.microfrontend.python.ops.audio_microfrontend_op.\\"\\n        )\\n\\n\\n@lru_cache(maxsize=1)\\ndef _microfrontend_module():\\n    if tf is None:\\n        return None\\n    try:\\n        from tensorflow.lite.experimental.microfrontend.python.ops import (  # type: ignore\\n            audio_microfrontend_op,\\n        )\\n    except Exception:\\n        return None\\n    return audio_microfrontend_op\\n\\n\\ndef _waveform_to_int16(waveform: torch.Tensor) -> Any:\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n    flattened = waveform.detach().cpu().to(torch.float32).view(-1).clamp_(-1.0, 1.0)\\n    tensor = tf.convert_to_tensor(flattened.numpy(), dtype=tf.float32)\\n    return tf.cast(tf.round(tensor * 32767.0), tf.int16)\\n\\n\\ndef _fit_exact_feature_frames(features: Any, experiment: ExperimentConfig) -> Any:\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n    config = experiment.feature\\n    features = tf.cast(features, tf.float32)\\n    features = features[: config.frame_count]\\n    pad_frames = tf.maximum(0, config.frame_count - tf.shape(features)[0])\\n    features = tf.pad(features, [[0, pad_frames], [0, 0]])\\n    features.set_shape([config.frame_count, config.n_mels])\\n    return features\\n\\n\\ndef _extract_exact_microfrontend_feature_map(\\n    waveform: torch.Tensor,\\n    experiment: ExperimentConfig,\\n) -> torch.Tensor:\\n    ensure_exact_microfrontend_available()\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n\\n    config = experiment.feature\\n    module = _microfrontend_module()\\n    if module is None:\\n        raise ImportError(\\"Exact microfrontend op is unavailable\\")\\n\\n    audio = _waveform_to_int16(waveform)\\n    common_kwargs = {\\n        \\"sample_rate\\": config.sample_rate,\\n        \\"num_channels\\": config.n_mels,\\n        \\"lower_band_limit\\": config.lower_band_limit,\\n        \\"upper_band_limit\\": config.upper_band_limit,\\n        \\"smoothing_bits\\": config.smoothing_bits,\\n        \\"even_smoothing\\": config.even_smoothing,\\n        \\"odd_smoothing\\": config.odd_smoothing,\\n        \\"min_signal_remaining\\": config.min_signal_remaining,\\n        \\"enable_pcan\\": config.enable_pcan,\\n        \\"pcan_strength\\": config.pcan_strength,\\n        \\"pcan_offset\\": config.pcan_offset,\\n        \\"gain_bits\\": config.gain_bits,\\n        \\"enable_log\\": config.enable_log,\\n        \\"scale_shift\\": config.scale_shift,\\n        \\"left_context\\": 0,\\n        \\"right_context\\": 0,\\n        \\"frame_stride\\": 1,\\n        \\"zero_padding\\": False,\\n    }\\n    signature_variants = (\\n        {\\n            \\"window_size\\": config.window_ms,\\n            \\"window_step\\": config.hop_ms,\\n        },\\n        {\\n            \\"window_size_ms\\": config.window_ms,\\n            \\"window_step_ms\\": config.hop_ms,\\n        },\\n    )\\n\\n    features = None\\n    for variant in signature_variants:\\n        try:\\n            features = module.audio_microfrontend(audio, **common_kwargs, **variant)\\n            break\\n        except TypeError:\\n            continue\\n    if features is None:\\n        raise RuntimeError(\\"Failed to invoke the TensorFlow microfrontend op\\")\\n\\n    divisor = max(float(config.exact_value_divisor), 1e-6)\\n    quantized = tf.round((tf.cast(features, tf.float32) * 256.0) / divisor) - 128.0\\n    quantized = tf.clip_by_value(quantized, -128.0, 127.0)\\n    quantized = _fit_exact_feature_frames(quantized, experiment)\\n    tensor = torch.from_numpy(quantized.numpy()).to(torch.float32)\\n    return tensor.transpose(0, 1).unsqueeze(0)\\n\\n\\ndef _cache_signature(experiment: ExperimentConfig, subset: str, training: bool) -> str:\\n    dataset = experiment.dataset\\n    payload = {\\n        \\"version\\": 2,\\n        \\"subset\\": subset,\\n        \\"training\\": training,\\n        \\"vocabulary_preset\\": experiment.vocabulary_preset,\\n        \\"commands\\": experiment.commands,\\n        \\"all_labels\\": experiment.all_labels,\\n        \\"feature\\": asdict(experiment.feature),\\n        \\"dataset_mix_and_waveform\\": {\\n            \\"seed\\": dataset.seed,\\n            \\"unknown_fraction\\": dataset.unknown_fraction,\\n            \\"silence_fraction\\": dataset.silence_fraction,\\n            \\"silence_reference\\": dataset.silence_reference,\\n            \\"time_shift_ms\\": dataset.time_shift_ms,\\n            \\"gain_min\\": dataset.gain_min,\\n            \\"gain_max\\": dataset.gain_max,\\n            \\"noise_stddev\\": dataset.noise_stddev,\\n            \\"cache_dtype\\": dataset.cache_dtype,\\n        },\\n    }\\n    encoded = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode(\\"utf-8\\")\\n    return hashlib.sha1(encoded).hexdigest()[:16]\\n\\n\\nclass AudioFeatureExtractor(torch.nn.Module):\\n    def __init__(self, experiment: ExperimentConfig) -> None:\\n        super().__init__()\\n        ensure_torchaudio_available()\\n        self.config = experiment.feature\\n        self.experiment = experiment\\n        self.mel = None\\n        if self.config.frontend_name == \\"log_mel\\":\\n            self.mel = torchaudio.transforms.MelSpectrogram(\\n                sample_rate=self.config.sample_rate,\\n                n_fft=self.config.window_samples,\\n                win_length=self.config.window_samples,\\n                hop_length=self.config.hop_samples,\\n                n_mels=self.config.n_mels,\\n                center=self.config.center,\\n                power=2.0,\\n            )\\n        elif self.config.frontend_name == \\"exact_microfrontend\\":\\n            if self.config.require_exact_microfrontend:\\n                ensure_exact_microfrontend_available()\\n        else:\\n            raise ValueError(f\\"Unsupported frontend_name: {self.config.frontend_name}\\")\\n\\n    def _normalize(self, spec: torch.Tensor) -> torch.Tensor:\\n        if self.config.normalize_mode == \\"none\\":\\n            return spec\\n        if self.config.normalize_mode == \\"instance\\":\\n            return (spec - spec.mean()) / (spec.std() + 1e-5)\\n        if self.config.normalize_mode == \\"per_frequency\\":\\n            mean = spec.mean(dim=-1, keepdim=True)\\n            std = spec.std(dim=-1, keepdim=True)\\n            return (spec - mean) / (std + 1e-5)\\n        raise ValueError(f\\"Unsupported normalize_mode: {self.config.normalize_mode}\\")\\n\\n    def forward(self, waveform: torch.Tensor) -> torch.Tensor:\\n        waveform = waveform.to(torch.float32)\\n        if self.config.frontend_name == \\"log_mel\\":\\n            if self.mel is None:\\n                raise RuntimeError(\\"log-mel frontend is not initialized\\")\\n            spec = self.mel(waveform)\\n            spec = torch.log(spec + self.config.log_offset)\\n        elif self.config.frontend_name == \\"exact_microfrontend\\":\\n            spec = _extract_exact_microfrontend_feature_map(waveform, self.experiment)\\n        else:\\n            raise ValueError(f\\"Unsupported frontend_name: {self.config.frontend_name}\\")\\n        return self._normalize(spec)\\n\\n\\ndef _apply_spec_augment(feature_map: torch.Tensor, experiment: ExperimentConfig, rng: random.Random) -> torch.Tensor:\\n    config = experiment.feature\\n    if config.specaugment_prob <= 0.0 or rng.random() > config.specaugment_prob:\\n        return feature_map\\n    augmented = feature_map.clone()\\n    if config.time_mask_max > 0:\\n        width = rng.randint(0, config.time_mask_max)\\n        if width > 0 and augmented.shape[-1] > width:\\n            start = rng.randint(0, augmented.shape[-1] - width)\\n            augmented[:, :, start : start + width] = 0.0\\n    if config.freq_mask_max > 0:\\n        height = rng.randint(0, config.freq_mask_max)\\n        if height > 0 and augmented.shape[-2] > height:\\n            start = rng.randint(0, augmented.shape[-2] - height)\\n            augmented[:, start : start + height, :] = 0.0\\n    return augmented\\n\\n\\nclass SpeechCommandsHashDataset(Dataset[tuple[torch.Tensor, int]]):\\n    def __init__(self, root: Path, experiment: ExperimentConfig, subset: str, training: bool) -> None:\\n        ensure_torchaudio_available()\\n        self.experiment = experiment\\n        self.subset = subset\\n        self.training = training\\n        self.sample_rate = experiment.feature.sample_rate\\n        self.clip_samples = experiment.feature.clip_samples\\n        self.feature_extractor = AudioFeatureExtractor(experiment)\\n        self.base = SPEECHCOMMANDS(root=str(root), download=True, subset=subset)\\n        self.base_path = Path(getattr(self.base, \\"_path\\", root))\\n\\n        walker = list(getattr(self.base, \\"_walker\\", []))\\n        if not walker:\\n            walker = list(range(len(self.base)))\\n\\n        known_indices: list[int] = []\\n        unknown_indices: list[int] = []\\n        discovered_labels: Counter[str] = Counter()\\n        wanted_words = set(experiment.commands)\\n\\n        for index, item in enumerate(walker):\\n            label = self._label_from_item(index, item)\\n            discovered_labels[label] += 1\\n            if label in wanted_words:\\n                known_indices.append(index)\\n            elif label != \\"_background_noise_\\":\\n                unknown_indices.append(index)\\n\\n        if not known_indices:\\n            preview = dict(discovered_labels.most_common(20))\\n            raise ValueError(\\n                f\\"No wanted words found for subset={subset!r}. Discovered labels sample={preview}\\"\\n            )\\n\\n        offset = {\\"training\\": 0, \\"validation\\": 1, \\"testing\\": 2}[subset]\\n        self.rng = random.Random(experiment.dataset.seed + offset)\\n        self.rng.shuffle(unknown_indices)\\n\\n        unknown_target = 0\\n        if experiment.dataset.unknown_fraction > 0 and unknown_indices:\\n            unknown_target = max(1, int(round(len(known_indices) * experiment.dataset.unknown_fraction)))\\n            unknown_target = min(len(unknown_indices), unknown_target)\\n        selected_unknown = unknown_indices[:unknown_target]\\n\\n        silence_reference_count = len(known_indices)\\n        if experiment.dataset.silence_reference == \\"mixed\\":\\n            silence_reference_count = len(known_indices) + len(selected_unknown)\\n        silence_target = 0\\n        if experiment.dataset.silence_fraction > 0:\\n            silence_target = max(1, int(round(silence_reference_count * experiment.dataset.silence_fraction)))\\n\\n        self.entries: list[tuple[str, int | None, int]] = []\\n        for index in known_indices:\\n            label = self._label_from_item(index, walker[index] if isinstance(walker[index], str) else index)\\n            self.entries.append((\\"speech\\", index, experiment.label_to_index[label]))\\n        for index in selected_unknown:\\n            self.entries.append((\\"speech\\", index, experiment.label_to_index[\\"unknown\\"]))\\n        for silence_index in range(silence_target):\\n            self.entries.append((\\"silence\\", silence_index, experiment.label_to_index[\\"silence\\"]))\\n\\n        self.rng.shuffle(self.entries)\\n        self.background_noises = self._load_background_noise()\\n        self.cache_enabled = bool(experiment.dataset.cache_features)\\n        self.cache_dtype = experiment.dataset.cache_dtype\\n        self.cached_features: torch.Tensor | None = None\\n        self.cached_labels: torch.Tensor | None = None\\n        self.cache_status: dict[str, Any] | None = None\\n        signature = _cache_signature(experiment, subset=subset, training=training)\\n        feature_cache_root = os.environ.get(\\"HASH_KWS_FEATURE_CACHE_ROOT\\", \\"\\").strip()\\n        cache_root = Path(feature_cache_root) if feature_cache_root else root / \\"hash_feature_cache\\"\\n        self.cache_path = cache_root / f\\"{signature}.pt\\"\\n\\n    def _label_from_item(self, index: int, item: Any) -> str:\\n        if isinstance(item, str):\\n            path = Path(item)\\n            if path.parent.name:\\n                return path.parent.name\\n        _, _, label, *_ = self.base[index]\\n        return str(label)\\n\\n    def _load_background_noise(self) -> list[torch.Tensor]:\\n        noises: list[torch.Tensor] = []\\n        candidate_dirs = [\\n            self.base_path / \\"_background_noise_\\",\\n            self.base_path.parent / \\"_background_noise_\\",\\n        ]\\n        data_root = os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", \\"\\").strip()\\n        if data_root:\\n            candidate_dirs.append(Path(data_root) / \\"_background_noise_\\")\\n\\n        noise_dir = None\\n        for candidate in candidate_dirs:\\n            if candidate.exists():\\n                noise_dir = candidate\\n                break\\n        if noise_dir is None:\\n            return noises\\n\\n        for noise_path in sorted(noise_dir.glob(\\"*.wav\\")):\\n            waveform, sample_rate = torchaudio.load(str(noise_path))\\n            waveform = waveform.mean(dim=0, keepdim=True)\\n            if sample_rate != self.sample_rate:\\n                waveform = torchaudio.functional.resample(waveform, sample_rate, self.sample_rate)\\n            noises.append(waveform)\\n        return noises\\n\\n    def _prepare_waveform(self, waveform: torch.Tensor, sample_rate: int) -> torch.Tensor:\\n        waveform = waveform.mean(dim=0, keepdim=True)\\n        if sample_rate != self.sample_rate:\\n            waveform = torchaudio.functional.resample(waveform, sample_rate, self.sample_rate)\\n        if waveform.shape[1] < self.clip_samples:\\n            waveform = F.pad(waveform, (0, self.clip_samples - waveform.shape[1]))\\n        elif waveform.shape[1] > self.clip_samples:\\n            waveform = waveform[:, : self.clip_samples]\\n        return waveform\\n\\n    def _augment_waveform(self, waveform: torch.Tensor, index: int) -> torch.Tensor:\\n        dataset_cfg = self.experiment.dataset\\n        if dataset_cfg.time_shift_ms > 0:\\n            shift_samples = (self.sample_rate * dataset_cfg.time_shift_ms) // 1000\\n            local_rng = random.Random(dataset_cfg.seed * 1_000_003 + index)\\n            shift = local_rng.randint(-shift_samples, shift_samples)\\n            if shift > 0:\\n                waveform = F.pad(waveform, (shift, 0))[:, : self.clip_samples]\\n            elif shift < 0:\\n                waveform = F.pad(waveform, (0, -shift))[:, -shift : -shift + self.clip_samples]\\n        if dataset_cfg.gain_min != 1.0 or dataset_cfg.gain_max != 1.0:\\n            local_rng = random.Random(dataset_cfg.seed * 1_000_033 + index)\\n            gain = local_rng.uniform(dataset_cfg.gain_min, dataset_cfg.gain_max)\\n            waveform = waveform * gain\\n        if dataset_cfg.noise_stddev > 0.0:\\n            waveform = waveform + torch.randn_like(waveform) * dataset_cfg.noise_stddev\\n        return waveform.clamp_(-1.0, 1.0)\\n\\n    def _make_silence(self, index: int) -> torch.Tensor:\\n        if not self.background_noises:\\n            return torch.zeros(1, self.clip_samples)\\n        noise = self.background_noises[index % len(self.background_noises)]\\n        if noise.shape[1] <= self.clip_samples:\\n            waveform = F.pad(noise, (0, max(self.clip_samples - noise.shape[1], 0)))\\n            return waveform[:, : self.clip_samples] * 0.1\\n        start = (index * 9973) % (noise.shape[1] - self.clip_samples + 1)\\n        return noise[:, start : start + self.clip_samples] * 0.1\\n\\n    def _can_store_int8_cache(self) -> bool:\\n        return (\\n            self.experiment.feature.frontend_name == \\"exact_microfrontend\\"\\n            and self.experiment.feature.normalize_mode == \\"none\\"\\n            and self.cache_dtype == \\"int8\\"\\n        )\\n\\n    def _compute_item(self, index: int) -> tuple[torch.Tensor, int]:\\n        kind, base_index, label_id = self.entries[index]\\n        if kind == \\"speech\\":\\n            waveform, sample_rate, _, *_ = self.base[int(base_index)]\\n            waveform = self._prepare_waveform(waveform, int(sample_rate))\\n        else:\\n            waveform = self._make_silence(index)\\n\\n        if self.training:\\n            waveform = self._augment_waveform(waveform, index=index)\\n\\n        features = self.feature_extractor(waveform)\\n        if self.training:\\n            local_rng = random.Random(self.experiment.dataset.seed * 65_537 + index)\\n            features = _apply_spec_augment(features, self.experiment, local_rng)\\n        return features.squeeze(0), label_id\\n\\n    def _cache_tensor(self, features: torch.Tensor) -> torch.Tensor:\\n        if self._can_store_int8_cache():\\n            return features.to(torch.int8)\\n        return features.to(torch.float32)\\n\\n    def _restore_cached_tensor(self, features: torch.Tensor) -> torch.Tensor:\\n        return features.to(torch.float32)\\n\\n    def materialize_feature_cache(self) -> dict[str, Any]:\\n        if not self.cache_enabled:\\n            self.cache_status = {\\n                \\"enabled\\": False,\\n                \\"status\\": \\"disabled\\",\\n                \\"path\\": str(self.cache_path),\\n            }\\n            return self.cache_status\\n\\n        if self.cached_features is not None and self.cached_labels is not None:\\n            self.cache_status = {\\n                \\"enabled\\": True,\\n                \\"status\\": \\"memory\\",\\n                \\"path\\": str(self.cache_path),\\n                \\"items\\": int(self.cached_labels.numel()),\\n            }\\n            return self.cache_status\\n\\n        if self.cache_path.exists():\\n            payload = torch.load(self.cache_path, map_location=\\"cpu\\")\\n            self.cached_features = payload[\\"features\\"].cpu()\\n            self.cached_labels = payload[\\"labels\\"].cpu()\\n            self.cache_status = {\\n                \\"enabled\\": True,\\n                \\"status\\": \\"loaded\\",\\n                \\"path\\": str(self.cache_path),\\n                \\"items\\": int(self.cached_labels.numel()),\\n                \\"dtype\\": str(self.cached_features.dtype),\\n            }\\n            return self.cache_status\\n\\n        self.cache_path.parent.mkdir(parents=True, exist_ok=True)\\n        cached_features: list[torch.Tensor] = []\\n        cached_labels: list[int] = []\\n        progress = tqdm(\\n            range(len(self.entries)),\\n            desc=f\\"cache {self.subset} {self.experiment.feature.frontend_name}\\",\\n            leave=False,\\n        )\\n        for index in progress:\\n            features, label_id = self._compute_item(index)\\n            cached_features.append(self._cache_tensor(features))\\n            cached_labels.append(int(label_id))\\n\\n        self.cached_features = torch.stack(cached_features, dim=0).cpu()\\n        self.cached_labels = torch.tensor(cached_labels, dtype=torch.int64)\\n        torch.save(\\n            {\\n                \\"features\\": self.cached_features,\\n                \\"labels\\": self.cached_labels,\\n                \\"feature_shape\\": list(self.cached_features.shape),\\n                \\"frontend_name\\": self.experiment.feature.frontend_name,\\n                \\"subset\\": self.subset,\\n                \\"training\\": self.training,\\n            },\\n            self.cache_path,\\n        )\\n        self.cache_status = {\\n            \\"enabled\\": True,\\n            \\"status\\": \\"built\\",\\n            \\"path\\": str(self.cache_path),\\n            \\"items\\": int(self.cached_labels.numel()),\\n            \\"dtype\\": str(self.cached_features.dtype),\\n        }\\n        return self.cache_status\\n\\n    def __len__(self) -> int:\\n        return len(self.entries)\\n\\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:\\n        if self.cached_features is not None and self.cached_labels is not None:\\n            features = self._restore_cached_tensor(self.cached_features[index])\\n            label_id = int(self.cached_labels[index].item())\\n            return features, label_id\\n        return self._compute_item(index)\\n\\n\\ndef describe_dataset(dataset: Dataset[tuple[torch.Tensor, int]], labels: list[str]) -> dict[str, int]:\\n    counts = {label: 0 for label in labels}\\n    source = dataset.dataset if isinstance(dataset, Subset) else dataset\\n    indices = dataset.indices if isinstance(dataset, Subset) else range(len(source))\\n    for index in indices:\\n        _, _, label_id = source.entries[index]\\n        counts[labels[int(label_id)]] += 1\\n    return counts\\n\\n\\ndef maybe_limit(dataset: Dataset[Any], limit: int | None) -> Dataset[Any]:\\n    if limit is None or limit >= len(dataset):\\n        return dataset\\n    if limit <= 0:\\n        raise ValueError(f\\"Dataset limit must be positive, got {limit}\\")\\n    return Subset(dataset, list(range(limit)))\\n\\n\\ndef ensure_non_empty(name: str, dataset: Dataset[Any]) -> None:\\n    if len(dataset) == 0:\\n        raise ValueError(f\\"{name} dataset is empty\\")\\n\\n\\ndef _project_root(start: Path) -> Path:\\n    current = start.resolve()\\n    for candidate in [current, *current.parents]:\\n        if (candidate / \\"code\\").exists() and (candidate / \\"notes\\").exists():\\n            return candidate\\n    return start.resolve()\\n\\n\\ndef prepare_dataloaders(\\n    project_root: Path | None,\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    ensure_torchaudio_available()\\n    if project_root is None:\\n        project_root = _project_root(Path.cwd())\\n\\n    data_root_override = os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", \\"\\").strip()\\n    data_root = Path(data_root_override) if data_root_override else project_root / experiment.dataset.data_dir\\n    data_root.mkdir(parents=True, exist_ok=True)\\n\\n    train_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"training\\", training=True),\\n        experiment.dataset.train_limit,\\n    )\\n    val_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"validation\\", training=False),\\n        experiment.dataset.val_limit,\\n    )\\n    test_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"testing\\", training=False),\\n        experiment.dataset.test_limit,\\n    )\\n\\n    ensure_non_empty(\\"train\\", train_dataset)\\n    ensure_non_empty(\\"validation\\", val_dataset)\\n    ensure_non_empty(\\"test\\", test_dataset)\\n\\n    cache_summary: dict[str, Any] = {}\\n    for split_name, dataset in (\\n        (\\"train\\", train_dataset),\\n        (\\"validation\\", val_dataset),\\n        (\\"test\\", test_dataset),\\n    ):\\n        if isinstance(dataset, Subset):\\n            source = dataset.dataset\\n            cache_summary[split_name] = {\\n                \\"enabled\\": False,\\n                \\"status\\": \\"skipped_for_limited_subset\\",\\n                \\"items\\": len(dataset),\\n                \\"source_items\\": len(source),\\n            }\\n            print(\\n                f\\"Skipping full feature-cache materialization for limited {split_name}: \\"\\n                f\\"{len(dataset)} of {len(source)} items.\\",\\n                flush=True,\\n            )\\n            continue\\n        source = dataset.dataset if isinstance(dataset, Subset) else dataset\\n        if hasattr(source, \\"materialize_feature_cache\\"):\\n            print(\\n                f\\"Preparing feature cache for {split_name}: \\"\\n                f\\"{len(source)} source items -> {source.cache_path}\\",\\n                flush=True,\\n            )\\n            cache_summary[split_name] = source.materialize_feature_cache()\\n\\n    loader_kwargs = {\\n        \\"batch_size\\": experiment.dataset.batch_size,\\n        \\"num_workers\\": experiment.dataset.num_workers,\\n        \\"pin_memory\\": device.type == \\"cuda\\",\\n    }\\n    if experiment.dataset.num_workers > 0:\\n        loader_kwargs[\\"persistent_workers\\"] = experiment.dataset.persistent_workers\\n        loader_kwargs[\\"prefetch_factor\\"] = experiment.dataset.prefetch_factor\\n    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)\\n    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)\\n    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)\\n\\n    feature_preview, _ = next(iter(train_loader))\\n    return {\\n        \\"project_root\\": project_root,\\n        \\"data_root\\": data_root,\\n        \\"loaders\\": {\\n            \\"train\\": train_loader,\\n            \\"validation\\": val_loader,\\n            \\"test\\": test_loader,\\n        },\\n        \\"summary\\": {\\n            \\"train\\": describe_dataset(train_dataset, experiment.all_labels),\\n            \\"validation\\": describe_dataset(val_dataset, experiment.all_labels),\\n            \\"test\\": describe_dataset(test_dataset, experiment.all_labels),\\n        },\\n        \\"cache_summary\\": cache_summary,\\n        \\"feature_preview_shape\\": list(feature_preview.shape),\\n    }\\n", "code/training/hash_kws_lab/models.py": "from __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom .config import ExperimentConfig\\n\\n\\nCONV_HASH_OC = 1337\\nCONV_HASH_IC = 7919\\nCONV_HASH_KH = 2971\\nCONV_HASH_KW = 6151\\nCONV_HASH_LAYER = 104729\\n\\nDW_HASH_CH = 1337\\nDW_HASH_KH = 7919\\nDW_HASH_KW = 2971\\nDW_HASH_LAYER = 104729\\n\\nLINEAR_HASH_A = 1337\\nLINEAR_HASH_B = 7919\\nLINEAR_HASH_C = 2971\\n\\nSIGN_HASH_A = 4099\\nSIGN_HASH_B = 6151\\nSIGN_HASH_C = 14887\\n\\n\\ndef _pair(value: int | tuple[int, int]) -> tuple[int, int]:\\n    if isinstance(value, tuple):\\n        return value\\n    return (value, value)\\n\\n\\ndef _layer_codebook_size(\\n    sizes: tuple[int, ...] | list[int],\\n    index: int,\\n    fallback: int,\\n) -> int:\\n    if index < len(sizes):\\n        selected = int(sizes[index])\\n        if selected <= 0:\\n            raise ValueError(f\\"Codebook size must be positive, got {selected}\\")\\n        return selected\\n    return int(fallback)\\n\\n\\nclass AnalyticHashLinear(nn.Module):\\n    def __init__(\\n        self,\\n        in_dim: int,\\n        out_dim: int,\\n        codebook_size: int,\\n        layer_id: int,\\n        hash_a: int,\\n        hash_b: int,\\n        hash_c: int,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.in_dim = in_dim\\n        self.out_dim = out_dim\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.hash_a = hash_a\\n        self.hash_b = hash_b\\n        self.hash_c = hash_c\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(out_dim))\\n\\n        self.register_buffer(\\"i_idx\\", torch.arange(out_dim, dtype=torch.long).view(out_dim, 1), persistent=False)\\n        self.register_buffer(\\"j_idx\\", torch.arange(in_dim, dtype=torch.long).view(1, in_dim), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (self.i_idx * self.hash_a + self.j_idx * self.hash_b + self.layer_id * self.hash_c)\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (self.i_idx * SIGN_HASH_A + self.j_idx * SIGN_HASH_B + self.layer_id * SIGN_HASH_C) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.out_dim * self.in_dim\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_linear\\",\\n            \\"in_dim\\": self.in_dim,\\n            \\"out_dim\\": self.out_dim,\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.linear(x, self.materialize_weight(), self.bias)\\n\\n\\nclass AnalyticHashConv2d(nn.Module):\\n    def __init__(\\n        self,\\n        in_channels: int,\\n        out_channels: int,\\n        kernel_size: int | tuple[int, int],\\n        codebook_size: int,\\n        layer_id: int,\\n        stride: int | tuple[int, int] = 1,\\n        padding: int | tuple[int, int] = 0,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.in_channels = in_channels\\n        self.out_channels = out_channels\\n        self.kernel_size = _pair(kernel_size)\\n        self.stride = _pair(stride)\\n        self.padding = _pair(padding)\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(out_channels))\\n\\n        self.register_buffer(\\"oc_idx\\", torch.arange(out_channels, dtype=torch.long).view(out_channels, 1, 1, 1), persistent=False)\\n        self.register_buffer(\\"ic_idx\\", torch.arange(in_channels, dtype=torch.long).view(1, in_channels, 1, 1), persistent=False)\\n        self.register_buffer(\\"kh_idx\\", torch.arange(self.kernel_size[0], dtype=torch.long).view(1, 1, self.kernel_size[0], 1), persistent=False)\\n        self.register_buffer(\\"kw_idx\\", torch.arange(self.kernel_size[1], dtype=torch.long).view(1, 1, 1, self.kernel_size[1]), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (\\n                self.oc_idx * CONV_HASH_OC\\n                + self.ic_idx * CONV_HASH_IC\\n                + self.kh_idx * CONV_HASH_KH\\n                + self.kw_idx * CONV_HASH_KW\\n                + self.layer_id * CONV_HASH_LAYER\\n            )\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (\\n            self.oc_idx * SIGN_HASH_A\\n            + self.ic_idx * SIGN_HASH_B\\n            + self.kh_idx * SIGN_HASH_C\\n            + self.kw_idx * (SIGN_HASH_A + SIGN_HASH_B)\\n            + self.layer_id * (SIGN_HASH_C + 11)\\n        ) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.out_channels * self.in_channels * self.kernel_size[0] * self.kernel_size[1]\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_conv2d\\",\\n            \\"in_channels\\": self.in_channels,\\n            \\"out_channels\\": self.out_channels,\\n            \\"kernel_size\\": list(self.kernel_size),\\n            \\"stride\\": list(self.stride),\\n            \\"padding\\": list(self.padding),\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.conv2d(\\n            x,\\n            self.materialize_weight(),\\n            self.bias,\\n            stride=self.stride,\\n            padding=self.padding,\\n        )\\n\\n\\nclass AnalyticHashDepthwiseConv2d(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        kernel_size: int | tuple[int, int],\\n        codebook_size: int,\\n        layer_id: int,\\n        stride: int | tuple[int, int] = 1,\\n        padding: int | tuple[int, int] = 0,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.channels = channels\\n        self.kernel_size = _pair(kernel_size)\\n        self.stride = _pair(stride)\\n        self.padding = _pair(padding)\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(channels))\\n\\n        self.register_buffer(\\"ch_idx\\", torch.arange(channels, dtype=torch.long).view(channels, 1, 1), persistent=False)\\n        self.register_buffer(\\"kh_idx\\", torch.arange(self.kernel_size[0], dtype=torch.long).view(1, self.kernel_size[0], 1), persistent=False)\\n        self.register_buffer(\\"kw_idx\\", torch.arange(self.kernel_size[1], dtype=torch.long).view(1, 1, self.kernel_size[1]), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (\\n                self.ch_idx * DW_HASH_CH\\n                + self.kh_idx * DW_HASH_KH\\n                + self.kw_idx * DW_HASH_KW\\n                + self.layer_id * DW_HASH_LAYER\\n            )\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (\\n            self.ch_idx * SIGN_HASH_A\\n            + self.kh_idx * SIGN_HASH_B\\n            + self.kw_idx * SIGN_HASH_C\\n            + self.layer_id * (SIGN_HASH_A + 29)\\n        ) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight.unsqueeze(1)\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.channels * self.kernel_size[0] * self.kernel_size[1]\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_depthwise_conv2d\\",\\n            \\"channels\\": self.channels,\\n            \\"kernel_size\\": list(self.kernel_size),\\n            \\"stride\\": list(self.stride),\\n            \\"padding\\": list(self.padding),\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.conv2d(\\n            x,\\n            self.materialize_weight(),\\n            self.bias,\\n            stride=self.stride,\\n            padding=self.padding,\\n            groups=self.channels,\\n        )\\n\\n\\nclass DSCNNBlock(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        depthwise: nn.Module,\\n        pointwise: nn.Module,\\n        residual: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.channels = channels\\n        self.depthwise = depthwise\\n        self.bn_dw = nn.BatchNorm2d(channels)\\n        self.pointwise = pointwise\\n        self.bn_pw = nn.BatchNorm2d(channels)\\n        self.residual = residual\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        shortcut = x\\n        x = torch.relu(self.bn_dw(self.depthwise(x)))\\n        x = self.bn_pw(self.pointwise(x))\\n        if self.residual:\\n            x = x + shortcut\\n        x = torch.relu(x)\\n        return x\\n\\n\\nclass HashDSCNN(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        num_blocks: int,\\n        num_classes: int,\\n        stem_codebook_size: int,\\n        depthwise_codebook_size: int,\\n        pointwise_codebook_size: int,\\n        linear_codebook_size: int,\\n        depthwise_codebook_sizes: tuple[int, ...] | list[int],\\n        pointwise_codebook_sizes: tuple[int, ...] | list[int],\\n        signed_hash: bool,\\n        hash_only_pointwise: bool,\\n        use_residual: bool,\\n        dropout: float,\\n    ) -> None:\\n        super().__init__()\\n        layer_id = 0\\n        self.use_residual = use_residual\\n\\n        if hash_only_pointwise:\\n            self.conv0 = nn.Conv2d(1, channels, kernel_size=3, stride=2, padding=1, bias=True)\\n        else:\\n            self.conv0 = AnalyticHashConv2d(\\n                1,\\n                channels,\\n                kernel_size=3,\\n                codebook_size=stem_codebook_size,\\n                layer_id=layer_id,\\n                stride=2,\\n                padding=1,\\n                signed_hash=signed_hash,\\n            )\\n            layer_id += 1\\n        self.bn0 = nn.BatchNorm2d(channels)\\n\\n        blocks: list[DSCNNBlock] = []\\n        for block_index in range(num_blocks):\\n            if hash_only_pointwise:\\n                depthwise = nn.Conv2d(\\n                    channels,\\n                    channels,\\n                    kernel_size=3,\\n                    padding=1,\\n                    groups=channels,\\n                    bias=True,\\n                )\\n            else:\\n                depthwise = AnalyticHashDepthwiseConv2d(\\n                    channels,\\n                    kernel_size=3,\\n                    codebook_size=_layer_codebook_size(\\n                        depthwise_codebook_sizes,\\n                        block_index,\\n                        depthwise_codebook_size,\\n                    ),\\n                    layer_id=layer_id,\\n                    padding=1,\\n                    signed_hash=signed_hash,\\n                )\\n                layer_id += 1\\n\\n            pointwise = AnalyticHashConv2d(\\n                channels,\\n                channels,\\n                kernel_size=1,\\n                codebook_size=_layer_codebook_size(\\n                    pointwise_codebook_sizes,\\n                    block_index,\\n                    pointwise_codebook_size,\\n                ),\\n                layer_id=layer_id,\\n                signed_hash=signed_hash,\\n            )\\n            layer_id += 1\\n            blocks.append(\\n                DSCNNBlock(\\n                    channels=channels,\\n                    depthwise=depthwise,\\n                    pointwise=pointwise,\\n                    residual=use_residual,\\n                )\\n            )\\n\\n        self.blocks = nn.ModuleList(blocks)\\n        self.pool = nn.AdaptiveAvgPool2d(1)\\n        self.dropout = nn.Dropout(dropout)\\n        self.fc = AnalyticHashLinear(\\n            channels,\\n            num_classes,\\n            codebook_size=linear_codebook_size,\\n            layer_id=layer_id,\\n            hash_a=LINEAR_HASH_A,\\n            hash_b=LINEAR_HASH_B,\\n            hash_c=LINEAR_HASH_C,\\n            signed_hash=signed_hash,\\n        )\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        if x.dim() == 3:\\n            x = x.unsqueeze(1)\\n        x = torch.relu(self.bn0(self.conv0(x)))\\n        for block in self.blocks:\\n            x = block(x)\\n        x = self.pool(x).view(x.shape[0], -1)\\n        x = self.dropout(x)\\n        return self.fc(x)\\n\\n\\nclass DenseDSCNN(nn.Module):\\n    def __init__(self, channels: int, num_blocks: int, num_classes: int, dropout: float) -> None:\\n        super().__init__()\\n        self.conv0 = nn.Conv2d(1, channels, kernel_size=3, stride=2, padding=1, bias=False)\\n        self.bn0 = nn.BatchNorm2d(channels)\\n\\n        blocks: list[DSCNNBlock] = []\\n        for _ in range(num_blocks):\\n            depthwise = nn.Conv2d(\\n                channels,\\n                channels,\\n                kernel_size=3,\\n                padding=1,\\n                groups=channels,\\n                bias=False,\\n            )\\n            pointwise = nn.Conv2d(channels, channels, kernel_size=1, bias=False)\\n            blocks.append(DSCNNBlock(channels=channels, depthwise=depthwise, pointwise=pointwise))\\n\\n        self.blocks = nn.ModuleList(blocks)\\n        self.pool = nn.AdaptiveAvgPool2d(1)\\n        self.dropout = nn.Dropout(dropout)\\n        self.fc = nn.Linear(channels, num_classes)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        if x.dim() == 3:\\n            x = x.unsqueeze(1)\\n        x = torch.relu(self.bn0(self.conv0(x)))\\n        for block in self.blocks:\\n            x = block(x)\\n        x = self.pool(x).view(x.shape[0], -1)\\n        x = self.dropout(x)\\n        return self.fc(x)\\n\\n\\ndef build_model_by_name(model_name: str, experiment: ExperimentConfig, role: str) -> nn.Module:\\n    if model_name == \\"hash_dscnn_deeper\\":\\n        return HashDSCNN(\\n            channels=experiment.model.channels,\\n            num_blocks=experiment.model.num_blocks,\\n            num_classes=experiment.num_labels,\\n            stem_codebook_size=experiment.model.stem_codebook_size or experiment.model.codebook_size,\\n            depthwise_codebook_size=experiment.model.depthwise_codebook_size or experiment.model.codebook_size,\\n            pointwise_codebook_size=experiment.model.pointwise_codebook_size or experiment.model.codebook_size,\\n            linear_codebook_size=experiment.model.linear_codebook_size or experiment.model.codebook_size,\\n            depthwise_codebook_sizes=experiment.model.depthwise_codebook_sizes,\\n            pointwise_codebook_sizes=experiment.model.pointwise_codebook_sizes,\\n            signed_hash=experiment.model.signed_hash,\\n            hash_only_pointwise=experiment.model.hash_only_pointwise,\\n            use_residual=experiment.model.use_residual,\\n            dropout=experiment.model.student_dropout,\\n        )\\n    if model_name == \\"dense_dscnn_teacher\\":\\n        channels = experiment.model.teacher_channels if role == \\"teacher\\" else experiment.model.channels\\n        num_blocks = experiment.model.teacher_num_blocks if role == \\"teacher\\" else experiment.model.num_blocks\\n        dropout = experiment.model.teacher_dropout if role == \\"teacher\\" else experiment.model.student_dropout\\n        return DenseDSCNN(channels=channels, num_blocks=num_blocks, num_classes=experiment.num_labels, dropout=dropout)\\n    raise KeyError(f\\"Unknown model name: {model_name}\\")\\n\\n\\ndef build_teacher_model(experiment: ExperimentConfig) -> nn.Module | None:\\n    if not experiment.model.teacher_name:\\n        return None\\n    return build_model_by_name(experiment.model.teacher_name, experiment=experiment, role=\\"teacher\\")\\n\\n\\ndef build_student_model(experiment: ExperimentConfig) -> nn.Module:\\n    return build_model_by_name(experiment.model.student_name, experiment=experiment, role=\\"student\\")\\n\\n\\ndef count_parameters(model: nn.Module) -> int:\\n    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\\n\\n\\ndef count_hash_compact_parameters(model: nn.Module) -> int:\\n    total = 0\\n    for module in model.modules():\\n        if hasattr(module, \\"compact_parameter_count\\"):\\n            total += int(module.compact_parameter_count())\\n    return total\\n\\n\\ndef count_virtual_dense_parameters(model: nn.Module) -> int:\\n    total = 0\\n    for module in model.modules():\\n        if hasattr(module, \\"virtual_parameter_count\\"):\\n            total += int(module.virtual_parameter_count())\\n    return total\\n\\n\\ndef collect_layer_inventory(model: nn.Module) -> list[dict[str, Any]]:\\n    inventory: list[dict[str, Any]] = []\\n    for name, module in model.named_modules():\\n        if name == \\"\\":\\n            continue\\n        if hasattr(module, \\"export_spec\\"):\\n            payload = dict(module.export_spec())\\n            payload[\\"name\\"] = name\\n            payload[\\"compact_parameters\\"] = int(module.compact_parameter_count())\\n            payload[\\"virtual_parameters\\"] = int(module.virtual_parameter_count())\\n            inventory.append(payload)\\n        elif isinstance(module, nn.Conv2d):\\n            inventory.append(\\n                {\\n                    \\"name\\": name,\\n                    \\"type\\": \\"conv2d\\",\\n                    \\"in_channels\\": module.in_channels,\\n                    \\"out_channels\\": module.out_channels,\\n                    \\"kernel_size\\": list(module.kernel_size),\\n                    \\"stride\\": list(module.stride),\\n                    \\"padding\\": list(module.padding),\\n                    \\"groups\\": module.groups,\\n                }\\n            )\\n        elif isinstance(module, nn.Linear):\\n            inventory.append(\\n                {\\n                    \\"name\\": name,\\n                    \\"type\\": \\"linear\\",\\n                    \\"in_features\\": module.in_features,\\n                    \\"out_features\\": module.out_features,\\n                }\\n            )\\n    return inventory\\n\\n\\ndef estimate_maccs(model: nn.Module, input_shape: tuple[int, int, int]) -> int:\\n    total = 0\\n    handles = []\\n\\n    def hook(module: nn.Module, inputs: tuple[torch.Tensor, ...], output: torch.Tensor) -> None:\\n        nonlocal total\\n        if not isinstance(output, torch.Tensor):\\n            return\\n        if isinstance(module, (nn.Conv2d, AnalyticHashConv2d)):\\n            in_tensor = inputs[0]\\n            out_h, out_w = int(output.shape[-2]), int(output.shape[-1])\\n            kh, kw = _pair(module.kernel_size)  # type: ignore[arg-type]\\n            groups = int(getattr(module, \\"groups\\", 1))\\n            out_channels = int(output.shape[1])\\n            in_channels = int(in_tensor.shape[1])\\n            total += out_h * out_w * out_channels * (in_channels // groups) * kh * kw\\n        elif isinstance(module, AnalyticHashDepthwiseConv2d):\\n            out_h, out_w = int(output.shape[-2]), int(output.shape[-1])\\n            kh, kw = module.kernel_size\\n            total += out_h * out_w * module.channels * kh * kw\\n        elif isinstance(module, nn.Linear):\\n            total += int(module.in_features) * int(module.out_features)\\n        elif isinstance(module, AnalyticHashLinear):\\n            total += int(module.in_dim) * int(module.out_dim)\\n\\n    for module in model.modules():\\n        if isinstance(module, (nn.Conv2d, nn.Linear, AnalyticHashConv2d, AnalyticHashDepthwiseConv2d, AnalyticHashLinear)):\\n            handles.append(module.register_forward_hook(hook))\\n\\n    device = next(model.parameters()).device\\n    dummy = torch.zeros((1, *input_shape), dtype=torch.float32, device=device)\\n    was_training = model.training\\n    model.eval()\\n    with torch.no_grad():\\n        model(dummy)\\n    if was_training:\\n        model.train()\\n    for handle in handles:\\n        handle.remove()\\n    return int(total)\\n\\n\\ndef summarize_model(model: nn.Module, experiment: ExperimentConfig) -> dict[str, Any]:\\n    return {\\n        \\"trainable_parameters\\": count_parameters(model),\\n        \\"hash_compact_parameters\\": count_hash_compact_parameters(model),\\n        \\"virtual_dense_parameters\\": count_virtual_dense_parameters(model),\\n        \\"maccs_rough\\": estimate_maccs(model, experiment.model_input_shape),\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n    }\\n", "code/training/hash_kws_lab/trainer.py": "from __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nimport time\\nfrom contextlib import nullcontext\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom torch.utils.data import DataLoader, Dataset\\nfrom tqdm.auto import tqdm\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef _autocast_context(device: torch.device, enabled: bool):\\n    if not enabled or device.type != \\"cuda\\":\\n        return nullcontext()\\n    return torch.autocast(device_type=\\"cuda\\", dtype=torch.float16)\\n\\n\\ndef _build_grad_scaler(device: torch.device, enabled: bool):\\n    scaler_enabled = bool(enabled and device.type == \\"cuda\\")\\n    if hasattr(torch, \\"amp\\") and hasattr(torch.amp, \\"GradScaler\\"):\\n        try:\\n            return torch.amp.GradScaler(device.type, enabled=scaler_enabled)\\n        except TypeError:\\n            return torch.amp.GradScaler(enabled=scaler_enabled)\\n    return torch.cuda.amp.GradScaler(enabled=scaler_enabled)\\n\\n\\nclass ModelEMA:\\n    def __init__(self, model: nn.Module, decay: float) -> None:\\n        self.decay = decay\\n        self.parameter_keys = {key for key, _ in model.named_parameters()}\\n        self.shadow = {key: value.detach().clone() for key, value in model.state_dict().items()}\\n        self.backup: dict[str, torch.Tensor] | None = None\\n\\n    @torch.no_grad()\\n    def update(self, model: nn.Module) -> None:\\n        for key, value in model.state_dict().items():\\n            if key not in self.parameter_keys:\\n                self.shadow[key] = value.detach().clone()\\n                continue\\n            if not torch.is_floating_point(value):\\n                self.shadow[key] = value.detach().clone()\\n                continue\\n            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)\\n\\n    def apply_to(self, model: nn.Module) -> None:\\n        self.backup = {key: value.detach().clone() for key, value in model.state_dict().items()}\\n        model.load_state_dict(self.shadow, strict=True)\\n\\n    def restore(self, model: nn.Module) -> None:\\n        if self.backup is not None:\\n            model.load_state_dict(self.backup, strict=True)\\n            self.backup = None\\n\\n\\ndef build_optimizer(model: nn.Module, lr: float, weight_decay: float, optimizer_name: str) -> torch.optim.Optimizer:\\n    params = [parameter for parameter in model.parameters() if parameter.requires_grad]\\n    if optimizer_name == \\"adam\\":\\n        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)\\n    if optimizer_name == \\"adamw\\":\\n        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)\\n    raise KeyError(f\\"Unknown optimizer_name: {optimizer_name}\\")\\n\\n\\ndef build_scheduler(\\n    optimizer: torch.optim.Optimizer,\\n    scheduler_name: str,\\n    epochs: int,\\n) -> torch.optim.lr_scheduler._LRScheduler | None:\\n    if scheduler_name == \\"none\\":\\n        return None\\n    if scheduler_name == \\"cosine\\":\\n        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))\\n    raise KeyError(f\\"Unknown scheduler_name: {scheduler_name}\\")\\n\\n\\ndef evaluate(\\n    model: nn.Module,\\n    loader: DataLoader[Any],\\n    device: torch.device,\\n    label_smoothing: float = 0.0,\\n    top_k: int = 3,\\n    use_amp: bool = True,\\n    desc: str = \\"eval\\",\\n) -> dict[str, float]:\\n    model.eval()\\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)\\n    total_loss = 0.0\\n    total_correct = 0\\n    total_topk = 0\\n    total_items = 0\\n\\n    progress = tqdm(loader, desc=desc, leave=False)\\n    for batch in progress:\\n        features, targets, _ = _unpack_batch(batch)\\n        features = features.to(device, non_blocking=True)\\n        targets = targets.to(device, non_blocking=True)\\n        with _autocast_context(device, enabled=use_amp):\\n            logits = model(features)\\n            loss = criterion(logits, targets)\\n\\n        batch_size = int(features.shape[0])\\n        total_loss += float(loss.item()) * batch_size\\n        total_correct += int((logits.argmax(dim=1) == targets).sum().item())\\n        topk = logits.topk(min(top_k, logits.shape[1]), dim=1).indices\\n        total_topk += int((topk == targets.unsqueeze(1)).any(dim=1).sum().item())\\n        total_items += batch_size\\n        progress.set_postfix(\\n            loss=f\\"{total_loss / max(total_items, 1):.4f}\\",\\n            acc=f\\"{total_correct / max(total_items, 1):.4f}\\",\\n        )\\n\\n    return {\\n        \\"loss\\": total_loss / max(total_items, 1),\\n        \\"accuracy\\": total_correct / max(total_items, 1),\\n        f\\"top{top_k}_accuracy\\": total_topk / max(total_items, 1),\\n    }\\n\\n\\ndef _cross_entropy(logits: torch.Tensor, targets: torch.Tensor, label_smoothing: float) -> torch.Tensor:\\n    return F.cross_entropy(logits, targets, label_smoothing=label_smoothing)\\n\\n\\ndef _kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, temperature: float) -> torch.Tensor:\\n    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)\\n    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)\\n    return F.kl_div(student_log_probs, teacher_probs, reduction=\\"batchmean\\") * (temperature ** 2)\\n\\n\\ndef _scheduled_value(\\n    start: float,\\n    final: float | None,\\n    schedule_name: str,\\n    epoch: int,\\n    epochs: int,\\n) -> float:\\n    name = schedule_name.strip().lower()\\n    if final is None or name in (\\"\\", \\"none\\", \\"constant\\"):\\n        return float(start)\\n\\n    progress = 0.0 if epochs <= 1 else float(epoch - 1) / float(epochs - 1)\\n    if name in (\\"linear\\", \\"linear_decay\\", \\"linear_ramp\\"):\\n        factor = progress\\n    elif name in (\\"cosine\\", \\"cosine_decay\\"):\\n        factor = 0.5 - (0.5 * math.cos(math.pi * progress))\\n    else:\\n        raise KeyError(f\\"Unknown schedule_name: {schedule_name}\\")\\n    return float(start + ((final - start) * factor))\\n\\n\\ndef _unpack_batch(batch: Any) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:\\n    if not isinstance(batch, (tuple, list)):\\n        raise TypeError(f\\"Expected dataloader batch tuple/list, got {type(batch)!r}\\")\\n    if len(batch) == 2:\\n        features, targets = batch\\n        return features, targets, None\\n    if len(batch) == 3:\\n        features, targets, teacher_logits = batch\\n        return features, targets, teacher_logits\\n    raise ValueError(f\\"Expected 2 or 3 batch items, got {len(batch)}\\")\\n\\n\\ndef _state_dict_fingerprint(model: nn.Module) -> str:\\n    digest = hashlib.sha1()\\n    for key, value in sorted(model.state_dict().items()):\\n        tensor = value.detach().cpu().contiguous()\\n        digest.update(key.encode(\\"utf-8\\"))\\n        digest.update(str(tuple(tensor.shape)).encode(\\"utf-8\\"))\\n        digest.update(str(tensor.dtype).encode(\\"utf-8\\"))\\n        digest.update(tensor.numpy().tobytes())\\n    return digest.hexdigest()[:16]\\n\\n\\ndef _teacher_logits_cache_path(\\n    teacher: nn.Module,\\n    loader: DataLoader[Any],\\n    experiment: ExperimentConfig,\\n    split_name: str,\\n) -> tuple[Path, dict[str, Any]]:\\n    configured = experiment.train.teacher_logits_cache_dir.strip()\\n    env_root = os.environ.get(\\"HASH_KWS_TEACHER_LOGITS_CACHE_ROOT\\", \\"\\").strip()\\n    root = Path(configured or env_root or (Path(experiment.export.artifacts_dir) / \\"teacher_logits\\"))\\n    metadata = {\\n        \\"version\\": 1,\\n        \\"split_name\\": split_name,\\n        \\"items\\": len(loader.dataset),\\n        \\"labels\\": experiment.all_labels,\\n        \\"feature\\": asdict(experiment.feature),\\n        \\"dataset\\": {\\n            \\"seed\\": experiment.dataset.seed,\\n            \\"unknown_fraction\\": experiment.dataset.unknown_fraction,\\n            \\"silence_fraction\\": experiment.dataset.silence_fraction,\\n            \\"silence_reference\\": experiment.dataset.silence_reference,\\n            \\"time_shift_ms\\": experiment.dataset.time_shift_ms,\\n            \\"gain_min\\": experiment.dataset.gain_min,\\n            \\"gain_max\\": experiment.dataset.gain_max,\\n            \\"noise_stddev\\": experiment.dataset.noise_stddev,\\n            \\"train_limit\\": experiment.dataset.train_limit,\\n            \\"val_limit\\": experiment.dataset.val_limit,\\n            \\"test_limit\\": experiment.dataset.test_limit,\\n        },\\n        \\"teacher\\": {\\n            \\"name\\": experiment.model.teacher_name,\\n            \\"channels\\": experiment.model.teacher_channels,\\n            \\"num_blocks\\": experiment.model.teacher_num_blocks,\\n            \\"dropout\\": experiment.model.teacher_dropout,\\n            \\"state_fingerprint\\": _state_dict_fingerprint(teacher),\\n        },\\n    }\\n    signature = hashlib.sha1(\\n        json.dumps(metadata, sort_keys=True, ensure_ascii=False).encode(\\"utf-8\\")\\n    ).hexdigest()[:16]\\n    return root / f\\"{signature}_{split_name}_teacher_logits.pt\\", metadata\\n\\n\\nclass TeacherLogitsDataset(Dataset[tuple[torch.Tensor, int, torch.Tensor]]):\\n    def __init__(self, source: Dataset[Any], logits: torch.Tensor) -> None:\\n        if len(source) != int(logits.shape[0]):\\n            raise ValueError(f\\"Dataset/logit length mismatch: {len(source)} vs {int(logits.shape[0])}\\")\\n        self.source = source\\n        self.logits = logits.cpu()\\n\\n    def __len__(self) -> int:\\n        return len(self.source)\\n\\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, torch.Tensor]:\\n        features, target = self.source[index]\\n        return features, int(target), self.logits[index]\\n\\n\\ndef _loader_like(loader: DataLoader[Any], dataset: Dataset[Any], shuffle: bool) -> DataLoader[Any]:\\n    kwargs: dict[str, Any] = {\\n        \\"batch_size\\": loader.batch_size or 1,\\n        \\"shuffle\\": shuffle,\\n        \\"num_workers\\": loader.num_workers,\\n        \\"pin_memory\\": loader.pin_memory,\\n        \\"drop_last\\": loader.drop_last,\\n        \\"collate_fn\\": loader.collate_fn,\\n    }\\n    if loader.num_workers > 0:\\n        kwargs[\\"persistent_workers\\"] = False\\n        if loader.prefetch_factor is not None:\\n            kwargs[\\"prefetch_factor\\"] = loader.prefetch_factor\\n    return DataLoader(dataset, **kwargs)\\n\\n\\ndef materialize_teacher_logits(\\n    teacher: nn.Module,\\n    loader: DataLoader[Any],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n    split_name: str = \\"train\\",\\n) -> dict[str, Any]:\\n    \\"\\"\\"Precompute teacher logits and return a train loader that serves them.\\n\\n    The cache key includes the exact feature/dataset setup and a fingerprint of\\n    the teacher state, so multiple student recipes can reuse logits only when\\n    the supervising model is actually identical.\\n    \\"\\"\\"\\n\\n    cache_path, metadata = _teacher_logits_cache_path(teacher, loader, experiment, split_name)\\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\\n    dtype_name = experiment.train.teacher_logits_cache_dtype.strip().lower()\\n    stored_dtype = torch.float16 if dtype_name == \\"float16\\" else torch.float32\\n\\n    cache_status = \\"built\\"\\n    logits: torch.Tensor | None = None\\n    if cache_path.exists() and not experiment.train.teacher_logits_cache_rebuild:\\n        payload = torch.load(cache_path, map_location=\\"cpu\\")\\n        cached_metadata = payload.get(\\"metadata\\", {})\\n        cached_logits = payload.get(\\"logits\\")\\n        if (\\n            isinstance(cached_logits, torch.Tensor)\\n            and int(cached_logits.shape[0]) == len(loader.dataset)\\n            and cached_metadata == metadata\\n        ):\\n            logits = cached_logits.cpu()\\n            cache_status = \\"loaded\\"\\n\\n    if logits is None:\\n        was_training = teacher.training\\n        teacher.eval()\\n        teacher.to(device)\\n        sequential_loader = _loader_like(loader, loader.dataset, shuffle=False)\\n        chunks: list[torch.Tensor] = []\\n        progress = tqdm(sequential_loader, desc=f\\"cache {split_name} teacher logits\\", leave=False)\\n        with torch.no_grad():\\n            for batch in progress:\\n                features, _, _ = _unpack_batch(batch)\\n                features = features.to(device, non_blocking=True)\\n                with _autocast_context(device, enabled=experiment.train.use_amp):\\n                    batch_logits = teacher(features)\\n                chunks.append(batch_logits.detach().to(\\"cpu\\", dtype=stored_dtype))\\n        if was_training:\\n            teacher.train()\\n        logits = torch.cat(chunks, dim=0)\\n        torch.save(\\n            {\\n                \\"logits\\": logits,\\n                \\"metadata\\": metadata,\\n                \\"dtype\\": str(logits.dtype),\\n            },\\n            cache_path,\\n        )\\n\\n    cached_dataset = TeacherLogitsDataset(loader.dataset, logits)\\n    cached_loader = _loader_like(loader, cached_dataset, shuffle=True)\\n    return {\\n        \\"loader\\": cached_loader,\\n        \\"path\\": str(cache_path),\\n        \\"status\\": cache_status,\\n        \\"items\\": int(logits.shape[0]),\\n        \\"dtype\\": str(logits.dtype),\\n        \\"metadata\\": metadata,\\n    }\\n\\n\\ndef load_model_checkpoint(\\n    model: nn.Module,\\n    checkpoint_path: str | Path,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    path = Path(checkpoint_path)\\n    payload = torch.load(path, map_location=device)\\n    state_dict: dict[str, torch.Tensor]\\n    if isinstance(payload, dict) and isinstance(payload.get(\\"state_dict\\"), dict):\\n        state_dict = payload[\\"state_dict\\"]\\n    elif isinstance(payload, dict) and isinstance(payload.get(\\"best_state\\"), dict):\\n        state_dict = payload[\\"best_state\\"]\\n    elif isinstance(payload, dict):\\n        state_dict = payload\\n    else:\\n        raise TypeError(f\\"Unsupported checkpoint payload type: {type(payload)!r}\\")\\n    model.load_state_dict(state_dict, strict=True)\\n    return {\\n        \\"path\\": str(path),\\n        \\"state_keys\\": len(state_dict),\\n        \\"payload_keys\\": sorted(payload.keys()) if isinstance(payload, dict) else [],\\n    }\\n\\n\\ndef _train_stage(\\n    model: nn.Module,\\n    train_loader: DataLoader[Any],\\n    val_loader: DataLoader[Any],\\n    device: torch.device,\\n    epochs: int,\\n    stage_name: str,\\n    lr: float,\\n    optimizer_name: str,\\n    scheduler_name: str,\\n    weight_decay: float,\\n    label_smoothing: float,\\n    grad_clip_norm: float,\\n    use_amp: bool,\\n    use_ema: bool,\\n    ema_decay: float,\\n    eval_with_ema: bool,\\n    top_k: int,\\n    early_stopping_patience: int,\\n    teacher: nn.Module | None = None,\\n    kd_alpha: float = 0.0,\\n    kd_alpha_schedule: str = \\"constant\\",\\n    kd_alpha_final: float | None = None,\\n    kd_temperature: float = 4.0,\\n    kd_temperature_schedule: str = \\"constant\\",\\n    kd_temperature_final: float | None = None,\\n) -> dict[str, Any]:\\n    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay, optimizer_name=optimizer_name)\\n    scheduler = build_scheduler(optimizer, scheduler_name=scheduler_name, epochs=epochs)\\n    scaler = _build_grad_scaler(device, enabled=use_amp)\\n    ema = ModelEMA(model, decay=ema_decay) if use_ema else None\\n    history: list[dict[str, float]] = []\\n    best_state = copy.deepcopy(model.state_dict())\\n    best_val_accuracy = -1.0\\n    best_epoch = 0\\n    stale_epochs = 0\\n    started = time.perf_counter()\\n\\n    if teacher is not None:\\n        teacher.eval()\\n        teacher.to(device)\\n\\n    for epoch in range(1, epochs + 1):\\n        model.train()\\n        total_loss = 0.0\\n        total_correct = 0\\n        total_items = 0\\n        epoch_kd_alpha = _scheduled_value(\\n            kd_alpha,\\n            kd_alpha_final,\\n            kd_alpha_schedule,\\n            epoch,\\n            epochs,\\n        )\\n        epoch_kd_temperature = _scheduled_value(\\n            kd_temperature,\\n            kd_temperature_final,\\n            kd_temperature_schedule,\\n            epoch,\\n            epochs,\\n        )\\n\\n        progress = tqdm(train_loader, desc=f\\"{stage_name} | epoch {epoch}/{epochs}\\", leave=True)\\n        for batch in progress:\\n            features, targets, cached_teacher_logits = _unpack_batch(batch)\\n            features = features.to(device, non_blocking=True)\\n            targets = targets.to(device, non_blocking=True)\\n\\n            optimizer.zero_grad(set_to_none=True)\\n            with _autocast_context(device, enabled=use_amp):\\n                logits = model(features)\\n                loss = _cross_entropy(logits, targets, label_smoothing=label_smoothing)\\n                if teacher is not None and epoch_kd_alpha > 0.0:\\n                    if cached_teacher_logits is None:\\n                        with torch.no_grad():\\n                            teacher_logits = teacher(features)\\n                    else:\\n                        teacher_logits = cached_teacher_logits.to(\\n                            device,\\n                            non_blocking=True,\\n                            dtype=logits.dtype,\\n                        )\\n                    loss = (1.0 - epoch_kd_alpha) * loss + epoch_kd_alpha * _kd_loss(\\n                        logits,\\n                        teacher_logits,\\n                        temperature=epoch_kd_temperature,\\n                    )\\n\\n            scaler.scale(loss).backward()\\n            if grad_clip_norm > 0.0:\\n                scaler.unscale_(optimizer)\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)\\n            scaler.step(optimizer)\\n            scaler.update()\\n            if ema is not None:\\n                ema.update(model)\\n\\n            batch_size = int(features.shape[0])\\n            total_loss += float(loss.item()) * batch_size\\n            total_correct += int((logits.argmax(dim=1) == targets).sum().item())\\n            total_items += batch_size\\n            progress.set_postfix(\\n                loss=f\\"{total_loss / max(total_items, 1):.4f}\\",\\n                acc=f\\"{total_correct / max(total_items, 1):.4f}\\",\\n            )\\n\\n        if scheduler is not None:\\n            scheduler.step()\\n\\n        train_metrics = {\\n            \\"loss\\": total_loss / max(total_items, 1),\\n            \\"accuracy\\": total_correct / max(total_items, 1),\\n        }\\n        if ema is not None and eval_with_ema:\\n            ema.apply_to(model)\\n        val_metrics = evaluate(\\n            model,\\n            val_loader,\\n            device=device,\\n            label_smoothing=0.0,\\n            top_k=top_k,\\n            use_amp=use_amp,\\n            desc=f\\"{stage_name} | val {epoch}/{epochs}\\",\\n        )\\n        if ema is not None and eval_with_ema:\\n            ema.restore(model)\\n\\n        history.append(\\n            {\\n                \\"epoch\\": float(epoch),\\n                \\"train_loss\\": train_metrics[\\"loss\\"],\\n                \\"train_accuracy\\": train_metrics[\\"accuracy\\"],\\n                \\"val_loss\\": val_metrics[\\"loss\\"],\\n                \\"val_accuracy\\": val_metrics[\\"accuracy\\"],\\n                \\"lr\\": float(optimizer.param_groups[0][\\"lr\\"]),\\n                \\"kd_alpha\\": float(epoch_kd_alpha if teacher is not None else 0.0),\\n                \\"kd_temperature\\": float(epoch_kd_temperature if teacher is not None else 0.0),\\n            }\\n        )\\n\\n        if val_metrics[\\"accuracy\\"] > best_val_accuracy:\\n            best_val_accuracy = val_metrics[\\"accuracy\\"]\\n            best_epoch = epoch\\n            stale_epochs = 0\\n            best_state = copy.deepcopy(ema.shadow if ema is not None and eval_with_ema else model.state_dict())\\n        else:\\n            stale_epochs += 1\\n\\n        if early_stopping_patience > 0 and stale_epochs >= early_stopping_patience:\\n            break\\n\\n    model.load_state_dict(best_state, strict=True)\\n    return {\\n        \\"history\\": history,\\n        \\"best_state\\": best_state,\\n        \\"best_val_accuracy\\": best_val_accuracy,\\n        \\"best_epoch\\": best_epoch,\\n        \\"elapsed_sec\\": time.perf_counter() - started,\\n    }\\n\\n\\ndef train_teacher(\\n    teacher: nn.Module,\\n    loaders: dict[str, DataLoader[Any]],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    return _train_stage(\\n        model=teacher.to(device),\\n        train_loader=loaders[\\"train\\"],\\n        val_loader=loaders[\\"validation\\"],\\n        device=device,\\n        epochs=experiment.train.teacher_epochs,\\n        stage_name=\\"hash_teacher\\",\\n        lr=experiment.train.teacher_lr,\\n        optimizer_name=experiment.train.optimizer_name,\\n        scheduler_name=experiment.train.teacher_scheduler_name,\\n        weight_decay=experiment.train.weight_decay,\\n        label_smoothing=experiment.train.teacher_label_smoothing,\\n        grad_clip_norm=experiment.train.grad_clip_norm,\\n        use_amp=experiment.train.use_amp,\\n        use_ema=experiment.train.use_ema,\\n        ema_decay=experiment.train.ema_decay,\\n        eval_with_ema=experiment.train.eval_with_ema,\\n        top_k=experiment.train.top_k,\\n        early_stopping_patience=experiment.train.teacher_early_stopping_patience,\\n    )\\n\\n\\ndef train_student(\\n    student: nn.Module,\\n    loaders: dict[str, DataLoader[Any]],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n    teacher: nn.Module | None = None,\\n) -> dict[str, Any]:\\n    student = student.to(device)\\n    full_history: list[dict[str, float]] = []\\n    stage_summaries: list[dict[str, Any]] = []\\n    best_state = copy.deepcopy(student.state_dict())\\n    teacher_logits_cache: dict[str, Any] | None = None\\n\\n    if experiment.train.student_pretrain_epochs > 0:\\n        pretrain = _train_stage(\\n            model=student,\\n            train_loader=loaders[\\"train\\"],\\n            val_loader=loaders[\\"validation\\"],\\n            device=device,\\n            epochs=experiment.train.student_pretrain_epochs,\\n            stage_name=\\"hash_student_pretrain\\",\\n            lr=experiment.train.student_lr,\\n            optimizer_name=experiment.train.optimizer_name,\\n            scheduler_name=experiment.train.student_scheduler_name,\\n            weight_decay=experiment.train.weight_decay,\\n            label_smoothing=experiment.train.label_smoothing,\\n            grad_clip_norm=experiment.train.grad_clip_norm,\\n            use_amp=experiment.train.use_amp,\\n            use_ema=experiment.train.use_ema,\\n            ema_decay=experiment.train.ema_decay,\\n            eval_with_ema=experiment.train.eval_with_ema,\\n            top_k=experiment.train.top_k,\\n            early_stopping_patience=experiment.train.student_early_stopping_patience,\\n        )\\n        best_state = copy.deepcopy(student.state_dict())\\n        full_history.extend(pretrain[\\"history\\"])\\n        stage_summaries.append(\\n            {\\n                \\"stage\\": \\"pretrain\\",\\n                \\"best_val_accuracy\\": pretrain[\\"best_val_accuracy\\"],\\n                \\"best_epoch\\": pretrain[\\"best_epoch\\"],\\n                \\"elapsed_sec\\": pretrain[\\"elapsed_sec\\"],\\n            }\\n        )\\n\\n    main_train_loader = loaders[\\"train\\"]\\n    if (\\n        teacher is not None\\n        and experiment.train.uses_distillation\\n        and experiment.train.cache_teacher_logits\\n    ):\\n        teacher_logits_cache = materialize_teacher_logits(\\n            teacher=teacher,\\n            loader=loaders[\\"train\\"],\\n            experiment=experiment,\\n            device=device,\\n            split_name=\\"train\\",\\n        )\\n        main_train_loader = teacher_logits_cache[\\"loader\\"]\\n\\n    main_train = _train_stage(\\n        model=student,\\n        train_loader=main_train_loader,\\n        val_loader=loaders[\\"validation\\"],\\n        device=device,\\n        epochs=experiment.train.student_epochs,\\n        stage_name=\\"hash_student\\",\\n        lr=experiment.train.student_lr,\\n        optimizer_name=experiment.train.optimizer_name,\\n        scheduler_name=experiment.train.student_scheduler_name,\\n        weight_decay=experiment.train.weight_decay,\\n        label_smoothing=experiment.train.label_smoothing,\\n        grad_clip_norm=experiment.train.grad_clip_norm,\\n        use_amp=experiment.train.use_amp,\\n        use_ema=experiment.train.use_ema,\\n        ema_decay=experiment.train.ema_decay,\\n        eval_with_ema=experiment.train.eval_with_ema,\\n        top_k=experiment.train.top_k,\\n        early_stopping_patience=experiment.train.student_early_stopping_patience,\\n        teacher=teacher,\\n        kd_alpha=experiment.train.kd_alpha,\\n        kd_alpha_schedule=experiment.train.kd_alpha_schedule,\\n        kd_alpha_final=experiment.train.kd_alpha_final,\\n        kd_temperature=experiment.train.kd_temperature,\\n        kd_temperature_schedule=experiment.train.kd_temperature_schedule,\\n        kd_temperature_final=experiment.train.kd_temperature_final,\\n    )\\n    best_state = copy.deepcopy(student.state_dict())\\n    full_history.extend(main_train[\\"history\\"])\\n    stage_summaries.append(\\n        {\\n            \\"stage\\": \\"student\\",\\n            \\"best_val_accuracy\\": main_train[\\"best_val_accuracy\\"],\\n            \\"best_epoch\\": main_train[\\"best_epoch\\"],\\n            \\"elapsed_sec\\": main_train[\\"elapsed_sec\\"],\\n            \\"distillation_enabled\\": bool(teacher is not None and experiment.train.uses_distillation),\\n            \\"teacher_logits_cache\\": {\\n                key: value\\n                for key, value in (teacher_logits_cache or {}).items()\\n                if key not in (\\"loader\\", \\"metadata\\")\\n            },\\n            \\"kd_alpha_schedule\\": experiment.train.kd_alpha_schedule,\\n            \\"kd_alpha_final\\": experiment.train.kd_alpha_final,\\n            \\"kd_temperature_schedule\\": experiment.train.kd_temperature_schedule,\\n            \\"kd_temperature_final\\": experiment.train.kd_temperature_final,\\n        }\\n    )\\n\\n    if experiment.train.student_polish_epochs > 0:\\n        polish_label_smoothing = (\\n            experiment.train.polish_label_smoothing\\n            if experiment.train.polish_label_smoothing is not None\\n            else max(0.0, experiment.train.label_smoothing * 0.5)\\n        )\\n        polish = _train_stage(\\n            model=student,\\n            train_loader=loaders[\\"train\\"],\\n            val_loader=loaders[\\"validation\\"],\\n            device=device,\\n            epochs=experiment.train.student_polish_epochs,\\n            stage_name=\\"hash_student_polish\\",\\n            lr=experiment.train.polish_lr,\\n            optimizer_name=experiment.train.optimizer_name,\\n            scheduler_name=\\"none\\",\\n            weight_decay=experiment.train.weight_decay,\\n            label_smoothing=polish_label_smoothing,\\n            grad_clip_norm=experiment.train.grad_clip_norm,\\n            use_amp=experiment.train.use_amp,\\n            use_ema=experiment.train.use_ema,\\n            ema_decay=experiment.train.ema_decay,\\n            eval_with_ema=experiment.train.eval_with_ema,\\n            top_k=experiment.train.top_k,\\n            early_stopping_patience=max(0, experiment.train.student_early_stopping_patience // 2),\\n        )\\n        best_state = copy.deepcopy(student.state_dict())\\n        full_history.extend(polish[\\"history\\"])\\n        stage_summaries.append(\\n            {\\n                \\"stage\\": \\"polish\\",\\n                \\"best_val_accuracy\\": polish[\\"best_val_accuracy\\"],\\n                \\"best_epoch\\": polish[\\"best_epoch\\"],\\n                \\"elapsed_sec\\": polish[\\"elapsed_sec\\"],\\n                \\"label_smoothing\\": polish_label_smoothing,\\n            }\\n        )\\n\\n    student.load_state_dict(best_state, strict=True)\\n    test_metrics = evaluate(\\n        student,\\n        loaders[\\"test\\"],\\n        device=device,\\n        label_smoothing=0.0,\\n        top_k=experiment.train.top_k,\\n        use_amp=experiment.train.use_amp,\\n        desc=\\"hash_student | test\\",\\n    )\\n    return {\\n        \\"history\\": full_history,\\n        \\"test_metrics\\": test_metrics,\\n        \\"best_state\\": best_state,\\n        \\"stage_summaries\\": stage_summaries,\\n    }\\n", "code/training/hash_kws_lab/reporting.py": "from __future__ import annotations\\n\\nimport json\\nfrom copy import deepcopy\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport matplotlib.pyplot as plt\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef get_run_dir(project_root: Path, experiment: ExperimentConfig) -> Path:\\n    run_dir = project_root / \\"code\\" / \\"training\\" / \\"hash_runs\\" / experiment.tag\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    return run_dir\\n\\n\\ndef _state_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_state.json\\"\\n\\n\\ndef _summary_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_summary.md\\"\\n\\n\\ndef _read_state(run_dir: Path) -> dict[str, Any]:\\n    path = _state_path(run_dir)\\n    if not path.exists():\\n        return {}\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _write_state(run_dir: Path, state: dict[str, Any]) -> None:\\n    _state_path(run_dir).write_text(\\n        json.dumps(state, ensure_ascii=False, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n\\ndef _merge_dict(base: dict[str, Any], update: dict[str, Any]) -> dict[str, Any]:\\n    merged = deepcopy(base)\\n    for key, value in update.items():\\n        if isinstance(value, dict) and isinstance(merged.get(key), dict):\\n            merged[key] = _merge_dict(merged[key], value)\\n        else:\\n            merged[key] = value\\n    return merged\\n\\n\\ndef initialize_run_state(\\n    project_root: Path,\\n    experiment: ExperimentConfig,\\n    recipe_name: str,\\n    dataset_summary: dict[str, Any] | None = None,\\n) -> Path:\\n    run_dir = get_run_dir(project_root, experiment)\\n    state = _merge_dict(\\n        _read_state(run_dir),\\n        {\\n            \\"experiment\\": experiment.to_dict(),\\n            \\"recipe_name\\": recipe_name,\\n            \\"dataset_summary\\": dataset_summary or {},\\n            \\"stages\\": {},\\n            \\"artifacts\\": {},\\n            \\"notes\\": [],\\n        },\\n    )\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n    return run_dir\\n\\n\\ndef save_json_artifact(run_dir: Path, name: str, payload: Any) -> Path:\\n    path = run_dir / name\\n    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_text_artifact(run_dir: Path, name: str, text: str) -> Path:\\n    path = run_dir / name\\n    path.write_text(text, encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_history(run_dir: Path, stage_name: str, history: list[dict[str, float]]) -> Path:\\n    return save_json_artifact(run_dir, f\\"{stage_name}_history.json\\", history)\\n\\n\\ndef save_metrics(run_dir: Path, stage_name: str, metrics: dict[str, Any]) -> Path:\\n    return save_json_artifact(run_dir, f\\"{stage_name}_metrics.json\\", metrics)\\n\\n\\ndef save_model_summary(run_dir: Path, stage_name: str, summary_text: str) -> Path:\\n    return save_text_artifact(run_dir, f\\"{stage_name}_model_summary.txt\\", summary_text)\\n\\n\\ndef save_history_plots(run_dir: Path, stage_name: str, history: list[dict[str, float]]) -> list[str]:\\n    if not history:\\n        return []\\n    keys = history[0].keys()\\n    plot_paths: list[str] = []\\n    groups = {\\n        \\"loss\\": [key for key in keys if \\"loss\\" in key],\\n        \\"metrics\\": [key for key in keys if \\"accuracy\\" in key and key != \\"epoch\\"],\\n        \\"lr\\": [key for key in keys if key == \\"lr\\"],\\n    }\\n    for suffix, selected_keys in groups.items():\\n        if not selected_keys:\\n            continue\\n        epochs = [entry[\\"epoch\\"] for entry in history]\\n        plt.figure(figsize=(10, 4))\\n        for key in selected_keys:\\n            plt.plot(epochs, [entry[key] for entry in history], label=key)\\n        plt.title(f\\"{stage_name} {suffix}\\")\\n        plt.xlabel(\\"Epoch\\")\\n        plt.grid(True, alpha=0.3)\\n        plt.legend()\\n        path = run_dir / f\\"{stage_name}_{suffix}.png\\"\\n        plt.savefig(path, bbox_inches=\\"tight\\")\\n        plt.close()\\n        plot_paths.append(str(path))\\n    return plot_paths\\n\\n\\ndef update_stage_state(\\n    run_dir: Path,\\n    stage_name: str,\\n    metrics: dict[str, Any] | None = None,\\n    history_path: str | None = None,\\n    plot_paths: list[str] | None = None,\\n    summary_path: str | None = None,\\n    extra: dict[str, Any] | None = None,\\n) -> None:\\n    state = _read_state(run_dir)\\n    stage_block = state.setdefault(\\"stages\\", {}).get(stage_name, {})\\n    update = {\\n        \\"metrics\\": metrics or stage_block.get(\\"metrics\\", {}),\\n        \\"history_path\\": history_path or stage_block.get(\\"history_path\\", \\"\\"),\\n        \\"plot_paths\\": plot_paths or stage_block.get(\\"plot_paths\\", []),\\n        \\"summary_path\\": summary_path or stage_block.get(\\"summary_path\\", \\"\\"),\\n    }\\n    if extra:\\n        update[\\"extra\\"] = _merge_dict(stage_block.get(\\"extra\\", {}), extra)\\n    state.setdefault(\\"stages\\", {})[stage_name] = _merge_dict(stage_block, update)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef add_note(run_dir: Path, title: str, body: str) -> None:\\n    state = _read_state(run_dir)\\n    state.setdefault(\\"notes\\", []).append({\\"title\\": title, \\"body\\": body})\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef record_export_artifacts(run_dir: Path, metadata: dict[str, Any]) -> None:\\n    state = _read_state(run_dir)\\n    state[\\"artifacts\\"] = _merge_dict(state.get(\\"artifacts\\", {}), metadata)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef write_run_summary(run_dir: Path) -> Path:\\n    state = _read_state(run_dir)\\n    lines = [\\n        f\\"# Run Summary: {state.get(\'experiment\', {}).get(\'tag\', run_dir.name)}\\",\\n        \\"\\",\\n        f\\"Recipe: `{state.get(\'recipe_name\', \'\')}`\\",\\n        \\"\\",\\n        \\"## Experiment\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"experiment\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Dataset\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"dataset_summary\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Stages\\",\\n        \\"\\",\\n    ]\\n\\n    for stage_name, payload in state.get(\\"stages\\", {}).items():\\n        lines.extend(\\n            [\\n                f\\"### {stage_name}\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(payload.get(\\"metrics\\", {}), ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n        if payload.get(\\"history_path\\"):\\n            lines.append(f\\"History: `{payload[\'history_path\']}`\\")\\n        if payload.get(\\"summary_path\\"):\\n            lines.append(f\\"Model summary: `{payload[\'summary_path\']}`\\")\\n        for plot_path in payload.get(\\"plot_paths\\", []):\\n            lines.append(f\\"Plot: `{plot_path}`\\")\\n        if payload.get(\\"extra\\"):\\n            lines.extend(\\n                [\\n                    \\"Extra:\\",\\n                    \\"```json\\",\\n                    json.dumps(payload[\\"extra\\"], ensure_ascii=False, indent=2),\\n                    \\"```\\",\\n                ]\\n            )\\n        lines.append(\\"\\")\\n\\n    if state.get(\\"artifacts\\"):\\n        lines.extend(\\n            [\\n                \\"## Artifacts\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(state[\\"artifacts\\"], ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n\\n    if state.get(\\"notes\\"):\\n        lines.extend([\\"## Notes\\", \\"\\"])\\n        for note in state[\\"notes\\"]:\\n            lines.append(f\\"### {note.get(\'title\', \'note\')}\\")\\n            lines.append(\\"\\")\\n            lines.append(note.get(\\"body\\", \\"\\"))\\n            lines.append(\\"\\")\\n\\n    summary_path = _summary_path(run_dir)\\n    summary_path.write_text(\\"\\\\n\\".join(lines), encoding=\\"utf-8\\")\\n    return summary_path\\n", "code/training/hash_kws_lab/export.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\nfrom .config import ExperimentConfig\\nfrom .models import collect_layer_inventory, summarize_model\\n\\n\\ndef _cpu_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:\\n    return {\\n        key: value.detach().cpu().clone()\\n        for key, value in model.state_dict().items()\\n    }\\n\\n\\ndef export_model_bundle(\\n    model: nn.Module,\\n    experiment: ExperimentConfig,\\n    stage_name: str = \\"student\\",\\n) -> dict[str, Any]:\\n    artifact_dir = Path(experiment.export.artifacts_dir) / experiment.tag\\n    artifact_dir.mkdir(parents=True, exist_ok=True)\\n\\n    model_stem = f\\"{experiment.export.model_stem}_{stage_name}\\"\\n    bundle_path = artifact_dir / f\\"{model_stem}.pt\\"\\n    metadata_path = artifact_dir / f\\"{model_stem}_metadata.json\\"\\n\\n    model_summary = summarize_model(model, experiment)\\n    bundle = {\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"stage_name\\": stage_name,\\n        \\"model_summary\\": model_summary,\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n        \\"state_dict\\": _cpu_state_dict(model),\\n    }\\n    torch.save(bundle, bundle_path)\\n\\n    metadata = {\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"stage_name\\": stage_name,\\n        \\"bundle_path\\": str(bundle_path),\\n        \\"model_summary\\": model_summary,\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n    }\\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return {\\n        \\"bundle\\": {\\n            \\"path\\": str(bundle_path),\\n            \\"metadata_path\\": str(metadata_path),\\n        },\\n        \\"model_summary\\": model_summary,\\n    }\\n", "code/training/hashednet95/__init__.py": "from .hashednet95_recipes import (\\n    build_hashednet95_recipe_book,\\n    describe_hashednet95_recipe,\\n    with_drive_cache_paths,\\n)\\n\\n__all__ = [\\n    \\"build_hashednet95_recipe_book\\",\\n    \\"describe_hashednet95_recipe\\",\\n    \\"with_drive_cache_paths\\",\\n]\\n", "code/training/hashednet95/hashednet95_recipes.py": "from __future__ import annotations\\n\\nimport os\\nfrom dataclasses import replace\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom hash_kws_lab.config import ExperimentConfig, make_experiment\\nfrom hash_kws_lab.models import build_student_model, summarize_model\\nfrom hash_kws_lab.recipes import build_recipe_book as build_hash_recipe_book\\n\\n\\ndef _tuned_exact_reference(base: ExperimentConfig) -> ExperimentConfig:\\n    return build_hash_recipe_book(base)[\\"hash_deeper_fair_ce_exact_microfrontend_tuned\\"]\\n\\n\\ndef _exact_cached(\\n    experiment: ExperimentConfig,\\n    *,\\n    batch_size: int,\\n    epochs: int,\\n    patience: int,\\n    label_smoothing: float = 0.02,\\n) -> ExperimentConfig:\\n    return replace(\\n        experiment,\\n        dataset=replace(\\n            experiment.dataset,\\n            batch_size=batch_size,\\n            num_workers=2,\\n            cache_features=True,\\n            cache_dtype=\\"int8\\",\\n            time_shift_ms=0,\\n            gain_min=1.0,\\n            gain_max=1.0,\\n            noise_stddev=0.0,\\n        ),\\n        feature=replace(\\n            experiment.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n        ),\\n        train=replace(\\n            experiment.train,\\n            student_epochs=epochs,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=label_smoothing,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=patience,\\n            use_ema=False,\\n            eval_with_ema=False,\\n        ),\\n    )\\n\\n\\ndef build_hashednet95_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    \\"\\"\\"High-upside HashedNet-style recipes for the deploy-aligned KWS branch.\\n\\n    These recipes keep the exact firmware microfrontend and current custom\\n    ESP32 hash-runtime format, while adding the missing HashedNets levers:\\n    signed hashing, per-layer codebook budgets, and virtual-width inflation.\\n    \\"\\"\\"\\n\\n    reference = _tuned_exact_reference(base)\\n\\n    hn95_paper_inflate96_fixedk = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_paper_inflate96_fixedk\\",\\n            model=replace(\\n                reference.model,\\n                channels=96,\\n                num_blocks=4,\\n                stem_codebook_size=500,\\n                depthwise_codebook_size=500,\\n                pointwise_codebook_size=500,\\n                linear_codebook_size=500,\\n                depthwise_codebook_sizes=(),\\n                pointwise_codebook_sizes=(),\\n                signed_hash=True,\\n                use_residual=False,\\n            ),\\n        ),\\n        batch_size=128,\\n        epochs=48,\\n        patience=8,\\n    )\\n\\n    hn95_layerwise96_signed_residual = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_layerwise96_signed_residual\\",\\n            model=replace(\\n                reference.model,\\n                channels=96,\\n                num_blocks=4,\\n                stem_codebook_size=384,\\n                depthwise_codebook_size=192,\\n                pointwise_codebook_size=1024,\\n                linear_codebook_size=384,\\n                depthwise_codebook_sizes=(224, 192, 160, 128),\\n                pointwise_codebook_sizes=(896, 1024, 1152, 1280),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=128,\\n        epochs=64,\\n        patience=10,\\n    )\\n\\n    hn95_layerwise96_signed_cached_specaug = replace(\\n        hn95_layerwise96_signed_residual,\\n        tag=f\\"{base.tag}_hn95_layerwise96_signed_cached_specaug\\",\\n        feature=replace(\\n            hn95_layerwise96_signed_residual.feature,\\n            specaugment_prob=0.25,\\n            time_mask_max=4,\\n            freq_mask_max=3,\\n        ),\\n        train=replace(\\n            hn95_layerwise96_signed_residual.train,\\n            label_smoothing=0.03,\\n        ),\\n    )\\n\\n    hn95_inflate128_fixed_storage_paper = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_inflate128_fixed_storage_paper\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=4,\\n                stem_codebook_size=500,\\n                depthwise_codebook_size=500,\\n                pointwise_codebook_size=500,\\n                linear_codebook_size=500,\\n                depthwise_codebook_sizes=(),\\n                pointwise_codebook_sizes=(),\\n                signed_hash=True,\\n                use_residual=False,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=64,\\n        patience=10,\\n    )\\n\\n    hn95_inflate128_layerwise_signed_residual = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_inflate128_layerwise_signed_residual\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=4,\\n                stem_codebook_size=512,\\n                depthwise_codebook_size=224,\\n                pointwise_codebook_size=1536,\\n                linear_codebook_size=512,\\n                depthwise_codebook_sizes=(288, 256, 224, 192),\\n                pointwise_codebook_sizes=(1280, 1536, 1792, 2048),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=80,\\n        patience=12,\\n    )\\n\\n    hn95_3block128_big_pointwise_signed = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_3block128_big_pointwise_signed\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=3,\\n                stem_codebook_size=512,\\n                depthwise_codebook_size=256,\\n                pointwise_codebook_size=2048,\\n                linear_codebook_size=512,\\n                depthwise_codebook_sizes=(288, 256, 224),\\n                pointwise_codebook_sizes=(1536, 2048, 2304),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=80,\\n        patience=12,\\n    )\\n\\n    hn95_kd128_layerwise_signed_residual = replace(\\n        hn95_inflate128_layerwise_signed_residual,\\n        tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        model=replace(\\n            hn95_inflate128_layerwise_signed_residual.model,\\n            teacher_name=\\"dense_dscnn_teacher\\",\\n            teacher_channels=128,\\n            teacher_num_blocks=6,\\n            teacher_dropout=0.05,\\n        ),\\n        train=replace(\\n            hn95_inflate128_layerwise_signed_residual.train,\\n            teacher_epochs=48,\\n            student_pretrain_epochs=6,\\n            student_epochs=40,\\n            student_polish_epochs=4,\\n            teacher_lr=8e-4,\\n            student_lr=8e-4,\\n            polish_lr=2e-4,\\n            teacher_scheduler_name=\\"cosine\\",\\n            student_scheduler_name=\\"cosine\\",\\n            teacher_label_smoothing=0.02,\\n            label_smoothing=0.02,\\n            kd_alpha=0.55,\\n            kd_temperature=4.5,\\n            teacher_early_stopping_patience=8,\\n            student_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule = replace(\\n        hn95_kd128_layerwise_signed_residual,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        train=replace(\\n            hn95_kd128_layerwise_signed_residual.train,\\n            student_epochs=52,\\n            student_polish_epochs=8,\\n            label_smoothing=0.025,\\n            polish_label_smoothing=0.0,\\n            kd_alpha=0.65,\\n            kd_alpha_schedule=\\"cosine\\",\\n            kd_alpha_final=0.35,\\n            kd_temperature=5.5,\\n            kd_temperature_schedule=\\"cosine\\",\\n            kd_temperature_final=3.0,\\n            cache_teacher_logits=True,\\n            teacher_logits_cache_dtype=\\"float16\\",\\n            student_early_stopping_patience=12,\\n        ),\\n    )\\n\\n    hn95_kd128_rich_teacher160_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_rich_teacher160_cached_schedule\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_teacher160_dense_reusable\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            teacher_channels=160,\\n            teacher_num_blocks=7,\\n            teacher_dropout=0.06,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            teacher_epochs=72,\\n            teacher_lr=6e-4,\\n            teacher_label_smoothing=0.035,\\n            teacher_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd112_latency_balanced_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd112_latency_balanced_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            channels=112,\\n            stem_codebook_size=448,\\n            depthwise_codebook_size=224,\\n            pointwise_codebook_size=1280,\\n            linear_codebook_size=448,\\n            depthwise_codebook_sizes=(256, 224, 192, 160),\\n            pointwise_codebook_sizes=(1024, 1280, 1536, 1792),\\n        ),\\n        dataset=replace(\\n            hn95_kd128_cached_schedule.dataset,\\n            batch_size=112,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=50,\\n            student_polish_epochs=6,\\n        ),\\n    )\\n\\n    hn95_kd96_latency_guard_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd96_latency_guard_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            channels=96,\\n            stem_codebook_size=384,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=1024,\\n            linear_codebook_size=384,\\n            depthwise_codebook_sizes=(224, 192, 160, 128),\\n            pointwise_codebook_sizes=(896, 1024, 1152, 1280),\\n        ),\\n        dataset=replace(\\n            hn95_kd128_cached_schedule.dataset,\\n            batch_size=128,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=48,\\n            student_polish_epochs=6,\\n            student_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd128_pw2048_budget_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_pw2048_budget_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            stem_codebook_size=448,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=2048,\\n            linear_codebook_size=640,\\n            depthwise_codebook_sizes=(256, 224, 192, 160),\\n            pointwise_codebook_sizes=(1536, 1792, 2048, 2304),\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=56,\\n            student_polish_epochs=8,\\n        ),\\n    )\\n\\n    hn95_kd128_hard_polish_cached = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_hard_polish_cached\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            label_smoothing=0.015,\\n            student_epochs=50,\\n            student_polish_epochs=10,\\n            polish_lr=1.2e-4,\\n            polish_label_smoothing=0.0,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule_s29 = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule_s29\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            seed=29,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule_s47 = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule_s47\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            seed=47,\\n        ),\\n    )\\n\\n    return {\\n        \\"hn95_paper_inflate96_fixedk\\": hn95_paper_inflate96_fixedk,\\n        \\"hn95_layerwise96_signed_residual\\": hn95_layerwise96_signed_residual,\\n        \\"hn95_layerwise96_signed_cached_specaug\\": hn95_layerwise96_signed_cached_specaug,\\n        \\"hn95_inflate128_fixed_storage_paper\\": hn95_inflate128_fixed_storage_paper,\\n        \\"hn95_inflate128_layerwise_signed_residual\\": hn95_inflate128_layerwise_signed_residual,\\n        \\"hn95_3block128_big_pointwise_signed\\": hn95_3block128_big_pointwise_signed,\\n        \\"hn95_kd128_layerwise_signed_residual\\": hn95_kd128_layerwise_signed_residual,\\n        \\"hn95_kd128_cached_schedule\\": hn95_kd128_cached_schedule,\\n        \\"hn95_kd128_rich_teacher160_cached_schedule\\": hn95_kd128_rich_teacher160_cached_schedule,\\n        \\"hn95_kd112_latency_balanced_cached_schedule\\": hn95_kd112_latency_balanced_cached_schedule,\\n        \\"hn95_kd96_latency_guard_cached_schedule\\": hn95_kd96_latency_guard_cached_schedule,\\n        \\"hn95_kd128_pw2048_budget_cached_schedule\\": hn95_kd128_pw2048_budget_cached_schedule,\\n        \\"hn95_kd128_hard_polish_cached\\": hn95_kd128_hard_polish_cached,\\n        \\"hn95_kd128_cached_schedule_s29\\": hn95_kd128_cached_schedule_s29,\\n        \\"hn95_kd128_cached_schedule_s47\\": hn95_kd128_cached_schedule_s47,\\n    }\\n\\n\\ndef with_drive_cache_paths(\\n    experiment: ExperimentConfig,\\n    drive_cache_root: str | Path,\\n) -> ExperimentConfig:\\n    \\"\\"\\"Persist Speech Commands, exact-feature cache, and exports on Drive.\\"\\"\\"\\n\\n    root = Path(drive_cache_root)\\n    data_root = Path(os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", root / \\"speechcommands_v2\\"))\\n    feature_cache_root = root / \\"hash_feature_cache\\"\\n    teacher_logits_cache_root = root / \\"teacher_logits_cache\\"\\n    artifact_root = root / \\"hash_artifacts\\"\\n    data_root.mkdir(parents=True, exist_ok=True)\\n    feature_cache_root.mkdir(parents=True, exist_ok=True)\\n    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)\\n    artifact_root.mkdir(parents=True, exist_ok=True)\\n    os.environ.setdefault(\\"SPEECHCOMMANDS_DATA_ROOT\\", str(data_root))\\n    os.environ.setdefault(\\"HASH_KWS_FEATURE_CACHE_ROOT\\", str(feature_cache_root))\\n    os.environ.setdefault(\\"HASH_KWS_TEACHER_LOGITS_CACHE_ROOT\\", str(teacher_logits_cache_root))\\n    return replace(\\n        experiment,\\n        train=replace(\\n            experiment.train,\\n            teacher_logits_cache_dir=str(teacher_logits_cache_root),\\n        ),\\n        export=replace(\\n            experiment.export,\\n            artifacts_dir=str(artifact_root),\\n        ),\\n    )\\n\\n\\ndef describe_hashednet95_recipe(experiment: ExperimentConfig) -> dict[str, Any]:\\n    student = build_student_model(experiment)\\n    summary = summarize_model(student, experiment)\\n    reference = _tuned_exact_reference(\\n        make_experiment(tag=\\"hash_kws12_iterlab_v1\\", vocabulary_preset=experiment.vocabulary_preset)\\n    )\\n    reference_summary = summarize_model(build_student_model(reference), reference)\\n    virtual = int(summary[\\"virtual_dense_parameters\\"])\\n    compact = int(summary[\\"hash_compact_parameters\\"])\\n    ref_virtual = int(reference_summary[\\"virtual_dense_parameters\\"])\\n    ref_compact = int(reference_summary[\\"hash_compact_parameters\\"])\\n    return {\\n        \\"tag\\": experiment.tag,\\n        \\"labels\\": experiment.all_labels,\\n        \\"model_input_shape\\": experiment.model_input_shape,\\n        \\"channels\\": experiment.model.channels,\\n        \\"num_blocks\\": experiment.model.num_blocks,\\n        \\"signed_hash\\": experiment.model.signed_hash,\\n        \\"use_residual\\": experiment.model.use_residual,\\n        \\"train_seed\\": experiment.train.seed,\\n        \\"teacher_reuse_tag\\": experiment.teacher_reuse_tag,\\n        \\"teacher_name\\": experiment.model.teacher_name,\\n        \\"teacher_channels\\": experiment.model.teacher_channels,\\n        \\"teacher_num_blocks\\": experiment.model.teacher_num_blocks,\\n        \\"teacher_dropout\\": experiment.model.teacher_dropout,\\n        \\"kd_alpha\\": experiment.train.kd_alpha,\\n        \\"kd_alpha_schedule\\": experiment.train.kd_alpha_schedule,\\n        \\"kd_alpha_final\\": experiment.train.kd_alpha_final,\\n        \\"kd_temperature\\": experiment.train.kd_temperature,\\n        \\"kd_temperature_schedule\\": experiment.train.kd_temperature_schedule,\\n        \\"kd_temperature_final\\": experiment.train.kd_temperature_final,\\n        \\"cache_teacher_logits\\": experiment.train.cache_teacher_logits,\\n        \\"student_polish_epochs\\": experiment.train.student_polish_epochs,\\n        \\"polish_label_smoothing\\": experiment.train.polish_label_smoothing,\\n        \\"depthwise_codebook_sizes\\": list(experiment.model.depthwise_codebook_sizes),\\n        \\"pointwise_codebook_sizes\\": list(experiment.model.pointwise_codebook_sizes),\\n        \\"student_summary\\": summary,\\n        \\"reference_summary\\": reference_summary,\\n        \\"virtual_inflation_vs_reference\\": virtual / max(ref_virtual, 1),\\n        \\"compact_growth_vs_reference\\": compact / max(ref_compact, 1),\\n        \\"virtual_per_compact_parameter\\": virtual / max(compact, 1),\\n        \\"reference_virtual_per_compact_parameter\\": ref_virtual / max(ref_compact, 1),\\n    }\\n", "code/training/hash_ensemble/__init__.py": "", "code/training/hash_ensemble/ensemble_recipes.py": "from __future__ import annotations\\n\\nfrom dataclasses import replace\\nfrom typing import Any\\n\\nfrom hash_kws_lab.config import ExperimentConfig\\nfrom hashednet95.hashednet95_recipes import build_hashednet95_recipe_book\\n\\n\\n# ---------------------------------------------------------------------------\\n# Three near-homo variants of the baseline `hn95_kd128_layerwise_signed_residual`\\n# Differ only in:\\n#   - pointwise codebook sizes (slight perturbation around baseline)\\n#   - train.seed (init + dataloader shuffle)\\n# Architecture (channels, num_blocks, signed_hash, residual, stem/dw/linear codebooks)\\n# is held identical so all three share the same firmware runtime path and so the\\n# teacher logits cache (keyed on feature+dataset+teacher fingerprint) is reused\\n# across all three student runs.\\n# ---------------------------------------------------------------------------\\n\\n\\n_VARIANT_DEFINITIONS: tuple[tuple[str, int, tuple[int, int, int, int]], ...] = (\\n    # name        seed   pointwise_codebook_sizes (per block)\\n    (\\"ens_a\\",     13,    (1024, 1280, 1536, 1792)),  # -20% PW vs baseline\\n    (\\"ens_b\\",     29,    (1280, 1536, 1792, 2048)),  # baseline (= hn95_kd128_layerwise_signed_residual)\\n    (\\"ens_c\\",     47,    (1536, 1792, 2048, 2304)),  # +13% PW vs baseline\\n)\\n\\nENSEMBLE_TEACHER_VARIANT = \\"ens_b\\"\\n\\n\\ndef variant_tag(base_tag: str, name: str, seed: int) -> str:\\n    return f\\"{base_tag}_hn95_{name}_s{seed}\\"\\n\\n\\ndef build_ensemble_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    \\"\\"\\"Three near-homo HashedNet KWS recipes for the distributed ensemble track.\\"\\"\\"\\n\\n    hn95_book = build_hashednet95_recipe_book(base)\\n    baseline = hn95_book[\\"hn95_kd128_layerwise_signed_residual\\"]\\n\\n    # Cache teacher logits so the second/third student reuse them.\\n    baseline_cached = replace(\\n        baseline,\\n        train=replace(\\n            baseline.train,\\n            cache_teacher_logits=True,\\n            teacher_logits_cache_dtype=\\"float16\\",\\n        ),\\n    )\\n\\n    teacher_anchor_tag = variant_tag(\\n        base.tag,\\n        ENSEMBLE_TEACHER_VARIANT,\\n        dict((name, seed) for name, seed, _ in _VARIANT_DEFINITIONS)[ENSEMBLE_TEACHER_VARIANT],\\n    )\\n\\n    recipes: dict[str, ExperimentConfig] = {}\\n    for name, seed, pointwise_codebook_sizes in _VARIANT_DEFINITIONS:\\n        tag = variant_tag(base.tag, name, seed)\\n        recipes[name] = replace(\\n            baseline_cached,\\n            tag=tag,\\n            # Anchor every student to the teacher trained for ens_b. This is the\\n            # value used in run summaries; the actual checkpoint reuse goes\\n            # through TEACHER_CHECKPOINT_PATH / FORCE_TEACHER_RETRAIN at runtime.\\n            teacher_reuse_tag=teacher_anchor_tag,\\n            model=replace(\\n                baseline_cached.model,\\n                pointwise_codebook_sizes=pointwise_codebook_sizes,\\n            ),\\n            train=replace(\\n                baseline_cached.train,\\n                seed=seed,\\n            ),\\n        )\\n    return recipes\\n\\n\\ndef describe_variant(experiment: ExperimentConfig) -> dict[str, Any]:\\n    return {\\n        \\"tag\\": experiment.tag,\\n        \\"seed\\": experiment.train.seed,\\n        \\"channels\\": experiment.model.channels,\\n        \\"num_blocks\\": experiment.model.num_blocks,\\n        \\"stem_codebook_size\\": experiment.model.stem_codebook_size,\\n        \\"depthwise_codebook_sizes\\": list(experiment.model.depthwise_codebook_sizes),\\n        \\"pointwise_codebook_sizes\\": list(experiment.model.pointwise_codebook_sizes),\\n        \\"linear_codebook_size\\": experiment.model.linear_codebook_size,\\n        \\"signed_hash\\": experiment.model.signed_hash,\\n        \\"use_residual\\": experiment.model.use_residual,\\n        \\"kd_alpha\\": experiment.train.kd_alpha,\\n        \\"kd_temperature\\": experiment.train.kd_temperature,\\n        \\"student_pretrain_epochs\\": experiment.train.student_pretrain_epochs,\\n        \\"student_epochs\\": experiment.train.student_epochs,\\n        \\"student_polish_epochs\\": experiment.train.student_polish_epochs,\\n        \\"cache_teacher_logits\\": experiment.train.cache_teacher_logits,\\n        \\"teacher_reuse_tag\\": experiment.teacher_reuse_tag,\\n    }\\n", "code/training/hash_ensemble/aggregation.py": "\\"\\"\\"Pure-numpy aggregation utilities for the 3-model hash KWS ensemble.\\n\\nAll functions operate on stacked logits with shape ``[N, B, C]`` where\\n``N`` is the number of ensemble members, ``B`` is the batch size and ``C``\\nis the class count. Outputs are ``[B, C]`` (logits or probabilities) or\\n``[B]`` (predicted labels) depending on the aggregator.\\n\\nThe two new-in-v2 aggregators (``temperature_scaled_mean`` and\\n``learned_weights_mean``) accept a tiny set of fitted parameters that fit\\ninto a few floats and can be hard-coded into the firmware aggregator.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\n\\n\\n# ---------------------------------------------------------------------------\\n# Numerical helpers\\n# ---------------------------------------------------------------------------\\n\\n\\ndef _check_logits(logits: np.ndarray) -> None:\\n    if logits.ndim != 3:\\n        raise ValueError(\\n            f\\"Expected logits with shape [N, B, C], got shape {tuple(logits.shape)}\\"\\n        )\\n\\n\\ndef _softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:\\n    shifted = x - np.max(x, axis=axis, keepdims=True)\\n    exp = np.exp(shifted)\\n    return exp / np.sum(exp, axis=axis, keepdims=True)\\n\\n\\ndef _log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:\\n    shifted = x - np.max(x, axis=axis, keepdims=True)\\n    return shifted - np.log(np.sum(np.exp(shifted), axis=axis, keepdims=True))\\n\\n\\ndef _entropy_per_row(probs: np.ndarray, axis: int = -1, eps: float = 1e-12) -> np.ndarray:\\n    return -np.sum(probs * np.log(probs + eps), axis=axis)\\n\\n\\ndef _normalize_weights(weights: np.ndarray) -> np.ndarray:\\n    weights = np.asarray(weights, dtype=np.float64)\\n    total = float(weights.sum())\\n    if total <= 0.0:\\n        raise ValueError(f\\"Weights must sum to a positive number, got {total}\\")\\n    return weights / total\\n\\n\\n# ---------------------------------------------------------------------------\\n# Basic aggregators (the ones the research already tested)\\n# ---------------------------------------------------------------------------\\n\\n\\ndef mean_logits(logits: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Headline aggregator from the research (`A4 best on KWS = mean_logits`).\\"\\"\\"\\n\\n    _check_logits(logits)\\n    return logits.mean(axis=0)\\n\\n\\ndef mean_probs(logits: np.ndarray) -> np.ndarray:\\n    _check_logits(logits)\\n    probs = _softmax(logits, axis=-1)\\n    return probs.mean(axis=0)\\n\\n\\ndef conf_weighted(logits: np.ndarray, eps: float = 1e-12) -> np.ndarray:\\n    \\"\\"\\"Per-input confidence-weighted mean of softmax probs.\\n\\n    Weight per (b, k) is ``1 / H(softmax(logits_k[b]))``. Refuted by the\\n    research as a system-level upgrade, kept for parity with §5 of NOTES.md.\\n    \\"\\"\\"\\n\\n    _check_logits(logits)\\n    probs = _softmax(logits, axis=-1)\\n    entropy = _entropy_per_row(probs, axis=-1)  # [N, B]\\n    weights = 1.0 / (entropy + eps)\\n    weights = weights / weights.sum(axis=0, keepdims=True)\\n    return np.einsum(\\"nb,nbc->bc\\", weights, probs)\\n\\n\\ndef trimmed_mean(logits: np.ndarray, drop: int = 1) -> np.ndarray:\\n    \\"\\"\\"Per-(b, c) trimmed mean: drop the highest and lowest ``drop`` values.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    n = logits.shape[0]\\n    if drop <= 0 or 2 * drop >= n:\\n        # With N=3 and drop=1 this falls through to the median.\\n        return np.median(logits, axis=0)\\n    sorted_logits = np.sort(logits, axis=0)\\n    return sorted_logits[drop : n - drop].mean(axis=0)\\n\\n\\ndef majority_vote(logits: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Hard-vote labels from per-model argmax. Returns shape ``[B]`` int labels.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    n, b, c = logits.shape\\n    preds = logits.argmax(axis=-1)  # [N, B]\\n    out = np.empty(b, dtype=np.int64)\\n    for i in range(b):\\n        counts = np.bincount(preds[:, i], minlength=c)\\n        max_count = counts.max()\\n        winners = np.flatnonzero(counts == max_count)\\n        if winners.size == 1:\\n            out[i] = int(winners[0])\\n        else:\\n            # Tie-break: pick the candidate with the highest summed logit\\n            tied_logits = np.array(\\n                [logits[:, i, label].sum() for label in winners], dtype=np.float64\\n            )\\n            out[i] = int(winners[int(tied_logits.argmax())])\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# Per-model temperature scaling (new in v2)\\n# ---------------------------------------------------------------------------\\n\\n\\ndef _nll_at_temperature(\\n    logits_one_model: np.ndarray, labels: np.ndarray, temperature: float\\n) -> float:\\n    log_probs = _log_softmax(logits_one_model / max(temperature, 1e-6), axis=-1)\\n    return float(-log_probs[np.arange(labels.shape[0]), labels].mean())\\n\\n\\ndef fit_per_model_temperatures(\\n    val_logits: np.ndarray,\\n    val_labels: np.ndarray,\\n    coarse_grid: tuple[float, ...] = (0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0),\\n    refine_iters: int = 30,\\n    refine_tol: float = 1e-3,\\n) -> np.ndarray:\\n    \\"\\"\\"Fit one temperature per model by NLL line-search on validation set.\\n\\n    Coarse grid first, then golden-section refinement. Returns ``[N]``.\\n    Calibrated logits are obtained as ``logits_k / T_k``.\\n    \\"\\"\\"\\n\\n    _check_logits(val_logits)\\n    if val_labels.ndim != 1 or val_labels.shape[0] != val_logits.shape[1]:\\n        raise ValueError(\\"val_labels must be 1D with length equal to batch size\\")\\n\\n    temps = np.empty(val_logits.shape[0], dtype=np.float64)\\n    for k in range(val_logits.shape[0]):\\n        per_model = val_logits[k]\\n        best_t = min(\\n            coarse_grid,\\n            key=lambda t: _nll_at_temperature(per_model, val_labels, t),\\n        )\\n        # Golden-section search around the best coarse value.\\n        lo = max(0.1, best_t / 2.0)\\n        hi = best_t * 2.0\\n        phi = (1.0 + 5.0 ** 0.5) / 2.0\\n        rho = 2.0 - phi\\n        a, b = lo, hi\\n        c = a + rho * (b - a)\\n        d = b - rho * (b - a)\\n        f_c = _nll_at_temperature(per_model, val_labels, c)\\n        f_d = _nll_at_temperature(per_model, val_labels, d)\\n        for _ in range(refine_iters):\\n            if f_c < f_d:\\n                b, d, f_d = d, c, f_c\\n                c = a + rho * (b - a)\\n                f_c = _nll_at_temperature(per_model, val_labels, c)\\n            else:\\n                a, c, f_c = c, d, f_d\\n                d = b - rho * (b - a)\\n                f_d = _nll_at_temperature(per_model, val_labels, d)\\n            if (b - a) < refine_tol:\\n                break\\n        temps[k] = 0.5 * (a + b)\\n    return temps\\n\\n\\ndef temperature_scaled_mean(\\n    logits: np.ndarray, temperatures: np.ndarray\\n) -> np.ndarray:\\n    \\"\\"\\"Mean of softmax(logits_k / T_k) across models. Returns probs ``[B, C]``.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    temperatures = np.asarray(temperatures, dtype=np.float64)\\n    if temperatures.shape != (logits.shape[0],):\\n        raise ValueError(\\n            f\\"Expected temperatures shape [{logits.shape[0]}], got {tuple(temperatures.shape)}\\"\\n        )\\n    scaled = logits / temperatures.reshape(-1, 1, 1)\\n    probs = _softmax(scaled, axis=-1)\\n    return probs.mean(axis=0)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Learned 3-weight aggregator (new in v2)\\n# ---------------------------------------------------------------------------\\n\\n\\ndef fit_learned_weights(\\n    val_logits: np.ndarray,\\n    val_labels: np.ndarray,\\n    n_iters: int = 200,\\n    lr: float = 0.05,\\n    init_z: tuple[float, ...] | None = None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Fit non-negative softmax-parametrized weights ``w = softmax(z)``.\\n\\n    Loss: cross-entropy on the weighted-sum logits ``Σ w_k · logits_k``.\\n    Returns ``{\\"weights\\": [N], \\"z\\": [N], \\"history\\": [...]}``.\\n    Trained on validation only — no leakage into the test report.\\n    \\"\\"\\"\\n\\n    _check_logits(val_logits)\\n    n_models = val_logits.shape[0]\\n    if val_labels.ndim != 1 or val_labels.shape[0] != val_logits.shape[1]:\\n        raise ValueError(\\"val_labels must be 1D with length equal to batch size\\")\\n\\n    z = np.zeros(n_models, dtype=np.float64) if init_z is None else np.asarray(init_z, dtype=np.float64)\\n    history: list[dict[str, float]] = []\\n    labels = val_labels.astype(np.int64)\\n    batch = val_logits.shape[1]\\n\\n    for step in range(1, n_iters + 1):\\n        w = _softmax(z, axis=-1)                              # [N]\\n        agg = np.einsum(\\"n,nbc->bc\\", w, val_logits)           # [B, C]\\n        log_p = _log_softmax(agg, axis=-1)                    # [B, C]\\n        loss = float(-log_p[np.arange(batch), labels].mean())\\n\\n        # ∂loss/∂agg = (softmax(agg) - one_hot(labels)) / B\\n        probs = np.exp(log_p)                                  # [B, C]\\n        grad_agg = probs.copy()\\n        grad_agg[np.arange(batch), labels] -= 1.0\\n        grad_agg /= batch                                      # [B, C]\\n\\n        # ∂agg/∂w_k = sum_{b,c} logits_k * grad_agg\\n        grad_w = np.einsum(\\"bc,nbc->n\\", grad_agg, val_logits)  # [N]\\n\\n        # ∂w/∂z: Jacobian of softmax: J = diag(w) - w w^T\\n        jac = np.diag(w) - np.outer(w, w)\\n        grad_z = jac @ grad_w\\n\\n        z -= lr * grad_z\\n        history.append({\\"step\\": float(step), \\"loss\\": loss})\\n        if step > 10 and abs(history[-1][\\"loss\\"] - history[-11][\\"loss\\"]) < 1e-6:\\n            break\\n\\n    weights = _softmax(z, axis=-1)\\n    return {\\"weights\\": weights, \\"z\\": z, \\"history\\": history}\\n\\n\\ndef learned_weights_mean(logits: np.ndarray, weights: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Sum_k w_k · logits_k. Returns ``[B, C]``.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    weights = _normalize_weights(weights)\\n    if weights.shape != (logits.shape[0],):\\n        raise ValueError(\\n            f\\"Expected weights shape [{logits.shape[0]}], got {tuple(weights.shape)}\\"\\n        )\\n    return np.einsum(\\"n,nbc->bc\\", weights, logits)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Diagnostics\\n# ---------------------------------------------------------------------------\\n\\n\\ndef oracle_topk(logits: np.ndarray, labels: np.ndarray, k: int = 1) -> float:\\n    \\"\\"\\"Hit if any of the N models has the true label in its top-k.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    n, b, c = logits.shape\\n    if labels.shape[0] != b:\\n        raise ValueError(\\"labels length must equal batch size\\")\\n    topk_idx = np.argsort(logits, axis=-1)[..., -k:]            # [N, B, k]\\n    label_col = labels.reshape(1, b, 1)\\n    hit_per_model = (topk_idx == label_col).any(axis=-1)        # [N, B]\\n    any_correct = hit_per_model.any(axis=0)\\n    return float(any_correct.mean())\\n\\n\\ndef pairwise_disagreement(logits: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Symmetric N×N matrix of disagreement rates between models (0 on diag).\\"\\"\\"\\n\\n    _check_logits(logits)\\n    preds = logits.argmax(axis=-1)                              # [N, B]\\n    n = preds.shape[0]\\n    matrix = np.zeros((n, n), dtype=np.float64)\\n    for i in range(n):\\n        for j in range(n):\\n            if i == j:\\n                matrix[i, j] = 0.0\\n            else:\\n                matrix[i, j] = float((preds[i] != preds[j]).mean())\\n    return matrix\\n\\n\\ndef per_class_disagreement_rate(\\n    logits: np.ndarray, labels: np.ndarray, label_names: list[str] | None = None\\n) -> dict[str, float]:\\n    \\"\\"\\"Fraction of inputs of each true class where ≥2 models disagree.\\"\\"\\"\\n\\n    _check_logits(logits)\\n    n, b, c = logits.shape\\n    preds = logits.argmax(axis=-1)                              # [N, B]\\n    out: dict[str, float] = {}\\n    unique = sorted(set(int(label) for label in labels.tolist()))\\n    for cls in unique:\\n        mask = labels == cls\\n        if not mask.any():\\n            continue\\n        cls_preds = preds[:, mask]                              # [N, n_cls]\\n        # disagreement = at least one pair of models predicts differently\\n        # equivalently: not all rows equal\\n        disagree = (cls_preds != cls_preds[0:1]).any(axis=0)\\n        rate = float(disagree.mean())\\n        key = label_names[cls] if (label_names and cls < len(label_names)) else str(cls)\\n        out[key] = rate\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# Convenience: collect everything in one call (used by notebook + sim)\\n# ---------------------------------------------------------------------------\\n\\n\\ndef evaluate_aggregators(\\n    test_logits: np.ndarray,\\n    test_labels: np.ndarray,\\n    val_logits: np.ndarray | None = None,\\n    val_labels: np.ndarray | None = None,\\n    label_names: list[str] | None = None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Run all aggregators on ``test_logits`` and report metrics.\\n\\n    Calibration / learned weights are fit on ``val_*`` if provided; otherwise\\n    those aggregators are skipped.\\n    \\"\\"\\"\\n\\n    _check_logits(test_logits)\\n    if test_labels.shape[0] != test_logits.shape[1]:\\n        raise ValueError(\\"test_labels length must equal test batch size\\")\\n\\n    def top1(probs_or_logits: np.ndarray) -> float:\\n        preds = probs_or_logits.argmax(axis=-1)\\n        return float((preds == test_labels).mean())\\n\\n    def topk_from_probs(probs: np.ndarray, k: int) -> float:\\n        topk_idx = np.argsort(probs, axis=-1)[..., -k:]\\n        label_col = test_labels.reshape(-1, 1)\\n        return float((topk_idx == label_col).any(axis=-1).mean())\\n\\n    aggregators: dict[str, Any] = {}\\n\\n    ml = mean_logits(test_logits)\\n    aggregators[\\"mean_logits\\"] = {\\n        \\"top1\\": top1(ml),\\n        \\"top3\\": topk_from_probs(_softmax(ml, axis=-1), k=3),\\n    }\\n\\n    mp = mean_probs(test_logits)\\n    aggregators[\\"mean_probs\\"] = {\\"top1\\": top1(mp), \\"top3\\": topk_from_probs(mp, k=3)}\\n\\n    cw = conf_weighted(test_logits)\\n    aggregators[\\"conf_weighted\\"] = {\\"top1\\": top1(cw), \\"top3\\": topk_from_probs(cw, k=3)}\\n\\n    tm = trimmed_mean(test_logits, drop=1)\\n    aggregators[\\"trimmed\\"] = {\\n        \\"top1\\": top1(tm),\\n        \\"top3\\": topk_from_probs(_softmax(tm, axis=-1), k=3),\\n    }\\n\\n    mv_preds = majority_vote(test_logits)\\n    aggregators[\\"majority_vote\\"] = {\\n        \\"top1\\": float((mv_preds == test_labels).mean()),\\n    }\\n\\n    if val_logits is not None and val_labels is not None:\\n        temps = fit_per_model_temperatures(val_logits, val_labels)\\n        ts = temperature_scaled_mean(test_logits, temps)\\n        aggregators[\\"temperature_scaled\\"] = {\\n            \\"T\\": temps.tolist(),\\n            \\"top1\\": top1(ts),\\n            \\"top3\\": topk_from_probs(ts, k=3),\\n        }\\n\\n        lw = fit_learned_weights(val_logits, val_labels)\\n        weights = lw[\\"weights\\"]\\n        lwm = learned_weights_mean(test_logits, weights)\\n        aggregators[\\"learned_weights\\"] = {\\n            \\"w\\": weights.tolist(),\\n            \\"z\\": lw[\\"z\\"].tolist(),\\n            \\"loss_history_tail\\": lw[\\"history\\"][-5:],\\n            \\"top1\\": top1(lwm),\\n            \\"top3\\": topk_from_probs(_softmax(lwm, axis=-1), k=3),\\n        }\\n\\n    return {\\n        \\"aggregators\\": aggregators,\\n        \\"oracle_top1\\": oracle_topk(test_logits, test_labels, k=1),\\n        \\"pairwise_disagreement\\": pairwise_disagreement(test_logits).tolist(),\\n        \\"per_class_disagreement_rate\\": per_class_disagreement_rate(\\n            test_logits, test_labels, label_names=label_names\\n        ),\\n    }\\n", "code/scripts/export_hash_kws_firmware.py": "from __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\n\\nREPO_ROOT = Path(__file__).resolve().parents[2]\\nTRAINING_ROOT = REPO_ROOT / \\"code\\" / \\"training\\"\\nif str(TRAINING_ROOT) not in sys.path:\\n    sys.path.insert(0, str(TRAINING_ROOT))\\n\\nfrom hash_kws_lab.config import ExperimentConfig, experiment_from_dict\\nfrom hash_kws_lab.models import (  # noqa: E402\\n    AnalyticHashConv2d,\\n    AnalyticHashDepthwiseConv2d,\\n    AnalyticHashLinear,\\n    HashDSCNN,\\n    build_student_model,\\n    summarize_model,\\n)\\n\\n\\ndef _parse_args() -> argparse.Namespace:\\n    parser = argparse.ArgumentParser(\\n        description=\\"Export a trained hash-KWS PyTorch bundle into ESP32 firmware arrays.\\",\\n    )\\n    parser.add_argument(\\n        \\"--bundle\\",\\n        type=Path,\\n        required=True,\\n        help=\\"Path to the .pt bundle produced by the hash KWS lab.\\",\\n    )\\n    parser.add_argument(\\n        \\"--output-dir\\",\\n        type=Path,\\n        default=REPO_ROOT / \\"code\\" / \\"firmware\\" / \\"hash_kws_runtime\\",\\n        help=\\"Directory where hash_model_data.cpp and metadata will be written.\\",\\n    )\\n    parser.add_argument(\\n        \\"--project-root\\",\\n        type=Path,\\n        default=REPO_ROOT,\\n        help=\\"Repository root for resolving data paths.\\",\\n    )\\n    parser.add_argument(\\n        \\"--device\\",\\n        type=str,\\n        default=\\"auto\\",\\n        choices=[\\"auto\\", \\"cpu\\", \\"cuda\\"],\\n        help=\\"Device used for calibration batches.\\",\\n    )\\n    parser.add_argument(\\n        \\"--calibration-split\\",\\n        type=str,\\n        default=\\"validation\\",\\n        choices=[\\"train\\", \\"validation\\", \\"test\\"],\\n        help=\\"Dataset split used for activation calibration.\\",\\n    )\\n    parser.add_argument(\\n        \\"--calibration-batches\\",\\n        type=int,\\n        default=8,\\n        help=\\"How many batches to use for activation scale calibration.\\",\\n    )\\n    parser.add_argument(\\n        \\"--firmware-export-snapshot\\",\\n        type=Path,\\n        default=None,\\n        help=(\\n            \\"Optional firmware_export_snapshot.json from a previous Colab run. \\"\\n            \\"When provided, activation_quant_params are reused and dataset calibration is skipped.\\"\\n        ),\\n    )\\n    return parser.parse_args()\\n\\n\\ndef _resolve_device(name: str) -> torch.device:\\n    if name == \\"cpu\\":\\n        return torch.device(\\"cpu\\")\\n    if name == \\"cuda\\":\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\"CUDA was requested for calibration, but it is not available.\\")\\n        return torch.device(\\"cuda\\")\\n    return torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n\\n\\ndef _load_bundle(bundle_path: Path) -> tuple[ExperimentConfig, dict[str, Any], nn.Module]:\\n    bundle = torch.load(bundle_path, map_location=\\"cpu\\")\\n    experiment_payload = bundle.get(\\"experiment\\")\\n    if not isinstance(experiment_payload, dict):\\n        raise ValueError(\\"Bundle does not contain a serialized experiment payload.\\")\\n\\n    experiment = experiment_from_dict(experiment_payload)\\n    model = build_student_model(experiment)\\n    state_dict = bundle.get(\\"state_dict\\")\\n    if not isinstance(state_dict, dict):\\n        raise ValueError(\\"Bundle does not contain a state_dict.\\")\\n    model.load_state_dict(state_dict, strict=True)\\n    model.eval()\\n    return experiment, bundle, model\\n\\n\\ndef _ensure_exportable_hash_model(model: nn.Module, experiment: ExperimentConfig) -> HashDSCNN:\\n    if not isinstance(model, HashDSCNN):\\n        raise TypeError(f\\"Expected HashDSCNN student model, got {type(model).__name__}\\")\\n    if experiment.model.hash_only_pointwise:\\n        raise NotImplementedError(\\n            \\"Firmware exporter currently supports fully hashed models only. \\"\\n            \\"The pointwise-only recipe needs dense-kernel support in the runtime.\\"\\n        )\\n    if not isinstance(model.conv0, AnalyticHashConv2d):\\n        raise TypeError(\\"Runtime exporter expects a hashed stem convolution.\\")\\n    for block in model.blocks:\\n        if not isinstance(block.depthwise, AnalyticHashDepthwiseConv2d):\\n            raise TypeError(\\"Runtime exporter expects hashed depthwise blocks.\\")\\n        if not isinstance(block.pointwise, AnalyticHashConv2d):\\n            raise TypeError(\\"Runtime exporter expects hashed pointwise blocks.\\")\\n    if not isinstance(model.fc, AnalyticHashLinear):\\n        raise TypeError(\\"Runtime exporter expects a hashed linear classifier.\\")\\n    return model\\n\\n\\ndef _bn_fold(conv_bias: torch.Tensor, bn: nn.BatchNorm2d) -> tuple[torch.Tensor, torch.Tensor]:\\n    gamma = bn.weight.detach().cpu().to(torch.float32)\\n    beta = bn.bias.detach().cpu().to(torch.float32)\\n    running_mean = bn.running_mean.detach().cpu().to(torch.float32)\\n    running_var = bn.running_var.detach().cpu().to(torch.float32)\\n    denom = torch.sqrt(running_var + float(bn.eps))\\n    post_scale = gamma / denom\\n    post_bias = beta + post_scale * (conv_bias.detach().cpu().to(torch.float32) - running_mean)\\n    return post_scale, post_bias\\n\\n\\ndef _quantize_codebook(codebook: torch.Tensor) -> tuple[list[int], float]:\\n    values = codebook.detach().cpu().to(torch.float32).view(-1)\\n    max_abs = float(values.abs().max().item())\\n    if max_abs <= 1e-12:\\n        return [0 for _ in range(values.numel())], 1.0\\n    scale = max_abs / 127.0\\n    quantized = torch.clamp(torch.round(values / scale), min=-127, max=127).to(torch.int8)\\n    return [int(item) for item in quantized.tolist()], float(scale)\\n\\n\\ndef _activation_forward(model: HashDSCNN, features: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:\\n    if features.dim() == 3:\\n        features = features.unsqueeze(1)\\n    stages: list[torch.Tensor] = []\\n    x = features\\n    x = torch.relu(model.bn0(model.conv0(x)))\\n    stages.append(x)\\n    for block in model.blocks:\\n        shortcut = x\\n        x = torch.relu(block.bn_dw(block.depthwise(x)))\\n        stages.append(x)\\n        x = block.bn_pw(block.pointwise(x))\\n        if block.residual:\\n            x = x + shortcut\\n        x = torch.relu(x)\\n        stages.append(x)\\n    return features, stages\\n\\n\\n@torch.no_grad()\\ndef _collect_activation_quant_params(\\n    model: HashDSCNN,\\n    loader: torch.utils.data.DataLoader[Any],\\n    device: torch.device,\\n    max_batches: int,\\n) -> list[dict[str, float]]:\\n    if max_batches <= 0:\\n        raise ValueError(\\"calibration-batches must be positive\\")\\n\\n    stage_names = [\\"stem\\"]\\n    for block_index in range(len(model.blocks)):\\n        stage_names.append(f\\"block{block_index}_depthwise\\")\\n        stage_names.append(f\\"block{block_index}_pointwise\\")\\n\\n    model = model.to(device)\\n    model.eval()\\n    input_abs_max = 0.0\\n    stage_abs_max = [0.0 for _ in stage_names]\\n\\n    for batch_index, (features, _) in enumerate(loader):\\n        if batch_index >= max_batches:\\n            break\\n        features = features.to(device=device, dtype=torch.float32, non_blocking=(device.type == \\"cuda\\"))\\n        input_tensor, stages = _activation_forward(model, features)\\n        input_abs_max = max(input_abs_max, float(input_tensor.abs().amax().item()))\\n        for index, stage in enumerate(stages):\\n            stage_abs_max[index] = max(stage_abs_max[index], float(stage.abs().amax().item()))\\n\\n    if input_abs_max <= 1e-12:\\n        input_abs_max = 1.0\\n    stage_output_scales = [max(value / 127.0, 1e-8) for value in stage_abs_max]\\n    input_scale = max(input_abs_max / 127.0, 1e-8)\\n\\n    quant_params: list[dict[str, float]] = []\\n    previous_scale = input_scale\\n    for stage_name, output_scale in zip(stage_names, stage_output_scales):\\n        quant_params.append(\\n            {\\n                \\"stage\\": stage_name,\\n                \\"input_scale\\": float(previous_scale),\\n                \\"output_scale\\": float(output_scale),\\n            }\\n        )\\n        previous_scale = output_scale\\n    return quant_params\\n\\n\\ndef _float_list(tensor: torch.Tensor) -> list[float]:\\n    return [float(value) for value in tensor.detach().cpu().to(torch.float32).view(-1).tolist()]\\n\\n\\ndef _format_cpp_float(value: float) -> str:\\n    text = f\\"{value:.9g}\\"\\n    if text in {\\"0\\", \\"-0\\"}:\\n        text = \\"0.0\\"\\n    elif all(marker not in text for marker in (\\".\\", \\"e\\", \\"E\\")):\\n        text = f\\"{text}.0\\"\\n    return f\\"{text}f\\"\\n\\n\\ndef _format_cpp_array(name: str, ctype: str, values: list[str], values_per_line: int = 8) -> str:\\n    lines: list[str] = [f\\"constexpr {ctype} {name}[] = {{\\"]\\n    for start in range(0, len(values), values_per_line):\\n        chunk = values[start : start + values_per_line]\\n        lines.append(\\"    \\" + \\", \\".join(chunk) + \\",\\")\\n    lines.append(\\"};\\")\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef _emit_conv_layer(layer_name: str, layer: AnalyticHashConv2d, post_scale: list[float], post_bias: list[float]) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    scale_name = f\\"k{layer_name}PostScale\\"\\n    bias_name = f\\"k{layer_name}PostBias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(scale_name, \\"float\\", [_format_cpp_float(value) for value in post_scale]),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in post_bias]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {scale_name},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.in_channels},\\",\\n            f\\"    {layer.out_channels},\\",\\n            f\\"    {layer.kernel_size[0]},\\",\\n            f\\"    {layer.kernel_size[1]},\\",\\n            f\\"    {layer.stride[0]},\\",\\n            f\\"    {layer.stride[1]},\\",\\n            f\\"    {layer.padding[0]},\\",\\n            f\\"    {layer.padding[1]},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _emit_depthwise_layer(\\n    layer_name: str,\\n    layer: AnalyticHashDepthwiseConv2d,\\n    post_scale: list[float],\\n    post_bias: list[float],\\n) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    scale_name = f\\"k{layer_name}PostScale\\"\\n    bias_name = f\\"k{layer_name}PostBias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(scale_name, \\"float\\", [_format_cpp_float(value) for value in post_scale]),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in post_bias]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {scale_name},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.channels},\\",\\n            f\\"    {layer.kernel_size[0]},\\",\\n            f\\"    {layer.kernel_size[1]},\\",\\n            f\\"    {layer.stride[0]},\\",\\n            f\\"    {layer.stride[1]},\\",\\n            f\\"    {layer.padding[0]},\\",\\n            f\\"    {layer.padding[1]},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _emit_linear_layer(layer_name: str, layer: AnalyticHashLinear) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    bias_name = f\\"k{layer_name}Bias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in _float_list(layer.bias)]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.in_dim},\\",\\n            f\\"    {layer.out_dim},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _build_cpp_source(\\n    experiment: ExperimentConfig,\\n    model: HashDSCNN,\\n    quant_params: list[dict[str, float]],\\n    source_bundle: Path,\\n) -> str:\\n    sections: list[str] = []\\n\\n    stem_post_scale, stem_post_bias = _bn_fold(model.conv0.bias, model.bn0)\\n    stem_arrays, stem_initializer = _emit_conv_layer(\\n        \\"Stem\\",\\n        model.conv0,\\n        _float_list(stem_post_scale),\\n        _float_list(stem_post_bias),\\n    )\\n    sections.append(stem_arrays)\\n\\n    depthwise_initializers: list[str] = []\\n    pointwise_initializers: list[str] = []\\n    residual_values: list[str] = []\\n    for block_index, block in enumerate(model.blocks):\\n        dw_post_scale, dw_post_bias = _bn_fold(block.depthwise.bias, block.bn_dw)\\n        dw_arrays, dw_initializer = _emit_depthwise_layer(\\n            f\\"Block{block_index}Depthwise\\",\\n            block.depthwise,\\n            _float_list(dw_post_scale),\\n            _float_list(dw_post_bias),\\n        )\\n        pw_post_scale, pw_post_bias = _bn_fold(block.pointwise.bias, block.bn_pw)\\n        pw_arrays, pw_initializer = _emit_conv_layer(\\n            f\\"Block{block_index}Pointwise\\",\\n            block.pointwise,\\n            _float_list(pw_post_scale),\\n            _float_list(pw_post_bias),\\n        )\\n        sections.append(dw_arrays)\\n        sections.append(pw_arrays)\\n        depthwise_initializers.append(dw_initializer)\\n        pointwise_initializers.append(pw_initializer)\\n        residual_values.append(\\"true\\" if block.residual else \\"false\\")\\n\\n    classifier_arrays, classifier_initializer = _emit_linear_layer(\\"Classifier\\", model.fc)\\n    sections.append(classifier_arrays)\\n\\n    activation_values: list[str] = []\\n    for quant in quant_params:\\n        activation_values.append(\\n            \\"{\\"\\n            f\\"{_format_cpp_float(quant[\'input_scale\'])}, \\"\\n            f\\"{_format_cpp_float(quant[\'output_scale\'])}\\"\\n            \\"}\\"\\n        )\\n    activations_initializer = \\",\\\\n        \\".join(activation_values)\\n\\n    body = [\\n        \'#include \\"hash_model_data.h\\"\',\\n        \\"\\",\\n        \\"namespace hash_kws {\\",\\n        \\"\\",\\n        \\"namespace {\\",\\n        \\"\\",\\n        f\\"// Generated from bundle: {source_bundle.as_posix()}\\",\\n        f\\"// Experiment tag: {experiment.tag}\\",\\n        f\\"// Frontend in training bundle: {experiment.feature.frontend_name}\\",\\n        \\"// Note: runtime input semantics must match the training frontend for accuracy to hold.\\",\\n        \\"\\",\\n        \\"\\\\n\\\\n\\".join(sections),\\n        \\"\\",\\n        \\"}  // namespace\\",\\n        \\"\\",\\n        \\"const HashDscnnModelData g_hash_model = {\\",\\n        \\"    true,\\",\\n        f\\"    {experiment.feature.n_mels},\\",\\n        f\\"    {experiment.feature.frame_count},\\",\\n        \\"    1,\\",\\n        f\\"    {model.conv0.out_channels},\\",\\n        f\\"    {len(model.blocks)},\\",\\n        f\\"    {experiment.num_labels},\\",\\n        f\\"    {stem_initializer},\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(depthwise_initializers),\\n        \\"    },\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(pointwise_initializers),\\n        \\"    },\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(residual_values),\\n        \\"    },\\",\\n        f\\"    {classifier_initializer},\\",\\n        \\"    {\\",\\n        f\\"        {activations_initializer}\\",\\n        \\"    },\\",\\n        \\"};\\",\\n        \\"\\",\\n        \\"}  // namespace hash_kws\\",\\n        \\"\\",\\n    ]\\n    return \\"\\\\n\\".join(body)\\n\\n\\ndef _bundle_flash_summary(model: HashDSCNN, experiment: ExperimentConfig, quant_params: list[dict[str, float]]) -> dict[str, Any]:\\n    model_summary = summarize_model(model, experiment)\\n    codebook_bytes = 0\\n    affine_bytes = 0\\n\\n    codebook_bytes += int(model.conv0.codebook.numel())\\n    affine_bytes += 4 * (int(model.conv0.out_channels) * 2)\\n    for block in model.blocks:\\n        codebook_bytes += int(block.depthwise.codebook.numel())\\n        affine_bytes += 4 * (int(block.depthwise.channels) * 2)\\n        codebook_bytes += int(block.pointwise.codebook.numel())\\n        affine_bytes += 4 * (int(block.pointwise.out_channels) * 2)\\n    codebook_bytes += int(model.fc.codebook.numel())\\n    affine_bytes += 4 * int(model.fc.out_dim)\\n\\n    activation_bytes = len(quant_params) * 8\\n    return {\\n        \\"hash_codebook_bytes_int8\\": codebook_bytes,\\n        \\"post_affine_and_classifier_bias_bytes_float32\\": affine_bytes,\\n        \\"activation_quant_bytes_float32\\": activation_bytes,\\n        \\"approx_runtime_model_bytes\\": codebook_bytes + affine_bytes + activation_bytes,\\n        \\"virtual_dense_int8_weight_bytes\\": model_summary[\\"virtual_dense_parameters\\"],\\n    }\\n\\n\\ndef _metadata_payload(\\n    experiment: ExperimentConfig,\\n    bundle: dict[str, Any],\\n    model: HashDSCNN,\\n    quant_params: list[dict[str, float]],\\n    source_bundle: Path,\\n    output_dir: Path,\\n) -> dict[str, Any]:\\n    return {\\n        \\"source_bundle\\": str(source_bundle),\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"bundle_stage_name\\": bundle.get(\\"stage_name\\", \\"student\\"),\\n        \\"frontend_warning\\": (\\n            \\"The exported runtime preserves model weights and activation scales, \\"\\n            \\"but on-device accuracy still depends on matching the training frontend.\\"\\n        ),\\n        \\"activation_quant_params\\": quant_params,\\n        \\"model_summary\\": summarize_model(model, experiment),\\n        \\"firmware_flash_summary\\": _bundle_flash_summary(model, experiment, quant_params),\\n        \\"runtime_output_dir\\": str(output_dir),\\n    }\\n\\n\\ndef export_bundle_to_firmware(\\n    bundle_path: Path,\\n    output_dir: Path,\\n    project_root: Path,\\n    device: torch.device,\\n    calibration_split: str = \\"validation\\",\\n    calibration_batches: int = 8,\\n    activation_quant_params: list[dict[str, float]] | None = None,\\n) -> dict[str, Any]:\\n    bundle_path = bundle_path.resolve()\\n    output_dir = output_dir.resolve()\\n    project_root = project_root.resolve()\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    experiment, bundle, model = _load_bundle(bundle_path)\\n    model = _ensure_exportable_hash_model(model, experiment)\\n\\n    if activation_quant_params is None:\\n        from hash_kws_lab.data import prepare_dataloaders\\n\\n        calibration = prepare_dataloaders(\\n            project_root=project_root,\\n            experiment=experiment,\\n            device=device,\\n        )\\n        loader = calibration[\\"loaders\\"][calibration_split]\\n        quant_params = _collect_activation_quant_params(\\n            model=model,\\n            loader=loader,\\n            device=device,\\n            max_batches=calibration_batches,\\n        )\\n    else:\\n        quant_params = activation_quant_params\\n\\n    cpp_source = _build_cpp_source(\\n        experiment=experiment,\\n        model=model,\\n        quant_params=quant_params,\\n        source_bundle=bundle_path,\\n    )\\n    cpp_path = output_dir / \\"hash_model_data.cpp\\"\\n    cpp_path.write_text(cpp_source, encoding=\\"utf-8\\")\\n\\n    metadata = _metadata_payload(\\n        experiment=experiment,\\n        bundle=bundle,\\n        model=model,\\n        quant_params=quant_params,\\n        source_bundle=bundle_path,\\n        output_dir=output_dir,\\n    )\\n    metadata_path = output_dir / \\"hash_model_export_metadata.json\\"\\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return {\\n        \\"cpp_path\\": str(cpp_path),\\n        \\"metadata_path\\": str(metadata_path),\\n        \\"output_dir\\": str(output_dir),\\n        \\"activation_quant_params\\": quant_params,\\n        \\"firmware_flash_summary\\": metadata[\\"firmware_flash_summary\\"],\\n    }\\n\\n\\ndef main() -> None:\\n    args = _parse_args()\\n    device = _resolve_device(args.device)\\n    activation_quant_params = None\\n    if args.firmware_export_snapshot is not None:\\n        snapshot = json.loads(args.firmware_export_snapshot.read_text(encoding=\\"utf-8\\"))\\n        activation_quant_params = snapshot.get(\\"activation_quant_params\\")\\n        if not isinstance(activation_quant_params, list):\\n            raise ValueError(\\n                \\"Snapshot does not contain activation_quant_params as a list: \\"\\n                f\\"{args.firmware_export_snapshot}\\"\\n            )\\n    result = export_bundle_to_firmware(\\n        bundle_path=args.bundle,\\n        output_dir=args.output_dir,\\n        project_root=args.project_root,\\n        device=device,\\n        calibration_split=args.calibration_split,\\n        calibration_batches=args.calibration_batches,\\n        activation_quant_params=activation_quant_params,\\n    )\\n    print(json.dumps(result, ensure_ascii=False, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "code/firmware/hash_kws_runtime/README.md": "# Hash KWS Runtime\\n\\nThis folder contains a custom low-level inference path for hash-compressed KWS models on ESP32.\\n\\nThe design goal is different from the existing TFLite Micro path:\\n\\n- preserve the memory win of analytic-hash codebooks;\\n- avoid dense weight materialization;\\n- reuse the existing `49 x 40` audio feature pipeline from the firmware side;\\n- keep the runtime simple enough to optimize incrementally on `ESP32-S3`.\\n\\n## Current Runtime Strategy\\n\\n- Input features are expected as a `49 x 40` `int8` spectrogram buffer.\\n- Internal activations use `int8` double-buffer scratch memory.\\n- Hash codebooks are stored as quantized `int8` arrays plus one scale per layer.\\n- BatchNorm is folded into per-output-channel post-affine parameters instead of into dense weights.\\n- The final layer produces logits.\\n- The current recommended firmware policy is not pure always-on smoothing:\\n  - sparse idle probes during quiet periods;\\n  - speech episodes when recent slices show activity;\\n  - peak-hold command selection inside the episode;\\n  - concise serial output by default for live board checks.\\n\\n## Current Board Status\\n\\n- Current live-board feedback says this branch works tolerably as a deployment baseline.\\n- The main remaining problem is heavy invoke latency on the deeper hash model, not microphone capture.\\n- Because of that, increasing temporal input size above `49` frames is not the right next step for the latency branch.\\n- If training-side changes are needed, prefer:\\n  - exact frontend alignment;\\n  - equal or smaller time footprint;\\n  - lower-cost temporal resolution before trying larger windows.\\n\\n## Export Path\\n\\nGenerate the firmware arrays from a trained hash bundle with:\\n\\n```powershell\\npython code/scripts/export_hash_kws_firmware.py `\\n  --bundle code/training/hash_artifacts/<experiment-tag>/hash_kws_student_student.pt\\n```\\n\\nThe exporter will:\\n\\n- restore the PyTorch hash model from the compact bundle;\\n- fold `BatchNorm` into post-convolution affine parameters;\\n- calibrate per-stage activation scales on a few dataset batches;\\n- quantize each codebook to `int8` without materializing dense weights;\\n- overwrite `hash_model_data.cpp` and emit `hash_model_export_metadata.json`.\\n\\n## Important Limitations\\n\\n- The placeholder `hash_model_data.cpp` intentionally marks the model as unavailable until a real export is generated.\\n- Current runtime math preserves the hash-compressed weights, but accuracy still depends on the on-device frontend matching the training frontend.\\n- Current runtime uses two full `int8` activation buffers. For the `64 x 20 x 25` deeper hash model this is about `64 KB` of scratch, so the next memory optimization target is fused `depthwise -> pointwise` streaming.\\n", "code/firmware/hash_kws_runtime/hash_model_types.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n\\n#include <cstddef>\\n#include <cstdint>\\n\\nnamespace hash_kws {\\n\\nconstexpr int kHashInputRows = 40;\\nconstexpr int kHashInputCols = 49;\\nconstexpr int kHashInputChannels = 1;\\nconstexpr int kHashMaxBlocks = 4;\\nconstexpr int kHashMaxActivationStages = 1 + (2 * kHashMaxBlocks);\\nconstexpr int kHashMaxChannels = 128;\\nconstexpr int kHashMaxClasses = 16;\\n\\nstruct HashActivationQuantParams {\\n  float input_scale;\\n  float output_scale;\\n};\\n\\nstruct HashConvLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* post_scale;\\n  const float* post_bias;\\n  int codebook_size;\\n  int in_channels;\\n  int out_channels;\\n  int kernel_h;\\n  int kernel_w;\\n  int stride_h;\\n  int stride_w;\\n  int padding_h;\\n  int padding_w;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashDepthwiseLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* post_scale;\\n  const float* post_bias;\\n  int codebook_size;\\n  int channels;\\n  int kernel_h;\\n  int kernel_w;\\n  int stride_h;\\n  int stride_w;\\n  int padding_h;\\n  int padding_w;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashLinearLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* bias;\\n  int codebook_size;\\n  int in_dim;\\n  int out_dim;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashDscnnModelData {\\n  bool available;\\n  int input_rows;\\n  int input_cols;\\n  int input_channels;\\n  int stem_out_channels;\\n  int num_blocks;\\n  int num_classes;\\n  HashConvLayerData stem;\\n  HashDepthwiseLayerData depthwise[kHashMaxBlocks];\\n  HashConvLayerData pointwise[kHashMaxBlocks];\\n  bool block_residual[kHashMaxBlocks];\\n  HashLinearLayerData classifier;\\n  HashActivationQuantParams activations[kHashMaxActivationStages];\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n", "code/firmware/hash_kws_runtime/hash_model_settings.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n\\n#include <cstdint>\\n\\nnamespace hash_kws {\\n\\nconstexpr int kCategoryCount = 12;\\nconstexpr int kUnknownIndex = 10;\\nconstexpr int kSilenceIndex = 11;\\n\\nextern const char* kCategoryLabels[kCategoryCount];\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n", "code/firmware/hash_kws_runtime/hash_model_settings.cpp": "#include \\"hash_model_settings.h\\"\\n\\nnamespace hash_kws {\\n\\nconst char* kCategoryLabels[kCategoryCount] = {\\n    \\"yes\\",\\n    \\"no\\",\\n    \\"up\\",\\n    \\"down\\",\\n    \\"left\\",\\n    \\"right\\",\\n    \\"on\\",\\n    \\"off\\",\\n    \\"stop\\",\\n    \\"go\\",\\n    \\"unknown\\",\\n    \\"silence\\",\\n};\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_model_data.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n\\n#include \\"hash_model_types.h\\"\\n\\nnamespace hash_kws {\\n\\nextern const HashDscnnModelData g_hash_model;\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n", "code/firmware/hash_kws_runtime/hash_model_data.cpp": "#include \\"hash_model_data.h\\"\\n\\nnamespace hash_kws {\\n\\nnamespace {\\n\\n// Generated from bundle: /content/code/training/hash_artifacts/hash_kws12_iterlab_v1_hash_deeper_fair_ce_exact_microfrontend/hash_kws_student_student.pt\\n// Experiment tag: hash_kws12_iterlab_v1_hash_deeper_fair_ce_exact_microfrontend\\n// Frontend in training bundle: exact_microfrontend\\n// Note: runtime input semantics must match the training frontend for accuracy to hold.\\n\\nconstexpr int8_t kStemCodebook[] = {\\n    1, 3, -14, 24, -5, -17, 14, -43, -14, -11, -7, 5, 3, -20, -22, -11,\\n    7, 44, -47, -19, 3, -13, -3, 5, -15, 18, -20, -2, -14, 7, 58, -21,\\n    30, -7, 24, 13, -18, -4, 0, 35, 1, -5, 66, -28, -3, 13, 8, -83,\\n    -9, -2, 11, 27, 36, 0, -4, -2, 2, 12, 104, 60, 1, -24, 3, -3,\\n    -2, 4, 7, 0, 25, 12, 24, 0, 48, 14, -29, 2, -15, 16, 1, -31,\\n    127, 5, 0, -28, -10, -22, -15, 2, -6, 2, -27, -1, 5, -22, 27, 14,\\n    -3, 14, -18, -43, 1, 32, 30, -4, 42, -18, -10, -5, -2, 18, 27, -2,\\n    -8, -3, -1, 78, 30, -70, 10, -4, -2, -1, 7, -30, 17, 19, -17, 5,\\n    -44, -19, 9, -5, -3, 1, -11, -21, 1, -14, -5, 0, 5, 15, 23, -19,\\n    1, 31, -39, -4, -4, -26, -1, -27, -10, -24, 6, -16, 10, -61, -3, -5,\\n    4, -8, -4, -2, -32, 8, -15, -3, 33, -3, -26, -5, -52, -1, 11, -26,\\n    14, 19, 21, -39, 8, 23, 19, -12, -2, 9, -29, -23, -12, 96, 1, 9,\\n    -20, -3, 1, -32, 9, 26, -6, -13, -21, -1, 70, -1, 0, 16, 0, -13,\\n    32, -5, -12, -3, -2, 6, -1, 18, 29, -11, -8, -2, 3, -33, 2, -3,\\n    22, -20, 13, 18, 25, -38, -1, 13, -14, -1, -1, 5, -13, -1, 2, -8,\\n    -13, 0, 35, -30, -16, -3, 2, -35, 8, 16, -25, 10, -1, 7, -17, 47,\\n    0, -18, -34, 19, -13, -8, 2, -5, -15, 15, -3, 2, -11, -63, 12, 4,\\n    -39, 0, 1, -15, 39, -16, -3, -6, -5, 28, -2, -13, -2, 0, -12, -27,\\n    -17, 48, -1, -5, -21, 14, 3, -1, 1, -7, -40, -3, -25, 23, 16, -7,\\n    16, -1, 8, 13, -23, 2, -34, -2, -5, 91, -9, 21, 29, -9, 24, -17,\\n    -26, -1, -3, 10, 8, -22, 15, -29, 45, 34, -18, 31, 1, 12, 2, 36,\\n    52, 7, -32, -21, 43, -16, 18, -30, 8, -16, -44, 12, -1, -5, 1, -25,\\n    16, 2, -25, 15, -1, 66, -13, -12, 15, -7, -10, 7, 1, 14, -28, -13,\\n    -17, 2, -12, -5, -5, -31, 7, 1, -13, 10, 10, 33, -26, 7, 1, -31,\\n    19, -30, 0, -10, -10, -40, 0, -4, -57, -2, -9, -24, -34, 0, 23, 9,\\n    -8, -3, -20, 7, 0, -18, 22, 6, 1, 0, 13, -16, 1, 1, 5, 2,\\n    33, -20, -9, 1, 16, 16, -22, -34, -3, 12, 0, 27, -28, -4, 31, 21,\\n    -10, 6, 20, 29, 3, 1, -22, -15, -20, 43, -23, -21, -10, 15, -26, -26,\\n    0, -21, 29, -4, -55, 40, 61, 9, 13, 30, -1, 87, -25, -31, -5, -5,\\n    47, -10, -50, -8, 26, -11, 7, -12, -6, -25, -1, -4, -2, -9, 9, 1,\\n    0, -28, 8, 55, 4, -41, -5, 6, 22, -6, 20, 25, -1, 1, -13, -21,\\n    4, 3, 29, -5,\\n};\\n\\nconstexpr float kStemPostScale[] = {\\n    0.0274524298f, 0.0131988935f, 0.0663866773f, 0.0101192212f, 0.0614218153f, 0.0162766781f, 0.0240329467f, 0.0233545844f,\\n    0.0220983848f, 0.0526469909f, 0.0195261817f, 0.051181864f, 0.0150989648f, 0.0236263871f, 0.0142066609f, 0.0122181121f,\\n    0.0201264769f, 0.0169047639f, 0.025105387f, 0.0624888912f, 0.0484807082f, 0.0356820077f, 0.0290185697f, 0.0242739748f,\\n    0.00941671804f, 0.00787928887f, 0.0408770107f, 0.0173563752f, 0.0107734418f, 0.0699072033f, 0.0502869487f, 0.0267868191f,\\n    0.0201193057f, 0.0299614985f, 0.00483641867f, 0.0650268272f, 0.025840722f, 0.0625862628f, 0.0677033663f, 0.0756662562f,\\n    0.0583017282f, 0.0573115386f, 0.026873894f, 0.0418500416f, 0.00388057716f, 0.0438049249f, 0.0495975651f, 0.0272251014f,\\n    0.0157587882f, 0.0291513857f, 0.0441569127f, 0.0274223555f, 0.00761247426f, 0.0133383358f, 0.0274064709f, 0.0128270928f,\\n    0.0520713218f, 0.0191278346f, 0.0190017261f, 0.041222699f, 0.00650399737f, 0.0625844896f, 0.031084476f, 0.0189757403f,\\n};\\n\\nconstexpr float kStemPostBias[] = {\\n    -0.370643318f, 0.177714333f, 0.201514691f, 0.83929497f, 0.400850654f, -0.168413132f, 0.861433983f, -0.690443754f,\\n    0.181137383f, 0.0419436395f, -0.427700818f, -0.583491921f, -0.306531191f, 0.345791101f, 0.220764458f, -0.222054154f,\\n    -0.606688023f, -0.727348149f, -0.575681806f, 0.436254203f, 0.801969349f, 0.545996368f, -0.449672103f, -0.0518669486f,\\n    -0.28775537f, -0.664759517f, 0.621991217f, -0.600734651f, -0.136847198f, 0.375429153f, 0.173612043f, -0.0414918065f,\\n    0.563298404f, -0.657793105f, -0.321603745f, 0.321129322f, -0.471146375f, -0.30761081f, 0.581059992f, -0.338875234f,\\n    0.505832851f, -0.285668314f, 0.20027715f, 0.878555059f, 0.762812018f, -0.479189038f, -0.596189022f, -0.534612179f,\\n    -0.646461606f, -0.673870802f, -0.439045668f, -0.432548285f, -0.761044025f, -0.257891953f, -0.589128971f, -0.228312984f,\\n    0.0967734158f, 0.625607371f, 0.647040725f, -0.425230086f, -0.325269938f, 0.216103122f, 0.694581568f, -0.184050962f,\\n};\\n\\nconstexpr int8_t kBlock0DepthwiseCodebook[] = {\\n    -19, 0, 2, -27, -15, -3, -21, -4, 0, -2, -7, -11, 0, 21, -71, 31,\\n    -2, 4, -5, 2, 28, 16, -9, -1, 21, -29, -13, 2, 0, -12, 21, 4,\\n    -1, 16, 3, 14, -6, -28, -31, -18, -5, -67, 3, -35, 23, 0, -25, -14,\\n    2, -31, -12, -21, -14, 23, 12, 6, 0, -44, -21, -5, -27, 6, 26, 65,\\n    24, -18, 3, -32, 4, 0, -3, 10, -87, -7, -60, -37, -2, -2, -8, -75,\\n    -10, -22, 23, -7, 6, 25, -1, 1, -20, -25, -8, 25, 8, 2, 16, -43,\\n    42, 35, 1, -11, 12, 1, -9, -3, -42, -1, 7, -8, -1, 1, -10, -14,\\n    -5, -30, 24, 57, -5, 5, 18, -4, 0, -17, -15, -12, 4, 11, -20, -2,\\n    -5, -5, 9, -2, -10, -1, 9, 0, -16, -24, -1, -4, -25, -6, 2, 7,\\n    7, 18, 0, -40, -13, -1, -9, 24, 19, 1, -17, -4, 2, 2, -14, -3,\\n    -2, -12, -39, -2, 1, 11, -18, 0, -1, -12, -25, 30, -2, 4, -16, -1,\\n    -23, -81, -2, -1, -5, -3, -15, -4, -10, -22, 0, 38, 20, -1, 1, 31,\\n    1, -38, -14, 10, -4, -1, -24, -28, 13, -2, -17, -2, -54, -25, 36, 14,\\n    3, -11, -32, -8, -64, -9, 7, -33, 4, -1, 17, 1, 0, 12, -1, 7,\\n    -29, 1, 36, -19, 10, -3, 1, -49, 31, 16, 16, -9, -2, 25, -13, 0,\\n    2, -4, 18, 16, -24, -18, -19, 18, -17, -3, 0, -10, -4, -17, -20, -11,\\n    -13, 5, 5, -24, -25, 2, 0, -3, 12, -45, 10, 22, 4, -19, 33, -73,\\n    0, -4, -36, -7, 31, -10, -10, -1, -16, -20, -44, -4, -4, 64, 13, 44,\\n    -1, -18, -1, -4, -127, -24, 0, 13, 19, 7, -26, -26, 14, 4, -10, -11,\\n    10, 1, 15, -1, -2, -2, -4, 48, 2, 32, 23, -56, -1, 1, 40, -14,\\n    1, 0, 14, -1, 3, -7, -15, -3, 3, 16, -4, -2, 20, -4, -1, 5,\\n    6, -3, -2, -17, 2, -10, 2, 1, -9, 16, -11, -7, -6, -1, 15, -9,\\n    6, 0, -21, -55, 76, -15, -8, -9, -3, 20, 30, -14, 1, 47, -13, -2,\\n    1, 15, -11, 2, -26, -17, 8, 47, -55, 18, -8, 2, -23, -12, 2, -21,\\n    22, -9, -7, 46, -11, -2, 38, -5, 17, 1, -12, 2, 6, 50, 11, -21,\\n    -70, -4, -1, -16, -18, -4, -18, 0, -2, -19, -20, -46, -23, 0, -11, -16,\\n    38, 10, 14, 4, -6, -3, -53, 37, -3, -16, -8, 5, 41, -15, 25, 17,\\n    -15, -17, 33, -4, -22, -2, -3, -18, 11, -18, 3, 29, -36, 34, -1, 17,\\n    6, -18, -26, 2, 3, 4, -13, -45, 8, 0, -9, 10, 31, -24, -4, 17,\\n    1, 14, -13, -10, 5, 9, -31, -9, -3, 8, 25, 2, 23, -5, 36, -1,\\n    10, -8, 10, -3, -2, 18, -1, -45, -10, -20, 1, -9, -35, -58, 1, -26,\\n    6, -1, 0, -11,\\n};\\n\\nconstexpr float kBlock0DepthwisePostScale[] = {\\n    5.94903135f, 17.9304428f, 3.70813632f, 11.9803343f, 11.4727325f, 48.3728065f, 2.42755365f, 8.68457413f,\\n    9.920578f, 4.99808741f, 5.77575731f, 2.6796782f, 16.5970783f, 9.44140053f, 9.80344772f, 40.6081467f,\\n    4.88569593f, 4.89915895f, 5.94364595f, 1.68227923f, 6.15781879f, 7.08906794f, 4.97152519f, 5.3566699f,\\n    159.778091f, 24.7189236f, 4.11771059f, 8.76995087f, 49.6462479f, 2.49942756f, 4.48157549f, 3.00673103f,\\n    5.60084963f, 3.73295379f, 148.141159f, 3.76947665f, 5.21164799f, 7.26837778f, 4.36310911f, 2.62594628f,\\n    8.43365955f, 3.6028409f, 6.44711351f, 4.69412088f, 24.0014915f, 4.85875559f, 5.44933176f, 8.60033321f,\\n    19.728405f, 8.75557041f, 4.26825285f, 6.34193754f, 27.0390854f, 69.0814514f, 9.22048283f, 8.78682995f,\\n    2.94115877f, 9.11514187f, 5.97941875f, 10.3131428f, 235.983444f, 3.47125506f, 9.03884506f, 43.7444725f,\\n};\\n\\nconstexpr float kBlock0DepthwisePostBias[] = {\\n    0.532057345f, -0.246403903f, 0.685514092f, 0.599320471f, 0.402630508f, -0.477514386f, 0.617864728f, -0.936891794f,\\n    0.566204011f, 0.0955446362f, 0.975583315f, 0.640869915f, 0.976056814f, 0.693793237f, -0.708592176f, -0.208683223f,\\n    -0.78711772f, 0.56632477f, 0.713944614f, 0.710312426f, 0.471567631f, -0.617999673f, -1.16705787f, 1.10828757f,\\n    -0.19521004f, -0.946388364f, 0.774649143f, -0.116290137f, -0.262219936f, 0.0865733027f, 0.951984048f, 0.840532184f,\\n    0.65498507f, 1.14446318f, 0.0378903076f, -0.186770409f, 0.517227352f, 0.0966325253f, 0.867909968f, 0.339827478f,\\n    1.00822926f, 0.617522359f, 0.659223199f, 0.568219066f, 1.08763206f, -0.856335044f, 0.401460677f, 0.925750911f,\\n    0.523251534f, 0.725804567f, 0.444639683f, -0.626074731f, -1.19842041f, -0.603245735f, 0.305699676f, -0.868868232f,\\n    0.75328207f, 0.40818879f, 0.688304305f, -0.202317506f, -0.292101622f, 0.57431978f, 0.965709805f, -0.123187572f,\\n};\\n\\nconstexpr int8_t kBlock0PointwiseCodebook[] = {\\n    -7, 11, 47, 65, -18, -28, 45, -20, 53, 30, 1, 8, 25, -64, -32, 4,\\n    43, 29, -31, 7, 78, 2, -11, 35, -20, -62, 6, -30, -43, 26, -48, -61,\\n    46, -28, 44, -23, -34, -5, -127, -79, 36, 9, 52, -63, -18, -12, -4, -17,\\n    19, 18, -25, 5, -45, 47, -25, -27, 17, -17, 35, 16, 27, -33, 15, -28,\\n    -1, 76, 49, 30, 34, -5, -27, -34, -26, -9, -13, 25, 38, 90, -63, -22,\\n    -48, 8, 5, -78, 55, 59, -28, 12, 59, -36, 49, -9, 29, 8, 47, -64,\\n    -20, -72, 0, 38, -33, 12, -10, 13, -52, -5, 24, -13, 7, 107, 21, -32,\\n    -46, 53, -32, 49, -38, 19, 30, 40, -32, -28, -17, -18, 24, -88, 6, -6,\\n    34, -33, 54, -58, 22, -18, 17, -1, 39, 38, -20, 12, -66, 0, 37, -54,\\n    30, 10, 70, -29, -49, -31, -37, 6, 8, -4, -32, 11, -7, -32, 69, 7,\\n    51, 69, -16, 1, -3, 4, 8, 2, -28, -33, 43, -54, 0, 36, -14, 11,\\n    -27, 40, -82, 73, 51, -1, -38, -8, 65, -26, -90, -1, -30, -34, 12, -11,\\n    -14, -3, 2, 38, -5, -10, -39, 30, 39, -1, 16, 22, 13, 2, 8, 77,\\n    -2, -19, 11, 15, -34, -41, -56, -66, 34, 11, 30, 41, -58, -5, -14, -57,\\n    -40, 29, 27, 49, 35, 73, 52, -20, -11, 1, -11, -43, -5, 3, -1, -28,\\n    0, 25, 18, 51, 9, -14, -40, -32, -7, -54, -21, -44, -16, 28, -19, -44,\\n    23, 0, -38, -31, 16, 24, -7, -2, -33, -5, -4, -4, -1, 15, -15, 16,\\n    -1, -11, 33, -21, 8, -25, 106, 34, -22, -22, -28, -20, 35, 73, 4, -47,\\n    47, -78, 28, -45, -60, 1, 27, 37, -45, -18, -36, 36, 23, -11, -7, -12,\\n    -25, 2, 44, -46, -6, 23, 33, -44, 28, 18, -48, -11, 77, 5, -47, -25,\\n    52, -5, 60, 38, 24, -32, 68, 1, 39, 18, 2, 52, -3, -2, 41, 53,\\n    -23, 3, 25, -65, -24, -11, -46, -9, -82, 6, 96, 20, -23, 42, 0, -6,\\n    19, 12, 28, 95, 27, 17, 52, 11, -7, 13, 68, 17, -1, -54, 22, -4,\\n    -11, -23, 20, 37, 6, -27, 14, -4, -58, 33, -82, -14, -40, 14, 40, 10,\\n    50, -22, -41, -2, 47, 28, 17, 69, 33, 26, -13, -8, -17, 14, 69, 4,\\n    25, -38, 33, -106, -31, -16, 37, 20, 13, 1, -20, -25, -17, -2, -8, -3,\\n    -39, 10, -30, 27, 10, 31, -48, 25, 15, 35, -31, -34, -20, -45, 30, -71,\\n    -26, 32, 1, -37, 84, -31, -20, 9, 117, 40, -9, -12, 16, 66, -36, 25,\\n    0, -1, -28, 46, 43, 0, 74, 49, 28, -10, -61, -9, -59, 79, 37, 21,\\n    -50, 41, 10, 80, -26, -24, -33, 31, -44, 95, 13, -50, -35, 5, 5, -9,\\n    19, -16, 3, 32, -15, -4, 30, 17, -4, -41, 10, -29, -12, -16, 31, -28,\\n    24, 35, -20, -83,\\n};\\n\\nconstexpr float kBlock0PointwisePostScale[] = {\\n    1.7958014f, 2.20405436f, 1.9708426f, 1.06649792f, 1.71859717f, 2.03103948f, 2.30404472f, 2.09905863f,\\n    1.38693321f, 2.10321307f, 2.04023695f, 2.72119427f, 1.48954391f, 2.87970924f, 1.73472154f, 2.04569006f,\\n    2.79115438f, 1.43731785f, 1.81529808f, 2.23951602f, 2.95713615f, 1.228585f, 2.5660131f, 2.81592607f,\\n    2.34051728f, 1.41342258f, 3.52773857f, 2.48662806f, 2.17572498f, 1.83069706f, 1.92073786f, 2.30423522f,\\n    1.31580865f, 2.87328482f, 2.53999305f, 1.39698255f, 2.11891437f, 1.56973672f, 2.6824069f, 1.83262241f,\\n    1.91958427f, 2.44637442f, 1.17318666f, 0.862079024f, 3.18744564f, 1.423648f, 1.43320441f, 2.99298787f,\\n    1.62412083f, 2.70822167f, 1.31854427f, 2.10066533f, 1.24393916f, 2.52452254f, 1.58492827f, 1.66282535f,\\n    1.16462302f, 2.15516996f, 1.6992712f, 1.07622373f, 2.61881375f, 2.49069786f, 2.44894147f, 2.58287358f,\\n};\\n\\nconstexpr float kBlock0PointwisePostBias[] = {\\n    -1.23212957f, 0.864770651f, -0.611691594f, -0.0235978961f, 0.0556518435f, 0.56363368f, -0.552818358f, -0.557392955f,\\n    0.337959111f, -0.238587976f, -1.11967528f, -0.898240805f, 1.05534828f, -0.262620091f, -0.475681603f, 1.56630516f,\\n    -0.127009064f, 0.0413986742f, -0.0802148581f, 0.0962275714f, -0.0601454675f, -0.259459853f, -0.433310986f, -0.595122397f,\\n    -0.724910378f, -0.231958032f, -0.0239577368f, -0.838436663f, 0.0193344802f, 0.773133636f, -0.0607467294f, -1.46977401f,\\n    0.724664271f, 0.106885895f, -1.14846826f, 0.514198244f, -0.893851578f, -0.0715118349f, -0.609482825f, -0.232959732f,\\n    -0.597825944f, -0.604743481f, 0.321643978f, 0.644926786f, -0.684749126f, -0.176806495f, 1.17646527f, -0.465003133f,\\n    -0.722736657f, 1.02941442f, -0.0343369842f, -0.933597088f, -0.0537264347f, -0.137231305f, -0.26567781f, -0.515652895f,\\n    0.171090692f, -0.205701947f, -0.701752007f, 0.184753105f, 1.25205827f, -0.433305323f, -1.00497997f, 1.3218168f,\\n};\\n\\nconstexpr int8_t kBlock1DepthwiseCodebook[] = {\\n    -5, 23, 10, -1, -32, 2, 17, 4, -39, 22, 41, -16, -52, -33, -2, -17,\\n    -4, -9, 15, 25, 26, -30, -50, -25, -5, -20, 19, -40, -3, 3, 100, -4,\\n    -33, -20, 2, -26, -9, 32, -54, 28, -1, -36, 14, 63, -2, 5, -27, 74,\\n    -88, 9, -32, -54, -6, 12, -16, 15, -2, -2, -84, 32, 10, -22, -36, -7,\\n    0, -37, -5, -1, 18, 4, -26, 36, 26, -36, -4, -9, -2, -55, -4, -39,\\n    -98, -4, 36, 16, -22, -1, 5, 10, 43, 2, -18, -14, 16, 1, -7, -16,\\n    -3, -2, 36, 27, -2, 6, -24, 7, -3, -1, -9, -1, 32, -3, -19, 3,\\n    -25, -25, 26, -2, 43, 16, 3, -92, 6, -11, 7, -6, -2, -26, -1, 5,\\n    40, -10, -3, -11, 2, -2, 6, 14, 3, 0, 14, -30, -31, -22, -2, -10,\\n    -2, 11, 48, 10, 3, -8, -13, 11, 14, 19, -31, 0, -10, 43, -37, 1,\\n    -10, -36, 5, 22, 93, -14, 1, 42, -27, -16, -31, 22, -33, 17, -53, 24,\\n    -127, 0, -52, 57, 7, 8, 59, -12, 4, 51, 8, 31, 2, 23, 7, -17,\\n    17, 33, -10, -69, 9, -2, 30, 14, -37, -58, -15, 2, 29, 5, 27, 0,\\n    1, -3, 11, 39, -9, 20, 56, 20, 16, -15, 23, 2, -4, -2, -34, 42,\\n    -9, 31, 0, 12, 4, -16, -1, -51, 5, 33, 9, 26, -61, -1, -13, -9,\\n    -34, 6, 25, -16, 13, -1, -35, -20, 1, 11, -4, 6, 1, 2, -46, -12,\\n    41, -14, -34, -5, 37, -1, -7, -3, 16, 10, 2, 0, 4, -68, -1, 1,\\n    30, 7, 2, 1, 0, 12, 4, -21, -21, 4, -4, -8, -78, 1, 21, -5,\\n    -18, -1, -6, -6, -2, -28, -41, 16, -5, 3, 5, 14, 4, 33, -1, -59,\\n    12, -18, 33, -1, -8, -58, 1, 2, -22, -40, 33, -11, -37, -5, -2, 7,\\n    0, 5, 6, -37, 6, -59, -13, -101, -12, -1, 28, -63, 15, 44, 32, -33,\\n    -10, 41, -20, 20, 1, 33, 5, 36, 3, -24, -34, 20, -41, 35, -4, 3,\\n    23, 4, -20, 1, 0, -9, 7, 3, 77, 26, -6, 15, 2, -22, -21, -54,\\n    9, -19, 37, 0, 21, -50, -5, -8, 27, -29, -7, 33, 54, -20, -4, -15,\\n    -80, 16, 14, -11, -19, 77, -3, -9, -4, 1, -12, -31, -45, -15, -25, 10,\\n    0, 16, -8, -5, -3, 7, -33, 28, 12, -21, -14, 0, -28, 18, -37, 1,\\n    -13, 40, 32, -14, 20, 40, 3, -7, 63, 11, 2, -30, 19, 2, -2, -16,\\n    -24, -1, -15, 7, 49, 1, 10, -21, 22, 2, 23, -15, 1, -23, 4, 18,\\n    3, 35, 32, -29, 1, 39, 15, -5, -13, -34, -1, -1, 4, -71, 8, 2,\\n    23, -15, -22, -69, 29, 2, 6, 12, 75, 21, -2, 1, 25, -26, -7, -27,\\n    13, 0, 33, 27, -12, -1, -31, 23, -52, 32, 24, 10, -1, -80, -7, -3,\\n    -24, -40, 9, 22,\\n};\\n\\nconstexpr float kBlock1DepthwisePostScale[] = {\\n    1.87490129f, 10.4596481f, 3.98686075f, 5.51209021f, 6.745502f, 5.77950191f, 3.3182838f, 5.55035257f,\\n    11.0915041f, 2.98995519f, 5.26210499f, 3.21541762f, 9.23589134f, 6.50996494f, 3.19152594f, 3.95525122f,\\n    5.82840109f, 7.36820936f, 5.7966423f, 5.54240084f, 3.35115027f, 9.95029926f, 5.94380045f, 6.22574997f,\\n    3.90832758f, 15.806715f, 6.21412182f, 2.94623828f, 4.15315723f, 3.92249322f, 5.35770273f, 6.64803743f,\\n    8.88174248f, 5.22520018f, 5.7705369f, 5.39112186f, 7.07938337f, 7.40739775f, 3.23383498f, 11.1261959f,\\n    11.2096682f, 3.51935458f, 5.481112f, 10.4906149f, 7.20955229f, 4.30350256f, 8.39336777f, 3.38172364f,\\n    6.26644182f, 2.59070516f, 5.07297421f, 3.98120379f, 7.97404003f, 3.18994975f, 2.60691452f, 4.19642019f,\\n    2.88471079f, 2.76010752f, 5.26517963f, 8.09781742f, 10.2623358f, 5.22209787f, 10.3605318f, 2.05122423f,\\n};\\n\\nconstexpr float kBlock1DepthwisePostBias[] = {\\n    0.580448687f, -0.0533953011f, 0.340074778f, -0.974324644f, 0.938353419f, -1.37318182f, -0.720680654f, 0.174482703f,\\n    0.866952419f, 0.265644997f, -0.653986335f, 1.26581717f, 1.44154763f, -0.0960146487f, -0.942640781f, 1.075266f,\\n    1.23034668f, -0.186885759f, 0.738558173f, 0.051208593f, 0.983510971f, 0.657961607f, -0.691090107f, -0.470239937f,\\n    0.542274773f, -0.755909026f, -0.13205038f, 0.500155151f, 1.33762252f, 0.738905668f, 0.0705169439f, -0.360244811f,\\n    -0.526848316f, -0.156616986f, -1.05127466f, -0.812238455f, -0.250724167f, 0.415510476f, 0.775819659f, 0.109691009f,\\n    -0.488992065f, 1.12525678f, -0.301442623f, -0.106590897f, -0.185007468f, 0.112174466f, 0.688051462f, -0.111288592f,\\n    0.802076817f, 0.843747497f, 1.44839406f, 0.395824432f, -0.214762092f, 0.820253491f, 0.30429706f, -0.324611604f,\\n    -1.02491748f, 0.599660695f, -0.467704177f, 0.481366545f, -0.903297365f, -1.18917894f, -0.183106646f, -0.999897838f,\\n};\\n\\nconstexpr int8_t kBlock1PointwiseCodebook[] = {\\n    61, 11, -110, 62, 1, 17, -46, 41, -10, -13, -54, 12, -72, -39, 37, -17,\\n    40, 5, -25, 30, 11, -9, -51, -59, 45, -32, 23, -23, 36, 50, -27, -58,\\n    15, -30, -20, -1, -20, 6, 2, 28, -44, 55, -103, 115, 38, 7, -22, 88,\\n    -10, -41, 74, -24, 25, 20, 33, 90, 4, 38, 22, -31, -6, -5, -9, -85,\\n    -46, -49, -76, 34, -52, 7, 66, -61, -47, -56, -17, -21, 2, 14, -31, -40,\\n    -5, -18, 21, -41, -50, 39, 65, 4, 1, 14, 67, -81, 25, -54, -107, -24,\\n    6, -7, 23, -23, -37, -94, 17, 5, 57, -30, 2, -2, 9, -51, -84, 6,\\n    -33, 12, 25, 29, 37, -85, -22, 19, -1, 56, 21, 32, 0, 17, -78, -46,\\n    53, -37, -6, 13, -63, -107, 23, -12, 72, 24, -10, 8, 65, -50, -8, 13,\\n    -68, -21, 75, -27, 13, -10, 14, -34, 98, 13, 28, -18, 12, -56, 10, -66,\\n    -15, 75, 73, -7, 42, -52, 29, -27, 45, 23, 11, 15, 33, 17, -22, -54,\\n    -10, -56, 66, -41, 21, -37, -22, 4, 14, 0, -58, 40, -22, 35, 18, -32,\\n    -45, -103, 14, -9, -9, 41, -53, 0, 33, -4, -13, -30, 74, -4, 5, -36,\\n    72, -63, -32, 34, -23, -40, 20, -25, 106, 5, -22, -39, 55, 5, 14, -10,\\n    -56, 27, -34, 4, 80, 14, 14, 32, 31, 11, 50, 106, 31, 16, -34, -28,\\n    -2, -18, -30, -30, -6, -24, -1, 10, -41, 78, -22, 8, -38, 33, -72, -8,\\n    -112, 52, 19, 14, -6, 5, 27, -9, -27, 13, -29, 28, 8, 16, -32, -19,\\n    -50, 23, 41, -16, -100, 103, 17, 44, 6, -92, -25, 38, 24, 17, 92, -18,\\n    -20, 42, -29, -44, -21, 22, 53, 22, -4, -54, -44, 3, -61, -5, -38, 37,\\n    26, 35, 52, -108, 57, 48, -50, 24, 54, 2, -46, -13, -33, -32, -34, 18,\\n    -24, 22, -2, -11, -41, -33, 5, -17, 51, 9, -40, -35, -81, -15, -3, 12,\\n    -40, 17, -58, -55, 7, -29, 9, 23, 90, -33, 74, 39, -62, -5, 13, -40,\\n    6, 5, 28, 45, 31, -70, -37, -3, 19, -7, -40, -2, 107, -21, 6, 19,\\n    26, 20, 23, -39, -27, -15, -42, 10, -48, 27, -26, -99, -2, -22, 1, 46,\\n    28, 3, -57, -16, -5, 12, -55, -48, -127, -42, 6, -26, 1, 38, -25, -8,\\n    12, -2, -24, 4, -53, -25, -22, -15, -56, -24, -86, -14, 22, -18, -12, -26,\\n    8, -62, -56, -8, -85, -20, 14, -80, 1, 7, 22, -23, 65, -23, -13, 52,\\n    48, 45, 97, -1, 81, -14, 44, -22, -4, 33, -90, 82, -44, -17, -38, 37,\\n    35, 65, -46, 27, 39, -8, 12, -39, -37, 36, -40, -29, 38, -61, -78, 38,\\n    120, 27, -86, 23, -15, 67, -52, -42, -8, -20, -4, 4, 16, -31, -22, 36,\\n    47, -1, -70, -108, 35, -34, -32, 50, 7, -83, -52, 15, 25, -33, 42, -9,\\n    -15, 17, -10, -34,\\n};\\n\\nconstexpr float kBlock1PointwisePostScale[] = {\\n    1.81444561f, 1.00699008f, 2.15786862f, 1.60853553f, 0.905249238f, 1.47670281f, 1.27799475f, 2.59834099f,\\n    1.61510611f, 1.78073728f, 1.27251959f, 1.354406f, 2.62582612f, 2.26024175f, 1.65663898f, 1.51685083f,\\n    1.3951416f, 2.99982548f, 1.73952115f, 1.35256588f, 1.9218812f, 2.6328795f, 2.41017747f, 1.48590076f,\\n    2.10370755f, 1.49010825f, 1.78417194f, 1.78060722f, 1.93775332f, 1.87953937f, 1.35787117f, 1.81712472f,\\n    2.51677108f, 2.50829911f, 2.52792668f, 2.18649554f, 1.15372682f, 2.12546539f, 1.68953061f, 1.2726897f,\\n    2.05741334f, 2.59879613f, 1.63827467f, 1.30800152f, 2.08987999f, 2.15232635f, 1.8770566f, 1.64151597f,\\n    2.64708376f, 1.33586025f, 1.6092664f, 1.24965441f, 1.68913269f, 2.98841405f, 1.27083588f, 0.829288661f,\\n    1.99110675f, 2.76093721f, 2.25040627f, 1.90417898f, 2.0540247f, 1.15523803f, 1.69926906f, 1.85883689f,\\n};\\n\\nconstexpr float kBlock1PointwisePostBias[] = {\\n    0.102237031f, 0.244899094f, 0.593072474f, 0.55092144f, 0.87624532f, 0.0709686279f, 0.550380707f, -0.241746515f,\\n    0.287575573f, 0.166386977f, 0.663148522f, 0.0728797019f, -0.400592744f, 0.278929293f, 0.200490519f, 0.655445218f,\\n    0.865184903f, 0.671060503f, 1.3539865f, 0.383891433f, 0.96803385f, 0.323061287f, 0.832378745f, 0.530781865f,\\n    -0.12194407f, 0.479195118f, -0.109222949f, -0.14899531f, -0.350241035f, -0.134846866f, 0.308944643f, -0.114992693f,\\n    1.02723289f, 0.690830052f, 1.30601525f, 1.61844051f, 0.334667981f, 0.666499078f, 0.769368589f, 0.507541358f,\\n    0.358237982f, -0.722604632f, 0.366723776f, 0.263525546f, -0.0608666688f, 0.236482859f, 0.123221599f, -0.0601809025f,\\n    0.755111933f, 0.548665762f, 0.987858534f, 1.18394542f, 1.19390452f, 0.829452395f, 0.387864947f, -0.193210214f,\\n    -0.0945197493f, 0.510257125f, -0.0119548738f, 0.568246305f, -0.508119345f, -0.0327092409f, 0.630478978f, 0.760090828f,\\n};\\n\\nconstexpr int8_t kBlock2DepthwiseCodebook[] = {\\n    10, -18, -32, 3, 18, -9, 8, 40, 14, 13, -21, -14, -6, -25, 3, 19,\\n    16, 10, -31, -39, 15, 0, 34, 29, 82, 0, 9, 27, 0, -92, -10, -6,\\n    -1, 20, -15, -57, 1, 2, -22, 12, -39, 7, -61, -3, -23, -86, -17, 0,\\n    -19, 46, -8, -2, 14, 15, 1, 26, -75, -11, 0, 17, -32, 12, 2, -18,\\n    0, -4, 32, 13, 28, 1, 19, 1, -18, 5, -45, 16, -2, 23, -42, 12,\\n    0, 16, -2, -34, 1, -12, -8, -31, 9, 13, -4, -2, -120, 113, -35, -3,\\n    5, -1, 22, 24, 21, 1, 2, 5, 58, 8, -3, -35, -7, -33, 5, 7,\\n    -25, 2, 16, 35, -18, 5, 0, 31, 28, 23, 108, 24, -2, 27, -15, 10,\\n    12, 6, 12, 7, 69, -11, -33, 0, -37, 38, -15, 9, 18, -35, 13, 34,\\n    -19, 24, 2, -12, 25, 9, -20, 48, 0, -6, 26, -3, 20, 35, -21, 25,\\n    25, -29, -127, 4, -63, -29, 0, -13, 41, -21, -9, 2, 2, -25, 13, 1,\\n    29, -3, -1, 37, -22, 3, 16, -4, 3, 0, 31, 69, -2, 17, 65, 0,\\n    21, 20, 12, 3, 33, 11, -6, 0, 28, -12, -7, 2, 0, 22, -1, 6,\\n    -2, 7, -1, 19, 37, -26, -25, -1, 24, -1, 9, 13, 28, -1, -5, 13,\\n    -30, 2, 13, -12, -2, 63, -2, -12, 1, -26, 11, 5, -3, 33, -3, 0,\\n    -24, 37, -4, 1, 8, 30, -26, 6, 1, 17, -1, 6, 6, 41, -1, -4,\\n    -40, 40, -3, -21, 23, 26, -21, 53, 0, -2, 20, 18, -1, 1, 16, 39,\\n    18, 0, 16, 13, 0, 79, 21, 7, 2, 4, 32, 11, 22, -16, 18, 0,\\n    -29, 53, 41, 14, 10, 33, 14, 15, -62, 14, -1, 12, 37, 32, 26, 26,\\n    31, 25, -29, -27, -9, 3, 20, 10, -1, -12, -36, 0, 24, -66, 8, 28,\\n    -33, 15, -32, 5, -38, -25, 15, -34, 21, 0, 1, -7, 33, 20, 5, -36,\\n    -33, 3, 50, -27, -5, 20, -10, 2, 27, 7, 4, -11, 22, -16, -45, 0,\\n    3, -21, 15, -40, -9, 27, 2, -5, 46, -1, 4, 17, -45, -26, 46, 3,\\n    -56, 0, -10, 57, 15, 2, -1, 58, -46, 3, 3, 13, -1, -10, 36, 7,\\n    1, -18, 8, -1, -2, -7, -19, -1, 5, 26, -27, 4, -4, 5, -33, -3,\\n    12, -24, 4, -22, 4, -3, 1, -16, 6, 15, 0, 22, -20, -2, 10, 33,\\n    -20, 2, -14, 19, -25, 1, 16, -3, 20, 21, 13, 4, 1, 13, -58, -15,\\n    3, 17, -26, 10, 35, 0, 6, 0, -7, -37, -26, 0, 9, 8, -4, -41,\\n    8, 29, 0, 40, 12, 22, 11, -26, 12, 1, 29, -58, 7, 0, 14, -33,\\n    0, 6, -25, 4, -38, 44, -3, 3, 0, -26, -11, -16, -11, -5, -3, -47,\\n    36, 43, 9, -37, 49, 5, 11, 38, -27, 6, -35, 40, 1, -23, 14, -39,\\n    98, -4, 30, -8,\\n};\\n\\nconstexpr float kBlock2DepthwisePostScale[] = {\\n    5.03933477f, 20.0057278f, 4.22439861f, 11.0018511f, 12.9953775f, 11.5630722f, 4.24355888f, 7.48181677f,\\n    4.8608923f, 5.3580327f, 8.13385296f, 3.09577847f, 3.04445338f, 2.28423905f, 8.92820835f, 5.8953743f,\\n    3.96773529f, 5.15678453f, 5.49111509f, 8.00524521f, 6.60096312f, 2.65551376f, 2.93100142f, 5.20620775f,\\n    4.60235405f, 7.35284615f, 6.58115864f, 3.60529637f, 6.72038269f, 4.86007023f, 3.74246335f, 3.15845799f,\\n    2.97316909f, 3.00777841f, 2.67517924f, 5.82055616f, 9.11527157f, 4.17927408f, 5.05801821f, 10.2336712f,\\n    8.30457783f, 7.67280579f, 4.28731537f, 5.45331955f, 6.20294237f, 6.86949492f, 7.5867753f, 6.39641619f,\\n    4.33859777f, 3.69229388f, 9.18460083f, 9.47565174f, 5.47909451f, 11.5052776f, 6.79224539f, 25.3808899f,\\n    5.38979578f, 4.59287548f, 1.82244444f, 6.20658827f, 6.1426568f, 10.512682f, 3.89569831f, 3.88579416f,\\n};\\n\\nconstexpr float kBlock2DepthwisePostBias[] = {\\n    0.529336929f, -0.156579062f, -1.43080688f, -1.0481925f, -0.219809666f, -0.780450463f, 0.960234761f, 0.472196162f,\\n    -0.46082285f, -1.318483f, 1.36442947f, -1.00581408f, -0.465257734f, 0.581052423f, -0.971039295f, -0.772114158f,\\n    0.43461597f, -0.800473988f, -1.0051502f, 0.130487159f, 0.273752332f, 0.517573357f, 0.386102378f, -1.35124826f,\\n    0.824612319f, -0.396292508f, -0.632907093f, -0.605561495f, -0.75144875f, -0.0175121874f, -0.700262129f, 0.734605908f,\\n    -1.1907177f, -1.03013706f, 0.583579302f, -1.12333333f, -0.704696357f, 0.398414493f, -1.09020376f, 0.714832783f,\\n    -0.043459937f, -0.551738203f, -1.15725648f, 0.869744897f, -0.933342576f, -1.03283012f, -1.23842072f, -0.611661077f,\\n    -0.883783877f, -0.628685832f, -0.0633256733f, -0.339599133f, -0.908749759f, -1.13798451f, 0.070371598f, -0.813390017f,\\n    0.537449658f, 0.67638123f, 0.767539263f, 0.0436912179f, 0.250026286f, 0.725908518f, -0.731061816f, -0.0281604361f,\\n};\\n\\nconstexpr int8_t kBlock2PointwiseCodebook[] = {\\n    48, 16, 13, 10, -24, 12, -87, -7, -16, -40, -11, -44, -3, -6, 20, -36,\\n    -28, -17, -40, -30, -5, 28, -13, 0, 50, 40, -9, 35, 10, 24, 2, -14,\\n    27, 27, 0, -35, 1, 31, -6, -11, 20, -6, -37, 7, -14, 26, 30, -8,\\n    -18, 19, -50, -37, -6, 21, -17, -9, -47, -76, -11, -30, 44, -7, 10, -20,\\n    -4, 15, -22, 19, -3, 17, 18, 1, -41, -43, -39, 14, -35, 7, 1, 18,\\n    16, -29, -18, 5, 2, -32, 20, 10, -51, 3, -20, -5, -47, 28, -41, 15,\\n    -16, 10, -3, -127, 25, -10, 32, -14, -7, -5, 1, -10, -36, 14, -8, 0,\\n    -2, 25, -46, 6, -53, 4, 14, 7, 36, 35, 13, 26, 27, -3, 16, 17,\\n    6, -3, 24, -25, 1, -31, 2, -13, -12, -6, 0, 51, -37, -35, -9, 16,\\n    -32, -74, 30, -20, 14, 17, -27, -61, 16, -5, 22, -2, 27, 75, -37, -9,\\n    -3, 54, -36, 15, 16, -7, 39, 3, 49, -25, -11, -54, 5, -30, 9, 0,\\n    28, 47, -3, 0, -10, -33, -15, 8, 6, 19, -9, -13, -18, 56, 63, -37,\\n    12, -16, 43, -61, -28, 37, -12, 30, -44, 31, -25, -71, -66, 23, -4, 48,\\n    7, 36, -53, 6, -70, -65, -18, -20, 24, 17, -36, 28, -22, 17, -46, 36,\\n    5, -8, 6, 24, 1, -30, -23, -48, 16, -40, -18, -5, -31, -8, 18, 29,\\n    43, -10, -48, -1, 26, 42, 35, -23, -21, -35, 32, -30, 20, -21, 19, -33,\\n    19, -2, -11, -47, -5, -1, 32, -55, 17, -66, 28, -24, -30, -34, 6, -25,\\n    -3, -19, 12, 55, 19, 20, 23, -30, 49, -34, -11, 42, 33, -29, -5, -1,\\n    15, -18, -46, 12, -17, 18, 3, -53, 2, 17, -62, -26, -12, -14, 9, -5,\\n    24, 13, -5, -9, -28, -11, -14, 3, -23, 15, 19, 32, -42, 7, -23, -41,\\n    19, -74, 32, 23, -15, -43, 10, -1, 10, 24, -46, -42, -60, -2, -22, 43,\\n    19, 28, -33, -6, 35, -9, 14, -56, 1, 37, 61, -55, -12, -36, -17, -101,\\n    -6, 12, 9, -51, 1, -8, -45, 17, 30, 17, 17, -30, 37, -17, 31, -13,\\n    19, 5, -4, 25, -3, -15, -6, -45, 19, -46, -28, -53, -19, 19, 11, 3,\\n    -2, -3, -45, -1, 28, 16, 52, -39, 14, 56, 16, 23, 10, -53, 21, -2,\\n    -15, 10, 20, -22, -61, 44, 54, -5, -11, -41, -22, 5, -10, -39, 12, -38,\\n    -55, -38, -34, -14, -31, 19, 7, 39, 27, -10, 37, -20, 31, -2, -12, -20,\\n    -17, -9, 8, -33, 24, 13, -35, -13, -26, -37, 13, -1, -10, -27, 5, -57,\\n    -12, -14, 44, -55, -6, -9, 47, -53, -15, -57, -65, -33, -5, -7, -9, 8,\\n    -28, -26, 12, 54, -29, 1, -6, 57, -11, 56, 17, 93, 19, -2, 14, -11,\\n    -30, -46, -57, -17, 2, 56, 10, -5, -25, 31, 21, -3, 72, 20, -15, -49,\\n    -22, -31, -33, 28,\\n};\\n\\nconstexpr float kBlock2PointwisePostScale[] = {\\n    0.917589962f, 1.5343672f, 1.35226834f, 0.467993796f, 1.65300095f, 1.70264161f, 1.9847306f, 0.870355487f,\\n    1.4043839f, 1.4016788f, 1.3734827f, 1.46554267f, 1.64480662f, 1.44888711f, 0.935991824f, 1.35799372f,\\n    2.13785887f, 1.47865438f, 1.23269629f, 1.60122144f, 0.938421667f, 1.58028722f, 0.974500716f, 1.56390715f,\\n    0.704336464f, 1.11231947f, 1.08485305f, 0.974869072f, 1.47862446f, 0.396859139f, 2.14036369f, 1.19694316f,\\n    1.33339155f, 1.59828162f, 0.992692411f, 1.17569268f, 1.25689018f, 0.821860492f, 1.65237045f, 0.725502133f,\\n    1.90004826f, 1.07068574f, 0.773956716f, 1.14313674f, 1.130234f, 1.49158537f, 1.71379328f, 2.25947237f,\\n    0.600569487f, 0.950667739f, 1.13199639f, 1.47329819f, 2.47656465f, 0.896393418f, 0.771206319f, 1.75514936f,\\n    0.833865821f, 1.12379396f, 0.821831107f, 0.785467565f, 1.26047158f, 2.06254458f, 0.834910631f, 2.09306097f,\\n};\\n\\nconstexpr float kBlock2PointwisePostBias[] = {\\n    0.213708699f, 0.846419215f, 0.197556585f, 0.893969655f, 0.16404666f, 0.666876137f, 1.20276773f, -0.122985303f,\\n    0.328231454f, 0.0256122649f, 0.426726669f, -0.0638471246f, 0.774533093f, 0.463900685f, -0.0912209451f, 0.726246178f,\\n    -0.231274605f, 1.09037578f, 0.244924873f, 0.0634472817f, 0.507350087f, 0.457049012f, 0.03679353f, 0.802262783f,\\n    -0.506761551f, 0.115521044f, 0.0505770743f, -0.504822433f, -0.228000879f, 0.73946166f, 0.332238078f, 0.628765821f,\\n    0.190469503f, -0.0326234996f, 0.493581802f, 0.616253018f, 0.9371562f, 0.991819978f, 0.457627416f, 0.120262861f,\\n    0.865105867f, 0.436968893f, 0.0887439549f, -0.111808807f, -0.068948403f, 0.266554624f, 0.528472304f, 0.459310383f,\\n    0.709913611f, 0.165424436f, -0.180991217f, 1.26314032f, 1.09130299f, 0.345013618f, 0.138863236f, 0.148583561f,\\n    -0.324238598f, -0.177376449f, -0.262843162f, -0.206654608f, 0.155639619f, -0.0349825323f, 0.626667738f, 0.392569423f,\\n};\\n\\nconstexpr int8_t kBlock3DepthwiseCodebook[] = {\\n    4, 0, -1, 31, 17, -1, 6, 32, 13, -1, -25, 15, 0, 17, 65, 2,\\n    1, -7, 24, 10, -1, 13, 18, -1, -22, -14, 24, -2, -5, 9, 6, 1,\\n    8, 24, 3, 0, 16, 0, 3, -13, 38, 12, 2, -28, 2, 18, 3, -23,\\n    11, 1, 14, 27, 11, -4, -13, 15, -25, 7, -18, -1, 2, 20, 23, 19,\\n    2, 10, 16, 12, -9, -86, 1, -5, 24, -81, 24, 1, 14, -14, 3, 14,\\n    -54, 11, -3, -22, 24, -5, -127, 15, 9, 18, -12, -60, 21, 1, 16, -37,\\n    10, -13, -18, 12, -58, -19, 18, 22, -2, -46, -26, 17, 16, -46, 12, 32,\\n    -19, -3, -14, -13, -18, 4, 11, -9, -44, 15, 18, 45, 0, 11, -18, -23,\\n    -31, 8, -10, 16, 13, 14, -8, 0, 5, 14, 1, 17, 15, -63, 3, -10,\\n    19, 34, 0, 19, -66, -5, 15, 9, 4, -2, -87, 20, 33, 1, -24, -12,\\n    17, 8, 11, 12, -3, 17, -26, 22, -3, 25, 1, 17, -22, 18, -11, -3,\\n    -31, 0, -7, 1, 20, 20, -32, -2, 7, -48, 1, -10, 26, 5, 1, 17,\\n    24, -32, 2, -19, 5, 1, 15, -44, -7, -1, 14, 40, 18, 3, -52, -8,\\n    0, -34, 26, -6, 3, 8, 16, 26, 1, 17, 13, -41, 14, -14, 10, 2,\\n    -27, 14, 9, 3, -58, -15, -23, 13, -44, 24, 3, -79, 12, -32, -2, -5,\\n    -5, 13, 12, 14, 34, -2, 22, 32, 14, -121, 11, 9, -10, -25, 18, 20,\\n    2, -78, -10, 0, 15, 18, 14, -6, 30, 32, 16, 0, -17, -5, 9, 4,\\n    8, 16, -20, -7, -24, 3, -123, -16, 15, -6, -17, 9, 17, 15, 9, -2,\\n    -34, 47, -37, 27, -11, 1, 9, 10, 14, -9, 3, 15, 17, 14, 9, -20,\\n    -23, -4, 14, 26, -36, -3, 4, -41, 16, 20, 18, -5, 2, -43, 14, -13,\\n    1, -22, 24, 12, 14, 7, -16, -1, -4, 17, -19, 2, 9, -12, 2, 6,\\n    12, -7, 0, -9, -4, 26, 0, 14, -20, 0, 2, 14, -30, 1, -92, 32,\\n    3, -5, -9, 3, -22, -1, 5, -36, 1, -26, -15, -5, -2, 11, -10, -11,\\n    0, 6, 11, -1, 18, 0, 20, 3, 7, -8, -43, 4, -1, 13, -58, 24,\\n    -13, -3, 1, 22, -34, -38, -1, -13, -7, 34, 13, 20, -11, 4, 18, 19,\\n    1, 1, 0, 13, 18, 12, 22, -5, 1, 47, 20, -6, 16, -48, 11, -8,\\n    14, 21, -7, 0, -53, -31, 13, 28, 11, 11, 10, 28, 16, -9, 0, -45,\\n    19, 7, 4, 38, -4, -16, -20, 28, 14, 25, -29, -12, -10, -45, 13, 4,\\n    -88, 34, -1, 10, 34, 14, -15, 13, 9, 11, -11, -29, 15, -2, 15, 9,\\n    21, 13, -5, -5, 14, 11, -45, 10, 2, -38, -74, 19, 23, -6, 22, 1,\\n    15, -29, 30, 0, 9, 12, 14, -1, -7, 17, 0, 5, 28, 17, 1, 4,\\n    -15, 8, 9, 7,\\n};\\n\\nconstexpr float kBlock3DepthwisePostScale[] = {\\n    48.2479591f, 1.1361258f, 2.74456453f, 4.51871872f, 3.65407801f, 3.25323129f, 9.80417538f, 33.630722f,\\n    6.64637518f, 1.46517742f, 3.83691955f, 2.95574212f, 3.30374098f, 3.08033037f, 11.3427334f, 3.80424666f,\\n    2.98820734f, 3.39833713f, 6.48718786f, 3.87281108f, 8.80635357f, 8.14434338f, 5.95369101f, 2.30292153f,\\n    71.988327f, 2.85812426f, 11.8159313f, 14.5890779f, 2.78309798f, 10.6223497f, 3.28235292f, 6.26199913f,\\n    11.6895971f, 5.54001188f, 15.7006702f, 4.83125925f, 1.50062931f, 4.33848381f, 3.44506788f, 20.6150627f,\\n    2.94674921f, 4.9861393f, 1.50521648f, 24.7328014f, 5.9028697f, 4.71634865f, 3.28635979f, 5.41680193f,\\n    6.83483791f, 14.1722698f, 16.5551777f, 3.37302399f, 2.44272923f, 5.44459295f, 49.0423393f, 2.26029682f,\\n    70.4194183f, 30.2523785f, 7.77053499f, 60.9864349f, 11.7592287f, 9.92189121f, 4.14502811f, 2.77538276f,\\n};\\n\\nconstexpr float kBlock3DepthwisePostBias[] = {\\n    -0.790444493f, 0.925536871f, -0.18222937f, -1.40155089f, -0.966346145f, 0.361792177f, -1.39470685f, -0.654981732f,\\n    -1.00219321f, -0.693857908f, 0.490018666f, -0.0742314756f, -1.11325002f, -0.981051862f, 0.280505419f, 0.337277681f,\\n    0.0421913862f, 0.282387108f, -1.25069523f, 0.0680572689f, -0.926056385f, -0.942725897f, -0.157553881f, -1.18951535f,\\n    -0.649757981f, -0.265133917f, -0.84862721f, -0.920262098f, 0.105768502f, -1.2098825f, -1.09027123f, -0.491835475f,\\n    -0.902608395f, -0.067476511f, -1.47442913f, 0.2813344f, 0.501612961f, -1.24850726f, -0.957023501f, -0.804628551f,\\n    0.431592047f, 1.22569656f, -0.452299625f, -1.13961697f, 0.733981907f, -0.191225499f, 0.293298364f, 0.373086333f,\\n    -1.33171797f, -0.0417872667f, -1.08464348f, 0.1449112f, 0.444393933f, -1.2044642f, -0.391246527f, 0.00665435195f,\\n    -0.649320126f, -0.675077617f, 0.0645111501f, -1.0295577f, -0.69645375f, -0.82826221f, -1.17562747f, 0.494880885f,\\n};\\n\\nconstexpr int8_t kBlock3PointwiseCodebook[] = {\\n    32, 84, 17, 16, 48, 16, 43, 28, 9, -6, -20, 18, -32, 7, 115, 25,\\n    9, 18, -22, 7, -4, 15, 3, -8, -19, -9, 62, -23, 3, 21, 3, 34,\\n    24, 1, -5, 13, -27, 123, 1, 6, 32, 18, -16, -5, -11, -4, 44, -11,\\n    -10, 20, -5, -30, 7, 7, 89, -22, 33, 7, 5, 42, -3, -27, 0, -21,\\n    13, 65, 43, -7, 28, -18, 49, -19, 20, -9, 1, 0, -25, -10, 5, -49,\\n    6, -10, 2, -14, -22, -27, 22, 16, -33, 32, -32, 6, -44, -14, -13, -3,\\n    25, 0, 80, -12, -35, 41, 4, -11, -38, -31, -44, -6, -22, -9, -24, -7,\\n    -37, -34, 35, -9, -30, 6, -18, 10, 26, 7, 78, 2, 9, 99, 1, -10,\\n    18, -11, -15, 18, -16, 76, 9, -11, -66, -40, 5, 31, -5, 32, 8, -15,\\n    32, -12, -5, -59, -10, 6, 82, 13, -8, -13, -6, -32, 43, 0, -27, 61,\\n    -33, -9, 7, 16, 45, 49, 31, 0, -20, 4, 5, 12, -5, -32, 9, 49,\\n    16, -11, -16, -11, -46, -44, -1, -13, 42, 17, 9, 42, 5, -25, -26, 14,\\n    2, 28, 17, -30, 57, -2, 46, 24, -5, -22, 32, -59, 29, -11, 0, 51,\\n    -5, 9, -5, 33, -7, -4, -1, -13, -38, 44, 47, -34, -4, 8, 74, -11,\\n    89, -24, 3, -2, -34, -41, 10, -1, -12, 67, 37, 12, 2, 4, -10, 4,\\n    -42, -46, -19, -10, -68, 28, -21, 18, 15, 23, 127, -27, 6, 48, -17, -3,\\n    8, 15, -1, 4, 9, 3, -5, -12, 22, -18, -31, -44, -14, -53, -16, -11,\\n    7, 3, -8, -32, -15, -10, 25, -31, -16, 29, 38, -11, 21, 69, 0, 7,\\n    -69, -23, 32, 19, -4, 62, 36, -22, 64, -16, -5, -36, 17, 7, -24, 3,\\n    1, 19, -16, -9, -11, -9, -61, 10, -6, -15, -12, 8, 5, 13, 28, 12,\\n    -8, 1, 27, 3, 3, -41, -12, -4, 20, -18, 16, -17, -7, -29, 26, 96,\\n    15, -13, 10, 55, -17, -24, -6, 55, 3, -9, 8, 11, -14, -13, 23, 55,\\n    11, 2, 25, 7, 21, -6, -34, -9, -2, 3, 75, 88, -19, -11, 36, -11,\\n    28, -14, 25, -13, 5, -23, 46, 28, -8, -40, 5, -28, -29, -4, -25, 4,\\n    -26, -5, 19, -5, -2, -2, -14, 20, -40, 50, 37, 4, 4, 20, -32, -11,\\n    -14, 23, 51, -30, 120, 50, -43, -44, 11, -10, 29, -22, 27, 9, 0, 5,\\n    13, 10, -20, 2, 24, 16, -22, 2, -14, -5, -7, -38, 3, -4, 11, -10,\\n    -26, -21, 23, 4, 2, 7, 5, -10, -37, -9, -17, -11, -8, 6, -17, -5,\\n    18, -30, 4, 17, -14, 30, 37, -13, 37, 36, -19, 24, -47, -16, -3, 49,\\n    -37, -18, 17, 40, -13, 55, 21, 66, -6, 3, 49, -39, 21, 5, 53, 61,\\n    12, 47, 16, 1, 30, -23, -33, 68, -14, 43, -17, -3, 42, -28, -20, 23,\\n    -12, 12, 121, 37,\\n};\\n\\nconstexpr float kBlock3PointwisePostScale[] = {\\n    5.83157015f, 6.38527012f, 8.65911865f, 4.59118319f, 5.10033369f, 6.8798089f, 12.0073042f, 5.71038532f,\\n    5.61795759f, 10.6019421f, 8.45257473f, 6.35755062f, 6.3843174f, 5.12247467f, 3.85368967f, 6.04757929f,\\n    5.42861128f, 7.09815025f, 3.88781452f, 9.77070332f, 6.94693232f, 8.35229206f, 6.86954832f, 7.0542655f,\\n    5.79902744f, 6.28863478f, 7.39705896f, 8.31702232f, 5.31456614f, 6.48951387f, 5.08287191f, 4.92903137f,\\n    5.99196815f, 7.34049892f, 7.91790438f, 10.0643339f, 8.44652939f, 10.1454306f, 10.1304874f, 6.91457939f,\\n    9.34891129f, 8.42470264f, 6.4563117f, 6.45275688f, 7.4738307f, 4.98658276f, 4.70957518f, 5.803514f,\\n    10.2612839f, 5.40911341f, 6.83122015f, 8.63717175f, 6.2925868f, 5.90203238f, 9.25614834f, 5.10992765f,\\n    6.74563408f, 8.78823471f, 11.5108643f, 4.96067238f, 5.74043322f, 8.53876209f, 7.24699402f, 5.82911921f,\\n};\\n\\nconstexpr float kBlock3PointwisePostBias[] = {\\n    0.261763781f, 0.809175849f, -1.02241278f, -0.21987325f, 0.38150239f, -1.62890375f, -0.96449101f, -0.370645255f,\\n    -0.588965535f, -1.88079321f, -0.0874642134f, -0.158610508f, -0.000660598278f, -0.51749444f, -0.214680582f, -0.263795972f,\\n    -0.00769367814f, -0.927231073f, -0.675605237f, -0.825585723f, -0.389058679f, -0.29807362f, -1.09560323f, -0.53105998f,\\n    -1.54766166f, -1.44800377f, -0.791611671f, -0.683110118f, -1.63014197f, -0.87794131f, -0.717519641f, -1.03055036f,\\n    -1.2718116f, -0.572460473f, -1.46188986f, -0.929856598f, -0.670378685f, 0.0288996696f, -0.70103991f, -0.819942296f,\\n    0.0135489702f, -0.454942465f, -1.13333178f, -0.296949685f, -0.190105379f, -0.996610045f, -1.06896853f, -0.221730649f,\\n    -0.467667222f, -1.04693758f, -0.392071307f, -0.549274802f, -0.508724272f, -0.710817218f, -0.0807201862f, -0.389961123f,\\n    -0.856827736f, 0.381766379f, 0.194760919f, -0.967772663f, -0.608191907f, 0.966904104f, -0.77391392f, -1.11513257f,\\n};\\n\\nconstexpr int8_t kClassifierCodebook[] = {\\n    0, 1, -102, 23, -40, -85, 81, -2, 17, 7, -19, 16, 13, -53, 1, 1,\\n    0, 24, 6, -75, -35, -18, -36, -62, -93, -10, 18, -47, -28, -33, -49, 16,\\n    -30, 83, -92, 6, -11, -55, 127, -61, 16, 27, 13, -58, -69, -19, 61, -48,\\n    -45, 25, 31, -89, -10, 28, -14, -76, -10, -42, -52, 4, -10, -60, -52, -71,\\n    -58, 123, -70, 0, 0, -37, 1, -12, -76, 7, -66, 83, 6, 3, 43, 17,\\n    -22, -1, 0, 8, 30, -79, 90, -45, 52, 86, 53, -49, -68, -85, -75, 0,\\n    2, -50, 60, 75, -23, -63, -68, 7, 2, 8, -45, -38, 10, -6, -44, 69,\\n    13, -29, -26, -9, -19, -4, 52, 47, -26, 64, -79, -25, -18, 101, -28, 12,\\n    -54, -46, -21, -66, -39, -23, -49, -3, 31, 23, -24, -42, 48, 48, -22, -15,\\n    5, -46, 120, 67, 1, -1, -55, -23, -37, -35, -45, -54, 92, 41, -115, -35,\\n    -45, 16, 0, 0, 64, 7, -91, -48, -57, -74, -62, -30, 93, -3, -76, 9,\\n    1, -1, -73, 21, 20, 41, -37, -78, -64, -21, 87, -6, 60, -13, -1, -6,\\n    -73, 68, 5, -46, -15, -42, 4, -58, -21, -29, 29, -77, -22, -23, 83, 61,\\n    31, 96, -83, -18, -36, 119, 81, 20, -81, 64, -1, 6, -48, -34, 118, -36,\\n    -2, 48, 66, -60, -38, -1, 0, -74, 27, -60, -25, -41, 14, -73, 23, -78,\\n    -3, -30, 16, 1, -1, -79, -21, -53, 15, -43, -76, -41, -26, -76, -12, -39,\\n    -16, 0, 2, -64, 12, -10, -42, -47, 18, 69, -73, -31, -40, 53, 4, 0,\\n    -13, -53, -28, 123, -70, -42, -39, 57, -99, -66, -56, -35, -11, 17, -16, -41,\\n    26, -2, -47, -45, -59, -81, -27, 36, 35, 19, 50, -12, -32, -30, -21, -2,\\n    73, -37, -57, 3, -30, -4, -30, -1, 21, -5, -6, -69, 50, 88, -50, -4,\\n    -73, -61, -21, -24, 0, -1, -103, -1, -49, 87, -11, -27, 85, -35, -82, -19,\\n    -33, -36, -1, 0, -40, 1, -40, -60, -16, -47, -69, 39, -36, -9, -80, -34,\\n    0, -2, -42, 41, 50, 3, -23, -56, -32, -59, 96, -23, 45, 38, -13, 7,\\n    -118, -37, 29, 68, 22, 10, -24, -61, -11, -45, -11, -52, 19, -50, -67, 15,\\n    34, 10, -47, -96, -33, 97, -62, -10, 0, 36, -7, -44, -38, -30, -25, -50,\\n    2, 10, -42, -30, -65, 0, 0, -80, 6, 35, -4, -54, 9, 71, -57, 8,\\n    -38, -14, -36, 0, 1, -49, 30, -63, -88, -56, -50, 17, 4, -51, 25, -59,\\n    -42, 0, -1, -31, 55, -42, -81, -58, 3, -18, -25, 48, -52, 42, -66, -5,\\n    -39, -58, 59, 56, 106, -52, 14, 71, -10, 51, -34, -57, -1, 10, -74, -23,\\n    83, 91, -53, 74, -1, -36, 54, 40, -2, -57, -59, -52, -29, -38, -65, -39,\\n    35, -14, -101, -69, -27, -47, 0, -1, 27, 2, -12, -83, -29, 43, 7, -42,\\n    83, 41, 36, 29,\\n};\\n\\nconstexpr float kClassifierBias[] = {\\n    0.0167008918f, -0.124071032f, 0.115727998f, 0.10656894f, 0.46967721f, -0.215417817f, -0.266257524f, -0.221629858f,\\n    -0.132646248f, 0.18248111f, 0.0445794202f, -0.191212162f,\\n};\\n\\n}  // namespace\\n\\nconst HashDscnnModelData g_hash_model = {\\n    true,\\n    40,\\n    49,\\n    1,\\n    64,\\n    4,\\n    12,\\n    {\\n    kStemCodebook,\\n    0.00486822016f,\\n    kStemPostScale,\\n    kStemPostBias,\\n    500,\\n    1,\\n    64,\\n    3,\\n    3,\\n    2,\\n    2,\\n    1,\\n    1,\\n    0,\\n    false,\\n},\\n    {\\n        {\\n    kBlock0DepthwiseCodebook,\\n    0.00470016885f,\\n    kBlock0DepthwisePostScale,\\n    kBlock0DepthwisePostBias,\\n    500,\\n    64,\\n    3,\\n    3,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    false,\\n},\\n        {\\n    kBlock1DepthwiseCodebook,\\n    0.00377819083f,\\n    kBlock1DepthwisePostScale,\\n    kBlock1DepthwisePostBias,\\n    500,\\n    64,\\n    3,\\n    3,\\n    1,\\n    1,\\n    1,\\n    1,\\n    3,\\n    false,\\n},\\n        {\\n    kBlock2DepthwiseCodebook,\\n    0.00416722683f,\\n    kBlock2DepthwisePostScale,\\n    kBlock2DepthwisePostBias,\\n    500,\\n    64,\\n    3,\\n    3,\\n    1,\\n    1,\\n    1,\\n    1,\\n    5,\\n    false,\\n},\\n        {\\n    kBlock3DepthwiseCodebook,\\n    0.00515738388f,\\n    kBlock3DepthwisePostScale,\\n    kBlock3DepthwisePostBias,\\n    500,\\n    64,\\n    3,\\n    3,\\n    1,\\n    1,\\n    1,\\n    1,\\n    7,\\n    false,\\n}\\n    },\\n    {\\n        {\\n    kBlock0PointwiseCodebook,\\n    0.00254094765f,\\n    kBlock0PointwisePostScale,\\n    kBlock0PointwisePostBias,\\n    500,\\n    64,\\n    64,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    2,\\n    false,\\n},\\n        {\\n    kBlock1PointwiseCodebook,\\n    0.00229064991f,\\n    kBlock1PointwisePostScale,\\n    kBlock1PointwisePostBias,\\n    500,\\n    64,\\n    64,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    4,\\n    false,\\n},\\n        {\\n    kBlock2PointwiseCodebook,\\n    0.00352387987f,\\n    kBlock2PointwisePostScale,\\n    kBlock2PointwisePostBias,\\n    500,\\n    64,\\n    64,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    6,\\n    false,\\n},\\n        {\\n    kBlock3PointwiseCodebook,\\n    0.00330732376f,\\n    kBlock3PointwisePostScale,\\n    kBlock3PointwisePostBias,\\n    500,\\n    64,\\n    64,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    8,\\n    false,\\n}\\n    },\\n    {\\n    kClassifierCodebook,\\n    0.0116869953f,\\n    kClassifierBias,\\n    500,\\n    64,\\n    12,\\n    9,\\n    false,\\n},\\n    {\\n        {1.00787402f, 0.0428081798f},\\n        {0.0428081798f, 0.0864292655f},\\n        {0.0864292655f, 0.0644126126f},\\n        {0.0644126126f, 0.141896045f},\\n        {0.141896045f, 0.0904647121f},\\n        {0.0904647121f, 0.121194697f},\\n        {0.121194697f, 0.0694761426f},\\n        {0.0694761426f, 0.370951555f},\\n        {0.370951555f, 0.626048111f}\\n    },\\n};\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_recognize_commands.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n\\n#include <cstdint>\\n\\n#include \\"hash_model_settings.h\\"\\n#include \\"tensorflow/lite/c/common.h\\"\\n#include \\"tensorflow/lite/micro/micro_error_reporter.h\\"\\n\\nnamespace hash_kws {\\n\\nclass PreviousResultsQueue {\\n public:\\n  explicit PreviousResultsQueue(tflite::ErrorReporter* error_reporter)\\n      : error_reporter_(error_reporter), front_index_(0), size_(0) {}\\n\\n  struct Result {\\n    Result() : time_(0), scores() {}\\n    Result(int32_t time, const int8_t* input_scores) : time_(time) {\\n      for (int i = 0; i < kCategoryCount; ++i) {\\n        scores[i] = input_scores[i];\\n      }\\n    }\\n    int32_t time_;\\n    int8_t scores[kCategoryCount];\\n  };\\n\\n  int size() const { return size_; }\\n  int capacity() const { return kMaxResults; }\\n  bool empty() const { return size_ == 0; }\\n  Result& front() { return results_[front_index_]; }\\n  Result& back() {\\n    int back_index = front_index_ + (size_ - 1);\\n    if (back_index >= kMaxResults) {\\n      back_index -= kMaxResults;\\n    }\\n    return results_[back_index];\\n  }\\n\\n  void push_back(const Result& entry);\\n  Result pop_front();\\n  Result& from_front(int offset);\\n\\n private:\\n  tflite::ErrorReporter* error_reporter_;\\n  static constexpr int kMaxResults = 50;\\n  Result results_[kMaxResults];\\n  int front_index_;\\n  int size_;\\n};\\n\\nclass HashRecognizeCommands {\\n public:\\n  explicit HashRecognizeCommands(tflite::ErrorReporter* error_reporter,\\n                                 int32_t average_window_duration_ms = 1000,\\n                                 uint8_t detection_threshold = 180,\\n                                 int32_t suppression_ms = 1500,\\n                                 int32_t minimum_count = 3);\\n\\n  int category_count() const { return kCategoryCount; }\\n\\n  TfLiteStatus ProcessLatestResults(const int8_t* latest_results,\\n                                    int latest_results_size,\\n                                    int32_t current_time_ms,\\n                                    const char** found_command,\\n                                    uint8_t* score,\\n                                    bool* is_new_command);\\n\\n private:\\n  tflite::ErrorReporter* error_reporter_;\\n  int32_t average_window_duration_ms_;\\n  uint8_t detection_threshold_;\\n  int32_t suppression_ms_;\\n  int32_t minimum_count_;\\n\\n  PreviousResultsQueue previous_results_;\\n  const char* previous_top_label_;\\n  int32_t previous_top_label_time_;\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n", "code/firmware/hash_kws_runtime/hash_recognize_commands.cpp": "#include \\"hash_recognize_commands.h\\"\\n\\n#include <limits>\\n\\nnamespace hash_kws {\\n\\nvoid PreviousResultsQueue::push_back(const Result& entry) {\\n  if (size() >= kMaxResults) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Couldn\'t push_back latest result, too many already!\\");\\n    return;\\n  }\\n  size_ += 1;\\n  back() = entry;\\n}\\n\\nPreviousResultsQueue::Result PreviousResultsQueue::pop_front() {\\n  if (size() <= 0) {\\n    TF_LITE_REPORT_ERROR(error_reporter_, \\"Couldn\'t pop_front result, none present!\\");\\n    return Result();\\n  }\\n  Result result = front();\\n  front_index_ += 1;\\n  if (front_index_ >= kMaxResults) {\\n    front_index_ = 0;\\n  }\\n  size_ -= 1;\\n  return result;\\n}\\n\\nPreviousResultsQueue::Result& PreviousResultsQueue::from_front(int offset) {\\n  if ((offset < 0) || (offset >= size_)) {\\n    TF_LITE_REPORT_ERROR(error_reporter_, \\"Attempt to read beyond the end of the queue!\\");\\n    offset = size_ - 1;\\n  }\\n  int index = front_index_ + offset;\\n  if (index >= kMaxResults) {\\n    index -= kMaxResults;\\n  }\\n  return results_[index];\\n}\\n\\nHashRecognizeCommands::HashRecognizeCommands(tflite::ErrorReporter* error_reporter,\\n                                             int32_t average_window_duration_ms,\\n                                             uint8_t detection_threshold,\\n                                             int32_t suppression_ms,\\n                                             int32_t minimum_count)\\n    : error_reporter_(error_reporter),\\n      average_window_duration_ms_(average_window_duration_ms),\\n      detection_threshold_(detection_threshold),\\n      suppression_ms_(suppression_ms),\\n      minimum_count_(minimum_count),\\n      previous_results_(error_reporter),\\n      previous_top_label_(kCategoryLabels[kSilenceIndex]),\\n      previous_top_label_time_(std::numeric_limits<int32_t>::min()) {}\\n\\nTfLiteStatus HashRecognizeCommands::ProcessLatestResults(const int8_t* latest_results,\\n                                                         int latest_results_size,\\n                                                         int32_t current_time_ms,\\n                                                         const char** found_command,\\n                                                         uint8_t* score,\\n                                                         bool* is_new_command) {\\n  if (latest_results_size != kCategoryCount) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Expected %d hash-KWS scores, got %d\\",\\n                         kCategoryCount, latest_results_size);\\n    return kTfLiteError;\\n  }\\n\\n  if ((!previous_results_.empty()) && (current_time_ms < previous_results_.front().time_)) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Results must be fed in increasing time order.\\");\\n    return kTfLiteError;\\n  }\\n\\n  const int64_t time_limit = current_time_ms - average_window_duration_ms_;\\n  while ((!previous_results_.empty()) && (previous_results_.front().time_ <= time_limit)) {\\n    previous_results_.pop_front();\\n  }\\n\\n  // In fake-mic mode timestamps can hit the window boundary exactly, so prune\\n  // first and keep the queue bounded before appending the newest result.\\n  while (previous_results_.size() >= previous_results_.capacity()) {\\n    previous_results_.pop_front();\\n  }\\n  previous_results_.push_back({current_time_ms, latest_results});\\n\\n  const int64_t how_many_results = previous_results_.size();\\n  const int64_t earliest_time = previous_results_.front().time_;\\n  const int64_t samples_duration = current_time_ms - earliest_time;\\n  if ((how_many_results < minimum_count_) || (samples_duration < (average_window_duration_ms_ / 4))) {\\n    *found_command = previous_top_label_;\\n    *score = 0;\\n    *is_new_command = false;\\n    return kTfLiteOk;\\n  }\\n\\n  int32_t average_scores[kCategoryCount];\\n  for (int offset = 0; offset < previous_results_.size(); ++offset) {\\n    PreviousResultsQueue::Result previous_result = previous_results_.from_front(offset);\\n    const int8_t* scores = previous_result.scores;\\n    for (int i = 0; i < kCategoryCount; ++i) {\\n      if (offset == 0) {\\n        average_scores[i] = scores[i] + 128;\\n      } else {\\n        average_scores[i] += scores[i] + 128;\\n      }\\n    }\\n  }\\n  for (int i = 0; i < kCategoryCount; ++i) {\\n    average_scores[i] /= how_many_results;\\n  }\\n\\n  int current_top_index = 0;\\n  int32_t current_top_score = 0;\\n  for (int i = 0; i < kCategoryCount; ++i) {\\n    if (average_scores[i] > current_top_score) {\\n      current_top_score = average_scores[i];\\n      current_top_index = i;\\n    }\\n  }\\n  const char* current_top_label = kCategoryLabels[current_top_index];\\n\\n  int64_t time_since_last_top;\\n  if ((previous_top_label_ == kCategoryLabels[kSilenceIndex]) ||\\n      (previous_top_label_time_ == std::numeric_limits<int32_t>::min())) {\\n    time_since_last_top = std::numeric_limits<int32_t>::max();\\n  } else {\\n    time_since_last_top = current_time_ms - previous_top_label_time_;\\n  }\\n\\n  if ((current_top_score > detection_threshold_) &&\\n      ((current_top_label != previous_top_label_) || (time_since_last_top > suppression_ms_))) {\\n    previous_top_label_ = current_top_label;\\n    previous_top_label_time_ = current_time_ms;\\n    *is_new_command = true;\\n  } else {\\n    *is_new_command = false;\\n  }\\n\\n  *found_command = current_top_label;\\n  *score = static_cast<uint8_t>(current_top_score);\\n  return kTfLiteOk;\\n}\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_kws_runner.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n\\n#include <cstddef>\\n#include <cstdint>\\n\\n#include \\"hash_model_data.h\\"\\n\\nnamespace hash_kws {\\n\\nstruct EspNnState;  // forward decl from hash_kws_espnn.h\\n\\nclass HashKwsRunner {\\n public:\\n  explicit HashKwsRunner(const HashDscnnModelData* model = &g_hash_model)\\n      : model_(model), esp_nn_state_(nullptr) {}\\n\\n  bool IsReady() const;\\n  int num_classes() const;\\n  size_t RequiredSingleScratchBytes() const;\\n  size_t RequiredScratchArenaBytes() const;\\n\\n  // One-shot weight materialisation for the ESP-NN fast path. Safe no-op if\\n  // ESP-NN is unavailable or the model has unsupported layers; the runner\\n  // simply stays on the int-MAC fallback.\\n  bool Prepare();\\n  bool IsEspNnReady() const;\\n\\n  // Converts the existing firmware feature layout [time][frequency]\\n  // into the model layout [channel=1][frequency][time].\\n  void PrepareInputFromMicroFeatures(const int8_t* feature_slices,\\n                                     int8_t* model_input) const;\\n\\n  bool Invoke(const int8_t* model_input,\\n              int8_t* scratch_a,\\n              int8_t* scratch_b,\\n              int8_t* output_scores) const;\\n\\n private:\\n  const HashDscnnModelData* model_;\\n  EspNnState* esp_nn_state_;\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n", "code/firmware/hash_kws_runtime/hash_kws_runner.cpp": "#include \\"hash_kws_runner.h\\"\\n#include \\"hash_kws_espnn.h\\"\\n\\n#include <algorithm>\\n#include <cmath>\\n#include <cstddef>\\n#include <cstdint>\\n#include <cstdio>\\n#include <Arduino.h>\\n\\nnamespace hash_kws {\\n\\nnamespace {\\n\\nconstexpr int kConvHashOc = 1337;\\nconstexpr int kConvHashIc = 7919;\\nconstexpr int kConvHashKh = 2971;\\nconstexpr int kConvHashKw = 6151;\\nconstexpr int kConvHashLayer = 104729;\\n\\nconstexpr int kDwHashCh = 1337;\\nconstexpr int kDwHashKh = 7919;\\nconstexpr int kDwHashKw = 2971;\\nconstexpr int kDwHashLayer = 104729;\\n\\nconstexpr int kLinearHashA = 1337;\\nconstexpr int kLinearHashB = 7919;\\nconstexpr int kLinearHashC = 2971;\\n\\nconstexpr int kSignHashA = 4099;\\nconstexpr int kSignHashB = 6151;\\nconstexpr int kSignHashC = 14887;\\n\\ninline int WrapPositiveMod(int value, int modulus) {\\n  int result = value % modulus;\\n  if (result < 0) {\\n    result += modulus;\\n  }\\n  return result;\\n}\\n\\ninline float HashSign(bool enabled, int value) {\\n  if (!enabled) {\\n    return 1.0f;\\n  }\\n  return ((value & 1) == 0) ? -1.0f : 1.0f;\\n}\\n\\ninline int OutputDim(int input, int kernel, int stride, int padding) {\\n  return ((input + (2 * padding) - kernel) / stride) + 1;\\n}\\n\\ninline int8_t QuantizeToInt8(float value, float scale) {\\n  if (scale <= 0.0f) {\\n    scale = 1.0f;\\n  }\\n  int quantized = static_cast<int>(std::lround(value / scale));\\n  if (quantized < -128) {\\n    quantized = -128;\\n  }\\n  if (quantized > 127) {\\n    quantized = 127;\\n  }\\n  return static_cast<int8_t>(quantized);\\n}\\n\\ninline float DequantizeCodebookValue(int8_t value, float scale) {\\n  if (scale <= 0.0f) {\\n    scale = 1.0f;\\n  }\\n  return static_cast<float>(value) * scale;\\n}\\n\\nfloat HashWeight(const HashConvLayerData& layer,\\n                 int output_channel,\\n                 int input_channel,\\n                 int kernel_row,\\n                 int kernel_col) {\\n  const int raw_index =\\n      (output_channel * kConvHashOc) + (input_channel * kConvHashIc) +\\n      (kernel_row * kConvHashKh) + (kernel_col * kConvHashKw) +\\n      (layer.layer_id * kConvHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (output_channel * kSignHashA) + (input_channel * kSignHashB) +\\n      (kernel_row * kSignHashC) + (kernel_col * (kSignHashA + kSignHashB)) +\\n      (layer.layer_id * (kSignHashC + 11));\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nfloat HashDepthwiseWeight(const HashDepthwiseLayerData& layer,\\n                          int channel,\\n                          int kernel_row,\\n                          int kernel_col) {\\n  const int raw_index =\\n      (channel * kDwHashCh) + (kernel_row * kDwHashKh) +\\n      (kernel_col * kDwHashKw) + (layer.layer_id * kDwHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (channel * kSignHashA) + (kernel_row * kSignHashB) +\\n      (kernel_col * kSignHashC) + (layer.layer_id * (kSignHashA + 29));\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nfloat HashLinearWeight(const HashLinearLayerData& layer, int output_index, int input_index) {\\n  const int raw_index =\\n      (output_index * kLinearHashA) + (input_index * kLinearHashB) +\\n      (layer.layer_id * kLinearHashC);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (output_index * kSignHashA) + (input_index * kSignHashB) +\\n      (layer.layer_id * kSignHashC);\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nvoid FillStemKernelWeights(const HashConvLayerData& layer,\\n                           int output_channel,\\n                           float* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashWeight(layer, output_channel, 0, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\nvoid FillDepthwiseKernelWeights(const HashDepthwiseLayerData& layer,\\n                                int channel,\\n                                float* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashDepthwiseWeight(layer, channel, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\nvoid FillPointwiseWeights(const HashConvLayerData& layer,\\n                          int output_channel,\\n                          float* weights_1x1) {\\n  for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n    weights_1x1[input_channel] = HashWeight(layer, output_channel, input_channel, 0, 0);\\n  }\\n}\\n\\n\\n// Integer-math companion to FillPointwiseWeights. Packs the hash-derived\\n// weight into a signed int8, with the sign from HashSign baked in and\\n// clamped to [-128, 127]. The per-layer scale is just codebook_scale.\\ninline int8_t HashPointwiseWeightInt8(const HashConvLayerData& layer,\\n                                     int output_channel,\\n                                     int input_channel) {\\n  const int raw_index =\\n      (output_channel * kConvHashOc) + (input_channel * kConvHashIc) +\\n      (layer.layer_id * kConvHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  int value = layer.codebook[bucket];\\n  if (layer.signed_hash) {\\n    const int sign_seed =\\n        (output_channel * kSignHashA) + (input_channel * kSignHashB) +\\n        (layer.layer_id * (kSignHashC + 11));\\n    if ((WrapPositiveMod(sign_seed, 2)) == 0) {\\n      value = -value;\\n    }\\n  }\\n  if (value < -128) value = -128;\\n  if (value > 127) value = 127;\\n  return static_cast<int8_t>(value);\\n}\\n\\nvoid FillPointwiseWeightsInt8(const HashConvLayerData& layer,\\n                              int output_channel,\\n                              int8_t* weights_1x1) {\\n  for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n    weights_1x1[input_channel] =\\n        HashPointwiseWeightInt8(layer, output_channel, input_channel);\\n  }\\n}\\n\\nvoid FillLinearWeights(const HashLinearLayerData& layer,\\n                       int output_index,\\n                       float* weights) {\\n  for (int input_index = 0; input_index < layer.in_dim; ++input_index) {\\n    weights[input_index] = HashLinearWeight(layer, output_index, input_index);\\n  }\\n}\\n\\n\\n// Integer-math companion for stem conv weights.\\ninline int8_t HashStemWeightInt8(const HashConvLayerData& layer,\\n                                 int output_channel,\\n                                 int kernel_row,\\n                                 int kernel_col) {\\n  const int raw_index =\\n      (output_channel * kConvHashOc) + (0 * kConvHashIc) +\\n      (kernel_row * kConvHashKh) + (kernel_col * kConvHashKw) +\\n      (layer.layer_id * kConvHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  int value = layer.codebook[bucket];\\n  if (layer.signed_hash) {\\n    const int sign_seed =\\n        (output_channel * kSignHashA) + (0 * kSignHashB) +\\n        (kernel_row * kSignHashC) + (kernel_col * (kSignHashA + kSignHashB)) +\\n        (layer.layer_id * (kSignHashC + 11));\\n    if (WrapPositiveMod(sign_seed, 2) == 0) {\\n      value = -value;\\n    }\\n  }\\n  if (value < -128) value = -128;\\n  if (value > 127) value = 127;\\n  return static_cast<int8_t>(value);\\n}\\n\\nvoid FillStemKernelWeightsInt8(const HashConvLayerData& layer,\\n                               int output_channel,\\n                               int8_t* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashStemWeightInt8(layer, output_channel, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\n// Integer-math companion for depthwise conv weights.\\ninline int8_t HashDepthwiseWeightInt8(const HashDepthwiseLayerData& layer,\\n                                      int channel,\\n                                      int kernel_row,\\n                                      int kernel_col) {\\n  const int raw_index =\\n      (channel * kDwHashCh) + (kernel_row * kDwHashKh) +\\n      (kernel_col * kDwHashKw) + (layer.layer_id * kDwHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  int value = layer.codebook[bucket];\\n  if (layer.signed_hash) {\\n    const int sign_seed =\\n        (channel * kSignHashA) + (kernel_row * kSignHashB) +\\n        (kernel_col * kSignHashC) + (layer.layer_id * (kSignHashA + 29));\\n    if (WrapPositiveMod(sign_seed, 2) == 0) {\\n      value = -value;\\n    }\\n  }\\n  if (value < -128) value = -128;\\n  if (value > 127) value = 127;\\n  return static_cast<int8_t>(value);\\n}\\n\\nvoid FillDepthwiseKernelWeightsInt8(const HashDepthwiseLayerData& layer,\\n                                    int channel,\\n                                    int8_t* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashDepthwiseWeightInt8(layer, channel, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\nvoid RunStemConv3x3(const HashConvLayerData& layer,\\n                    const HashActivationQuantParams& quant,\\n                    const int8_t* __restrict__ input,\\n                    int input_rows,\\n                    int input_cols,\\n                    int8_t* __restrict__ output) {\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n  const float mac_scale = quant.input_scale * layer.codebook_scale;\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    int8_t weights_3x3[9];\\n    FillStemKernelWeightsInt8(layer, output_channel, weights_3x3);\\n    const float combined_scale = layer.post_scale[output_channel] * mac_scale;\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        int32_t accum = 0;\\n        int weight_index = 0;\\n        for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n          const int input_row =\\n              (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n          for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n            const int input_col =\\n                (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n            if ((input_row >= 0) && (input_row < input_rows) &&\\n                (input_col >= 0) && (input_col < input_cols)) {\\n              const int input_index = (input_row * input_cols) + input_col;\\n              accum += static_cast<int32_t>(input[input_index]) *\\n                       static_cast<int32_t>(weights_3x3[weight_index]);\\n            }\\n            ++weight_index;\\n          }\\n        }\\n        float activated = (combined_scale * static_cast<float>(accum)) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunPointwiseConv1x1(const HashConvLayerData& layer,\\n                         const HashActivationQuantParams& quant,\\n                         const int8_t* __restrict__ input,\\n                         int input_rows,\\n                         int input_cols,\\n                         int8_t* __restrict__ output) {\\n  // Integer-math fast path. Uses the fact that each hash-derived weight\\n  // is just ±1 * codebook_scale * codebook[bucket]. We bake the sign\\n  // into an int8 weight and the codebook_scale into a single float\\n  // that multiplies the int32 accumulator once per output pixel.\\n  int8_t weights_1x1[kHashMaxChannels];\\n  const float mac_scale = quant.input_scale * layer.codebook_scale;\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    FillPointwiseWeightsInt8(layer, output_channel, weights_1x1);\\n    const float combined_scale = layer.post_scale[output_channel] * mac_scale;\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < input_rows; ++output_row) {\\n      for (int output_col = 0; output_col < input_cols; ++output_col) {\\n        int32_t accum = 0;\\n#if defined(__GNUC__)\\n#pragma GCC unroll 8\\n#endif\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          const int input_index =\\n              ((input_channel * input_rows) + output_row) * input_cols + output_col;\\n          accum += static_cast<int32_t>(input[input_index]) *\\n                   static_cast<int32_t>(weights_1x1[input_channel]);\\n        }\\n        float activated = (combined_scale * static_cast<float>(accum)) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * input_rows) + output_row) * input_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunPointwiseResidualConv1x1(const HashConvLayerData& layer,\\n                                 const HashActivationQuantParams& quant,\\n                                 const int8_t* __restrict__ input,\\n                                 int input_rows,\\n                                 int input_cols,\\n                                 const int8_t* __restrict__ residual_input,\\n                                 float residual_input_scale,\\n                                 int8_t* __restrict__ output) {\\n  int8_t weights_1x1[kHashMaxChannels];\\n  const float mac_scale = quant.input_scale * layer.codebook_scale;\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    FillPointwiseWeightsInt8(layer, output_channel, weights_1x1);\\n    const float combined_scale = layer.post_scale[output_channel] * mac_scale;\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < input_rows; ++output_row) {\\n      for (int output_col = 0; output_col < input_cols; ++output_col) {\\n        int32_t accum = 0;\\n#if defined(__GNUC__)\\n#pragma GCC unroll 8\\n#endif\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          const int input_index =\\n              ((input_channel * input_rows) + output_row) * input_cols + output_col;\\n          accum += static_cast<int32_t>(input[input_index]) *\\n                   static_cast<int32_t>(weights_1x1[input_channel]);\\n        }\\n        float activated = (combined_scale * static_cast<float>(accum)) + post_bias;\\n        const int output_index =\\n            ((output_channel * input_rows) + output_row) * input_cols + output_col;\\n        const float residual_value =\\n            static_cast<float>(residual_input[output_index]) * residual_input_scale;\\n        activated += residual_value;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashConv2D(const HashConvLayerData& layer,\\n                   const HashActivationQuantParams& quant,\\n                   const int8_t* input,\\n                   int input_rows,\\n                   int input_cols,\\n                   int8_t* output) {\\n  if ((layer.kernel_h == 3) && (layer.kernel_w == 3) && (layer.in_channels == 1) &&\\n      (layer.out_channels <= kHashMaxChannels)) {\\n    RunStemConv3x3(layer, quant, input, input_rows, input_cols, output);\\n    return;\\n  }\\n  if ((layer.kernel_h == 1) && (layer.kernel_w == 1) && (layer.in_channels <= kHashMaxChannels) &&\\n      (layer.out_channels <= kHashMaxChannels) && (layer.stride_h == 1) &&\\n      (layer.stride_w == 1) && (layer.padding_h == 0) && (layer.padding_w == 0)) {\\n    RunPointwiseConv1x1(layer, quant, input, input_rows, input_cols, output);\\n    return;\\n  }\\n\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    const float post_scale = layer.post_scale[output_channel];\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        float accum = 0.0f;\\n#if defined(__GNUC__)\\n#pragma GCC unroll 8\\n#endif\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          for (int kernel_row = 0; kernel_row < layer.kernel_h; ++kernel_row) {\\n            const int input_row =\\n                (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n            if ((input_row < 0) || (input_row >= input_rows)) {\\n              continue;\\n            }\\n            for (int kernel_col = 0; kernel_col < layer.kernel_w; ++kernel_col) {\\n              const int input_col =\\n                  (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n              if ((input_col < 0) || (input_col >= input_cols)) {\\n                continue;\\n              }\\n              const int input_index =\\n                  ((input_channel * input_rows) + input_row) * input_cols + input_col;\\n              const float input_value =\\n                  static_cast<float>(input[input_index]) * quant.input_scale;\\n              accum +=\\n                  input_value *\\n                  HashWeight(layer, output_channel, input_channel, kernel_row, kernel_col);\\n            }\\n          }\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashDepthwiseConv2D(const HashDepthwiseLayerData& layer,\\n                            const HashActivationQuantParams& quant,\\n                            const int8_t* __restrict__ input,\\n                            int input_rows,\\n                            int input_cols,\\n                            int8_t* __restrict__ output) {\\n  if ((layer.kernel_h == 3) && (layer.kernel_w == 3) && (layer.channels <= kHashMaxChannels) &&\\n      (layer.stride_h == 1) && (layer.stride_w == 1) && (layer.padding_h == 1) &&\\n      (layer.padding_w == 1)) {\\n    const float mac_scale = quant.input_scale * layer.codebook_scale;\\n    for (int channel = 0; channel < layer.channels; ++channel) {\\n      int8_t weights_3x3[9];\\n      FillDepthwiseKernelWeightsInt8(layer, channel, weights_3x3);\\n      const float combined_scale = layer.post_scale[channel] * mac_scale;\\n      const float post_bias = layer.post_bias[channel];\\n      for (int output_row = 0; output_row < input_rows; ++output_row) {\\n        for (int output_col = 0; output_col < input_cols; ++output_col) {\\n          int32_t accum = 0;\\n          int weight_index = 0;\\n          for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n            const int input_row = output_row + kernel_row - 1;\\n            for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n              const int input_col = output_col + kernel_col - 1;\\n              if ((input_row >= 0) && (input_row < input_rows) &&\\n                  (input_col >= 0) && (input_col < input_cols)) {\\n                const int input_index =\\n                    ((channel * input_rows) + input_row) * input_cols + input_col;\\n                accum += static_cast<int32_t>(input[input_index]) *\\n                         static_cast<int32_t>(weights_3x3[weight_index]);\\n              }\\n              ++weight_index;\\n            }\\n          }\\n          float activated = (combined_scale * static_cast<float>(accum)) + post_bias;\\n          if (activated < 0.0f) {\\n            activated = 0.0f;\\n          }\\n          const int output_index =\\n              ((channel * input_rows) + output_row) * input_cols + output_col;\\n          output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n        }\\n      }\\n    }\\n    return;\\n  }\\n\\n  // Fallback: non-3x3 or non-unit stride/padding path (unoptimised float).\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n\\n  for (int channel = 0; channel < layer.channels; ++channel) {\\n    const float post_scale = layer.post_scale[channel];\\n    const float post_bias = layer.post_bias[channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        float accum = 0.0f;\\n        for (int kernel_row = 0; kernel_row < layer.kernel_h; ++kernel_row) {\\n          const int input_row =\\n              (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n          if ((input_row < 0) || (input_row >= input_rows)) continue;\\n          for (int kernel_col = 0; kernel_col < layer.kernel_w; ++kernel_col) {\\n            const int input_col =\\n                (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n            if ((input_col < 0) || (input_col >= input_cols)) continue;\\n            const int input_index =\\n                ((channel * input_rows) + input_row) * input_cols + input_col;\\n            const float input_value =\\n                static_cast<float>(input[input_index]) * quant.input_scale;\\n            accum += input_value * HashDepthwiseWeight(layer, channel, kernel_row, kernel_col);\\n          }\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) activated = 0.0f;\\n        const int output_index =\\n            ((channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashLinear(const HashLinearLayerData& layer,\\n                   const int8_t* input,\\n                   float input_scale,\\n                   float* logits) {\\n  float weights[kHashMaxChannels];\\n  for (int output_index = 0; output_index < layer.out_dim; ++output_index) {\\n    FillLinearWeights(layer, output_index, weights);\\n    float accum = layer.bias[output_index];\\n    for (int input_index = 0; input_index < layer.in_dim; ++input_index) {\\n      accum += (static_cast<float>(input[input_index]) * input_scale) *\\n               weights[input_index];\\n    }\\n    logits[output_index] = accum;\\n  }\\n}\\n\\nvoid AveragePoolChannels(const int8_t* input,\\n                         int channels,\\n                         int rows,\\n                         int cols,\\n                         float input_scale,\\n                         int8_t* pooled_output,\\n                         float pooled_output_scale) {\\n  const int spatial_size = rows * cols;\\n  for (int channel = 0; channel < channels; ++channel) {\\n    int32_t sum = 0;\\n    const int base_index = channel * spatial_size;\\n    for (int i = 0; i < spatial_size; ++i) {\\n      sum += static_cast<int32_t>(input[base_index + i]);\\n    }\\n    const float mean_value = (static_cast<float>(sum) / static_cast<float>(spatial_size)) * input_scale;\\n    pooled_output[channel] = QuantizeToInt8(mean_value, pooled_output_scale);\\n  }\\n}\\n\\nvoid SoftmaxToCenteredInt8(const float* logits, int count, int8_t* output_scores) {\\n  float max_logit = logits[0];\\n  for (int i = 1; i < count; ++i) {\\n    max_logit = std::max(max_logit, logits[i]);\\n  }\\n\\n  float sum = 0.0f;\\n  float probabilities[kHashMaxClasses];\\n  for (int i = 0; i < count; ++i) {\\n    probabilities[i] = std::exp(logits[i] - max_logit);\\n    sum += probabilities[i];\\n  }\\n\\n  for (int i = 0; i < count; ++i) {\\n    const float normalized = probabilities[i] / sum;\\n    int quantized = static_cast<int>(std::lround(normalized * 255.0f)) - 128;\\n    if (quantized < -128) {\\n      quantized = -128;\\n    }\\n    if (quantized > 127) {\\n      quantized = 127;\\n    }\\n    output_scores[i] = static_cast<int8_t>(quantized);\\n  }\\n}\\n\\nbool ValidateModel(const HashDscnnModelData* model) {\\n  if (model == nullptr) {\\n    return false;\\n  }\\n  if (!model->available) {\\n    return false;\\n  }\\n  if (model->input_channels != kHashInputChannels) {\\n    return false;\\n  }\\n  if ((model->num_blocks <= 0) || (model->num_blocks > kHashMaxBlocks)) {\\n    return false;\\n  }\\n  if ((model->stem.out_channels <= 0) || (model->stem.out_channels > kHashMaxChannels)) {\\n    return false;\\n  }\\n  if ((model->classifier.in_dim <= 0) || (model->classifier.in_dim > kHashMaxChannels)) {\\n    return false;\\n  }\\n  if ((model->num_classes <= 0) || (model->num_classes > kHashMaxClasses)) {\\n    return false;\\n  }\\n  for (int block = 0; block < model->num_blocks; ++block) {\\n    if ((model->depthwise[block].channels <= 0) ||\\n        (model->depthwise[block].channels > kHashMaxChannels)) {\\n      return false;\\n    }\\n    if ((model->pointwise[block].in_channels <= 0) ||\\n        (model->pointwise[block].in_channels > kHashMaxChannels) ||\\n        (model->pointwise[block].out_channels <= 0) ||\\n        (model->pointwise[block].out_channels > kHashMaxChannels)) {\\n      return false;\\n    }\\n    if (model->block_residual[block]) {\\n      if ((model->depthwise[block].stride_h != 1) || (model->depthwise[block].stride_w != 1) ||\\n          (model->pointwise[block].kernel_h != 1) || (model->pointwise[block].kernel_w != 1) ||\\n          (model->pointwise[block].stride_h != 1) || (model->pointwise[block].stride_w != 1) ||\\n          (model->pointwise[block].padding_h != 0) || (model->pointwise[block].padding_w != 0) ||\\n          (model->pointwise[block].in_channels != model->pointwise[block].out_channels) ||\\n          (model->depthwise[block].channels != model->pointwise[block].out_channels)) {\\n        return false;\\n      }\\n    }\\n  }\\n  return true;\\n}\\n\\nsize_t MaxActivationElements(const HashDscnnModelData& model) {\\n  int rows = model.input_rows;\\n  int cols = model.input_cols;\\n  size_t max_elements = 0;\\n\\n  rows = OutputDim(rows, model.stem.kernel_h, model.stem.stride_h, model.stem.padding_h);\\n  cols = OutputDim(cols, model.stem.kernel_w, model.stem.stride_w, model.stem.padding_w);\\n  max_elements = std::max(max_elements, static_cast<size_t>(model.stem.out_channels * rows * cols));\\n\\n  for (int block = 0; block < model.num_blocks; ++block) {\\n    rows = OutputDim(rows, model.depthwise[block].kernel_h, model.depthwise[block].stride_h, model.depthwise[block].padding_h);\\n    cols = OutputDim(cols, model.depthwise[block].kernel_w, model.depthwise[block].stride_w, model.depthwise[block].padding_w);\\n    max_elements = std::max(max_elements, static_cast<size_t>(model.depthwise[block].channels * rows * cols));\\n\\n    rows = OutputDim(rows, model.pointwise[block].kernel_h, model.pointwise[block].stride_h, model.pointwise[block].padding_h);\\n    cols = OutputDim(cols, model.pointwise[block].kernel_w, model.pointwise[block].stride_w, model.pointwise[block].padding_w);\\n    max_elements = std::max(max_elements, static_cast<size_t>(model.pointwise[block].out_channels * rows * cols));\\n  }\\n\\n  max_elements = std::max(max_elements, static_cast<size_t>(model.classifier.in_dim));\\n  return max_elements;\\n}\\n\\n}  // namespace\\n\\nbool HashKwsRunner::IsReady() const { return ValidateModel(model_); }\\n\\nbool HashKwsRunner::Prepare() {\\n  if (esp_nn_state_ != nullptr) return IsEspNnReady();\\n  if (!IsReady()) return false;\\n  if (!EspNnHeaderAvailable()) return false;\\n  if (!EspNnPrepare(*model_, &esp_nn_state_)) {\\n    if (esp_nn_state_ != nullptr) {\\n      EspNnRelease(esp_nn_state_);\\n      esp_nn_state_ = nullptr;\\n    }\\n    return false;\\n  }\\n  return IsEspNnReady();\\n}\\n\\nbool HashKwsRunner::IsEspNnReady() const {\\n  return (esp_nn_state_ != nullptr) && esp_nn_state_->ready;\\n}\\n\\nint HashKwsRunner::num_classes() const {\\n  return (model_ != nullptr) ? model_->num_classes : 0;\\n}\\n\\nsize_t HashKwsRunner::RequiredSingleScratchBytes() const {\\n  if (model_ == nullptr) {\\n    return 0;\\n  }\\n  return MaxActivationElements(*model_) * sizeof(int8_t);\\n}\\n\\nsize_t HashKwsRunner::RequiredScratchArenaBytes() const {\\n  return 2 * RequiredSingleScratchBytes();\\n}\\n\\nvoid HashKwsRunner::PrepareInputFromMicroFeatures(const int8_t* feature_slices,\\n                                                  int8_t* model_input) const {\\n  if ((feature_slices == nullptr) || (model_input == nullptr)) {\\n    return;\\n  }\\n  for (int time_index = 0; time_index < kHashInputCols; ++time_index) {\\n    for (int freq_index = 0; freq_index < kHashInputRows; ++freq_index) {\\n      const int source_index = (time_index * kHashInputRows) + freq_index;\\n      const int dest_index = (freq_index * kHashInputCols) + time_index;\\n      model_input[dest_index] = feature_slices[source_index];\\n    }\\n  }\\n}\\n\\nbool HashKwsRunner::Invoke(const int8_t* model_input,\\n                           int8_t* scratch_a,\\n                           int8_t* scratch_b,\\n                           int8_t* output_scores) const {\\n  if (!IsReady() || (model_input == nullptr) || (scratch_a == nullptr) ||\\n      (scratch_b == nullptr) || (output_scores == nullptr)) {\\n    return false;\\n  }\\n\\n  // ESP-NN fast path: full conv chain + AvgPool inside the SIMD module,\\n  // pooled int8 lands in scratch_b. Falls back to the int-MAC path below\\n  // if the model has unsupported layers (e.g. residual blocks today).\\n  if (IsEspNnReady()) {\\n    // Pass 4 diagnostic: run the int-MAC stem on the SAME model_input first\\n    // and dump channel-0 stats. scratch_b is safe to use as a temp here —\\n    // EspNnInvokeUpToPooled below will write to scratch_a (stem) and then\\n    // scratch_b (DW block 0) before AvgPool reads it. The first non-silence\\n    // invoke triggers it; subsequent invokes are unaffected.\\n    {\\n      bool dual_input_signal = false;\\n      for (int i = 0; i < 64 && !dual_input_signal; ++i) {\\n        if (model_input[i] > -64) dual_input_signal = true;\\n      }\\n      static bool s_pass4_dual_stem_dumped = false;\\n      if (!s_pass4_dual_stem_dumped && dual_input_signal) {\\n        s_pass4_dual_stem_dumped = true;\\n        RunHashConv2D(model_->stem, model_->activations[0],\\n                      model_input, model_->input_rows, model_->input_cols,\\n                      scratch_b);\\n        const int OH = OutputDim(model_->input_rows,\\n                                 model_->stem.kernel_h,\\n                                 model_->stem.stride_h,\\n                                 model_->stem.padding_h);\\n        const int OW = OutputDim(model_->input_cols,\\n                                 model_->stem.kernel_w,\\n                                 model_->stem.stride_w,\\n                                 model_->stem.padding_w);\\n        const int spatial = OH * OW;\\n        int32_t sum = 0; int32_t maxv = -128; int nz = 0;\\n        for (int p = 0; p < spatial; ++p) {\\n          const int v = scratch_b[p];\\n          sum += v;\\n          if (v > maxv) maxv = v;\\n          if (v != 0) ++nz;\\n        }\\n        Serial.printf(\\"hash_dbg pass4 dual_stem int_mac_on_same_input ch0_at00=%d ch0_sum=%ld ch0_max=%d ch0_nonzero=%d\\\\n\\",\\n                      static_cast<int>(scratch_b[0]),\\n                      static_cast<long>(sum),\\n                      static_cast<int>(maxv), nz);\\n        // Hand-computed MAC at output (0, 0, oc=0) via int-MAC\'s hash helpers\\n        // and direct model_input access. This is the gold-standard reference.\\n        int32_t hand_mac = 0;\\n        for (int kh = 0; kh < model_->stem.kernel_h; ++kh) {\\n          const int ih = 0 - model_->stem.padding_h + kh;\\n          if (ih < 0 || ih >= model_->input_rows) continue;\\n          for (int kw = 0; kw < model_->stem.kernel_w; ++kw) {\\n            const int iw = 0 - model_->stem.padding_w + kw;\\n            if (iw < 0 || iw >= model_->input_cols) continue;\\n            int8_t w = HashStemWeightInt8(model_->stem, 0, kh, kw);\\n            int32_t in_v = model_input[ih * model_->input_cols + iw];\\n            hand_mac += in_v * w;\\n          }\\n        }\\n        Serial.printf(\\"hash_dbg pass4 hand_mac stem path=int_mac oc=0 pos=[0,0] mac=%ld\\\\n\\",\\n                      static_cast<long>(hand_mac));\\n        // Hand-MAC for DW block 0, channel 0, position (0, 0) using the\\n        // int-MAC stem output sitting in scratch_b (CHW layout). This is\\n        // the gold-standard reference for what the next conv (DW) should\\n        // accumulate before bias / requantize.\\n        const auto& dw0_im = model_->depthwise[0];\\n        const int H_dw_im = OutputDim(model_->input_rows, model_->stem.kernel_h,\\n                                      model_->stem.stride_h, model_->stem.padding_h);\\n        const int W_dw_im = OutputDim(model_->input_cols, model_->stem.kernel_w,\\n                                      model_->stem.stride_w, model_->stem.padding_w);\\n        int32_t dw0_hand_mac_im = 0;\\n        for (int kh = 0; kh < dw0_im.kernel_h; ++kh) {\\n          const int ih = 0 - dw0_im.padding_h + kh;\\n          if (ih < 0 || ih >= H_dw_im) continue;\\n          for (int kw = 0; kw < dw0_im.kernel_w; ++kw) {\\n            const int iw = 0 - dw0_im.padding_w + kw;\\n            if (iw < 0 || iw >= W_dw_im) continue;\\n            int8_t w = HashDepthwiseWeightInt8(dw0_im, 0, kh, kw);\\n            // CHW: scratch_b[ch * H_dw * W_dw + ih * W_dw + iw], ch=0.\\n            int32_t in_v = scratch_b[(0 * H_dw_im + ih) * W_dw_im + iw];\\n            dw0_hand_mac_im += in_v * w;\\n          }\\n        }\\n        Serial.printf(\\"hash_dbg pass4 hand_mac dw0 path=int_mac ch=0 pos=[0,0] mac=%ld\\\\n\\",\\n                      static_cast<long>(dw0_hand_mac_im));\\n        // First row of channel 0 from int-MAC stem (CHW: scratch_b[0..W-1]).\\n        Serial.print(\\"hash_dbg pass4 stem path=int_mac ch0_row0=[\\");\\n        for (int i = 0; i < W_dw_im; ++i) {\\n          Serial.printf(\\"%d\\", static_cast<int>(scratch_b[i]));\\n          if (i + 1 < W_dw_im) Serial.print(\\",\\");\\n        }\\n        Serial.println(\\"]\\");\\n        // Run int-MAC DW block 0 on scratch_b → scratch_a, dump ch0 stats.\\n        // DW block 0 is stride 1 padding 1 kernel 3, so output dims match input.\\n        RunHashDepthwiseConv2D(model_->depthwise[0], model_->activations[1],\\n                               scratch_b, H_dw_im, W_dw_im, scratch_a);\\n        const int dw_spatial = H_dw_im * W_dw_im;\\n        int32_t dw_sum = 0; int32_t dw_max = -128; int dw_nz = 0;\\n        for (int p = 0; p < dw_spatial; ++p) {\\n          int v = scratch_a[p];  // CHW: ch0 at scratch_a[0..spatial-1].\\n          dw_sum += v;\\n          if (v > dw_max) dw_max = v;\\n          if (v != 0) ++dw_nz;\\n        }\\n        Serial.printf(\\"hash_dbg pass4 dw0 path=int_mac ch0_at00=%d ch0_sum=%ld ch0_max=%d ch0_nonzero=%d\\\\n\\",\\n                      static_cast<int>(scratch_a[0]),\\n                      static_cast<long>(dw_sum), static_cast<int>(dw_max), dw_nz);\\n        Serial.print(\\"hash_dbg pass4 dw0 path=int_mac ch0_row0=[\\");\\n        for (int i = 0; i < W_dw_im; ++i) {\\n          Serial.printf(\\"%d\\", static_cast<int>(scratch_a[i]));\\n          if (i + 1 < W_dw_im) Serial.print(\\",\\");\\n        }\\n        Serial.println(\\"]\\");\\n        // scratch_a will be overwritten by EspNnInvokeUpToPooled\'s stem next.\\n      }\\n    }\\n\\n    float pooled_scale_fast = 0.0f;\\n    const bool fast_ok = EspNnInvokeUpToPooled(*model_, *esp_nn_state_,\\n                                               model_input,\\n                                               scratch_a, scratch_b,\\n                                               scratch_b,\\n                                               &pooled_scale_fast);\\n    if (fast_ok) {\\n      float logits_fast[kHashMaxClasses];\\n      RunHashLinear(model_->classifier, scratch_b, pooled_scale_fast, logits_fast);\\n      // Diagnostic one-shot: dump pooled int8 + logits on the first invoke\\n      // that has actual speech in the input (so we can compare same-input\\n      // captures across HASH_KWS_USE_ESP_NN=0 / =1 builds).\\n      bool esp_input_has_signal = false;\\n      for (int i = 0; i < 64 && !esp_input_has_signal; ++i) {\\n        if (model_input[i] > -64) esp_input_has_signal = true;\\n      }\\n      static bool s_pass4_first_invoke_dumped = false;\\n      if (!s_pass4_first_invoke_dumped && esp_input_has_signal) {\\n        s_pass4_first_invoke_dumped = true;\\n        Serial.printf(\\"hash_dbg pass4 path=esp_nn pooled_scale=%.6f\\\\n\\", pooled_scale_fast);\\n        Serial.print(\\"hash_dbg pooled_int8=[\\");\\n        for (int i = 0; i < model_->classifier.in_dim; ++i) {\\n          Serial.printf(\\"%d\\", static_cast<int>(scratch_b[i]));\\n          if (i + 1 < model_->classifier.in_dim) Serial.print(\\",\\");\\n        }\\n        Serial.println(\\"]\\");\\n        Serial.print(\\"hash_dbg logits=[\\");\\n        for (int i = 0; i < model_->num_classes; ++i) {\\n          Serial.printf(\\"%.4f\\", logits_fast[i]);\\n          if (i + 1 < model_->num_classes) Serial.print(\\",\\");\\n        }\\n        Serial.println(\\"]\\");\\n      }\\n      SoftmaxToCenteredInt8(logits_fast, model_->num_classes, output_scores);\\n      return true;\\n    }\\n    // else: fall through to int-MAC path below.\\n  }\\n\\n  int rows = model_->input_rows;\\n  int cols = model_->input_cols;\\n\\n  RunHashConv2D(model_->stem, model_->activations[0], model_input, rows, cols, scratch_a);\\n  // Diagnostic: post-stem channel-0 stats in CHW layout. Channel 0 lives at\\n  // scratch_a[0..H*W-1]. Sum/max are layout-independent so they compare 1:1\\n  // with the ESP-NN dump.\\n  // Skip silence frames so both paths can be compared on similar inputs.\\n  bool int_mac_input_has_signal = false;\\n  for (int i = 0; i < 64 && !int_mac_input_has_signal; ++i) {\\n    if (model_input[i] > -64) int_mac_input_has_signal = true;\\n  }\\n  {\\n    static bool s_pass4_stem_int_mac_dumped = false;\\n    if (!s_pass4_stem_int_mac_dumped && int_mac_input_has_signal) {\\n      s_pass4_stem_int_mac_dumped = true;\\n      // Dump int-MAC\'s stem channel-0 weights so we can directly compare\\n      // with the ESP-NN BuildStemFilter output (filter9 dump).\\n      int8_t int_mac_stem_w[9];\\n      FillStemKernelWeightsInt8(model_->stem, /*output_channel=*/0, int_mac_stem_w);\\n      Serial.print(\\"hash_dbg pass4 stem path=int_mac ch0_filter9=[\\");\\n      for (int i = 0; i < 9; ++i) {\\n        Serial.printf(\\"%d\\", static_cast<int>(int_mac_stem_w[i]));\\n        if (i + 1 < 9) Serial.print(\\",\\");\\n      }\\n      Serial.println(\\"]\\");\\n      // Also dump first few input values for cross-check.\\n      Serial.print(\\"hash_dbg pass4 stem path=int_mac model_input_first16=[\\");\\n      for (int i = 0; i < 16; ++i) {\\n        Serial.printf(\\"%d\\", static_cast<int>(model_input[i]));\\n        if (i + 1 < 16) Serial.print(\\",\\");\\n      }\\n      Serial.println(\\"]\\");\\n      const int OH = OutputDim(rows, model_->stem.kernel_h, model_->stem.stride_h, model_->stem.padding_h);\\n      const int OW = OutputDim(cols, model_->stem.kernel_w, model_->stem.stride_w, model_->stem.padding_w);\\n      const int spatial = OH * OW;\\n      int32_t ch0_sum = 0;\\n      int32_t ch0_max = -128;\\n      int     ch0_nz  = 0;\\n      for (int p = 0; p < spatial; ++p) {\\n        const int v = scratch_a[0 * spatial + p];\\n        ch0_sum += v;\\n        if (v > ch0_max) ch0_max = v;\\n        if (v != 0) ++ch0_nz;\\n      }\\n      Serial.printf(\\"hash_dbg pass4 stem path=int_mac rows=%d cols=%d oc=%d ch0_at00=%d ch0_sum=%ld ch0_max=%d ch0_nonzero=%d\\\\n\\",\\n                    OH, OW, model_->stem.out_channels, static_cast<int>(scratch_a[0]),\\n                    static_cast<long>(ch0_sum), static_cast<int>(ch0_max), ch0_nz);\\n    }\\n  }\\n  rows = OutputDim(rows, model_->stem.kernel_h, model_->stem.stride_h, model_->stem.padding_h);\\n  cols = OutputDim(cols, model_->stem.kernel_w, model_->stem.stride_w, model_->stem.padding_w);\\n\\n  for (int block = 0; block < model_->num_blocks; ++block) {\\n    const int depthwise_stage = 1 + (2 * block);\\n    const int pointwise_stage = depthwise_stage + 1;\\n    RunHashDepthwiseConv2D(model_->depthwise[block],\\n                           model_->activations[depthwise_stage],\\n                           scratch_a,\\n                           rows,\\n                           cols,\\n                           scratch_b);\\n    rows = OutputDim(rows, model_->depthwise[block].kernel_h,\\n                     model_->depthwise[block].stride_h,\\n                     model_->depthwise[block].padding_h);\\n    cols = OutputDim(cols, model_->depthwise[block].kernel_w,\\n                     model_->depthwise[block].stride_w,\\n                     model_->depthwise[block].padding_w);\\n\\n    if (model_->block_residual[block]) {\\n      RunPointwiseResidualConv1x1(model_->pointwise[block],\\n                                  model_->activations[pointwise_stage],\\n                                  scratch_b,\\n                                  rows,\\n                                  cols,\\n                                  scratch_a,\\n                                  model_->activations[depthwise_stage].input_scale,\\n                                  scratch_a);\\n    } else {\\n      RunHashConv2D(model_->pointwise[block],\\n                    model_->activations[pointwise_stage],\\n                    scratch_b,\\n                    rows,\\n                    cols,\\n                    scratch_a);\\n    }\\n    rows = OutputDim(rows, model_->pointwise[block].kernel_h,\\n                     model_->pointwise[block].stride_h,\\n                     model_->pointwise[block].padding_h);\\n    cols = OutputDim(cols, model_->pointwise[block].kernel_w,\\n                     model_->pointwise[block].stride_w,\\n                     model_->pointwise[block].padding_w);\\n  }\\n\\n  const float pooled_scale = model_->activations[2 * model_->num_blocks].output_scale;\\n  AveragePoolChannels(scratch_a,\\n                      model_->classifier.in_dim,\\n                      rows,\\n                      cols,\\n                      pooled_scale,\\n                      scratch_b,\\n                      pooled_scale);\\n\\n  float logits[kHashMaxClasses];\\n  RunHashLinear(model_->classifier, scratch_b, pooled_scale, logits);\\n  // Diagnostic one-shot mirror of the ESP-NN dump above: when the int-MAC\\n  // path is taken first (e.g. HASH_KWS_USE_ESP_NN=0 build) we still want a\\n  // pooled+logits snapshot for comparison.\\n  static bool s_pass4_int_mac_first_invoke_dumped = false;\\n  if (!s_pass4_int_mac_first_invoke_dumped) {\\n    s_pass4_int_mac_first_invoke_dumped = true;\\n    Serial.printf(\\"hash_dbg pass4 path=int_mac pooled_scale=%.6f\\\\n\\", pooled_scale);\\n    Serial.print(\\"hash_dbg pooled_int8=[\\");\\n    for (int i = 0; i < model_->classifier.in_dim; ++i) {\\n      Serial.printf(\\"%d\\", static_cast<int>(scratch_b[i]));\\n      if (i + 1 < model_->classifier.in_dim) Serial.print(\\",\\");\\n    }\\n    Serial.println(\\"]\\");\\n    Serial.print(\\"hash_dbg logits=[\\");\\n    for (int i = 0; i < model_->num_classes; ++i) {\\n      Serial.printf(\\"%.4f\\", logits[i]);\\n      if (i + 1 < model_->num_classes) Serial.print(\\",\\");\\n    }\\n    Serial.println(\\"]\\");\\n  }\\n  SoftmaxToCenteredInt8(logits, model_->num_classes, output_scores);\\n  return true;\\n}\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_micro_speech.cpp": "#include <TensorFlowLite_ESP32.h>\\n\\n#include <cstdint>\\n#include <cstdlib>\\n#include <esp_heap_caps.h>\\n\\n#include \\"../micro_speech_sim/audio_provider.h\\"\\n#include \\"../micro_speech_sim/command_responder.h\\"\\n#include \\"../micro_speech_sim/feature_provider.h\\"\\n#include \\"../micro_speech_sim/micro_model_settings.h\\"\\n#include \\"hash_kws_runner.h\\"\\n#include \\"hash_model_data.h\\"\\n#include \\"hash_recognize_commands.h\\"\\n#include \\"tensorflow/lite/micro/micro_error_reporter.h\\"\\n#include \\"tensorflow/lite/micro/system_setup.h\\"\\n\\nnamespace {\\n\\ntflite::ErrorReporter* error_reporter = nullptr;\\nFeatureProvider* feature_provider = nullptr;\\nhash_kws::HashKwsRunner* runner = nullptr;\\nhash_kws::HashRecognizeCommands* recognizer = nullptr;\\n\\nint32_t previous_time = 0;\\nint8_t feature_buffer[kFeatureElementCount];\\nint8_t* model_input_buffer = nullptr;\\nint8_t* scratch_a = nullptr;\\nint8_t* scratch_b = nullptr;\\nint8_t output_scores[hash_kws::kCategoryCount];\\n\\n}  // namespace\\n\\nvoid setup() {\\n  static tflite::MicroErrorReporter micro_error_reporter;\\n  error_reporter = &micro_error_reporter;\\n\\n  if (!hash_kws::g_hash_model.available) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"hash_kws model is not available. Export hash_model_data.cpp first.\\");\\n    return;\\n  }\\n\\n  static FeatureProvider static_feature_provider(kFeatureElementCount, feature_buffer);\\n  feature_provider = &static_feature_provider;\\n\\n  static hash_kws::HashKwsRunner static_runner(&hash_kws::g_hash_model);\\n  runner = &static_runner;\\n\\n  if (!runner->IsReady()) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"hash_kws runtime model validation failed.\\");\\n    return;\\n  }\\n\\n  static hash_kws::HashRecognizeCommands static_recognizer(error_reporter);\\n  recognizer = &static_recognizer;\\n\\n  const size_t input_bytes =\\n      static_cast<size_t>(hash_kws::g_hash_model.input_rows) *\\n      static_cast<size_t>(hash_kws::g_hash_model.input_cols) *\\n      static_cast<size_t>(hash_kws::g_hash_model.input_channels);\\n  const size_t scratch_bytes = runner->RequiredSingleScratchBytes();\\n\\n  model_input_buffer = static_cast<int8_t*>(heap_caps_malloc(input_bytes, MALLOC_CAP_INTERNAL | MALLOC_CAP_8BIT));\\n  scratch_a = static_cast<int8_t*>(heap_caps_malloc(scratch_bytes, MALLOC_CAP_INTERNAL | MALLOC_CAP_8BIT));\\n  scratch_b = static_cast<int8_t*>(heap_caps_malloc(scratch_bytes, MALLOC_CAP_INTERNAL | MALLOC_CAP_8BIT));\\n  if ((model_input_buffer == nullptr) || (scratch_a == nullptr) || (scratch_b == nullptr)) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"hash_kws allocation failed: input=%d scratch=%d\\",\\n                         static_cast<int>(input_bytes),\\n                         static_cast<int>(scratch_bytes));\\n    return;\\n  }\\n\\n  TF_LITE_REPORT_ERROR(\\n      error_reporter,\\n      \\"hash_kws ready: classes=%d input=%dx%d scratch_total=%d bytes\\",\\n      runner->num_classes(), hash_kws::g_hash_model.input_rows,\\n      hash_kws::g_hash_model.input_cols,\\n      static_cast<int>(runner->RequiredScratchArenaBytes()));\\n\\n  previous_time = 0;\\n}\\n\\nvoid loop() {\\n  if ((feature_provider == nullptr) || (runner == nullptr) || (recognizer == nullptr) ||\\n      (model_input_buffer == nullptr) || (scratch_a == nullptr) || (scratch_b == nullptr)) {\\n    return;\\n  }\\n\\n  const int32_t current_time = LatestAudioTimestamp();\\n  int how_many_new_slices = 0;\\n  TfLiteStatus feature_status = feature_provider->PopulateFeatureData(\\n      error_reporter, previous_time, current_time, &how_many_new_slices);\\n  if (feature_status != kTfLiteOk) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"Feature generation failed\\");\\n    return;\\n  }\\n  previous_time = current_time;\\n  if (how_many_new_slices == 0) {\\n    return;\\n  }\\n\\n  runner->PrepareInputFromMicroFeatures(feature_buffer, model_input_buffer);\\n  if (!runner->Invoke(model_input_buffer, scratch_a, scratch_b, output_scores)) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"hash_kws Invoke failed\\");\\n    return;\\n  }\\n\\n  const char* found_command = nullptr;\\n  uint8_t score = 0;\\n  bool is_new_command = false;\\n  TfLiteStatus process_status = recognizer->ProcessLatestResults(\\n      output_scores, hash_kws::kCategoryCount, current_time, &found_command,\\n      &score, &is_new_command);\\n  if (process_status != kTfLiteOk) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"HashRecognizeCommands::ProcessLatestResults() failed\\");\\n    return;\\n  }\\n\\n  RespondToCommand(error_reporter, current_time, found_command, score, is_new_command);\\n}\\n"}')

def ensure_runtime_files(root: Path, payloads: dict, overwrite: bool = False):
    created, skipped = [], []
    for relative_path, content in payloads.items():
        target_path = root / relative_path
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if target_path.exists() and not overwrite:
            skipped.append(relative_path)
            continue
        target_path.write_text(content, encoding="utf-8")
        created.append(relative_path)
    return created, skipped

created_files, skipped_files = ensure_runtime_files(
    PROJECT_ROOT, FILE_PAYLOADS, overwrite=FORCE_SYNC_RUNTIME_FILES,
)

for path in (TRAINING_ROOT, SCRIPTS_ROOT, HASHEDNET95_ROOT, HASH_ENSEMBLE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if (
        module_name == "hash_kws_lab"
        or module_name.startswith("hash_kws_lab.")
        or module_name == "hashednet95"
        or module_name.startswith("hashednet95.")
        or module_name == "hash_ensemble"
        or module_name.startswith("hash_ensemble.")
    ):
        del sys.modules[module_name]

local_speechcommands_root = PROJECT_ROOT / "data"
drive_speechcommands_root = DRIVE_CACHE_ROOT / "speechcommands_v2"
speechcommands_root = (
    drive_speechcommands_root
    if DRIVE_CACHE_ACTIVE and CACHE_SPEECHCOMMANDS_ON_DRIVE
    else local_speechcommands_root
)
feature_cache_root = DRIVE_CACHE_ROOT / "hash_feature_cache"
teacher_logits_cache_root = DRIVE_CACHE_ROOT / "teacher_logits_cache"

speechcommands_root.mkdir(parents=True, exist_ok=True)
os.environ["SPEECHCOMMANDS_DATA_ROOT"] = str(speechcommands_root)

if DRIVE_CACHE_ACTIVE:
    DRIVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    feature_cache_root.mkdir(parents=True, exist_ok=True)
    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HASH_KWS_FEATURE_CACHE_ROOT"] = str(feature_cache_root)
    os.environ["HASH_KWS_TEACHER_LOGITS_CACHE_ROOT"] = str(teacher_logits_cache_root)
else:
    feature_cache_root = PROJECT_ROOT / "data" / "hash_feature_cache"
    teacher_logits_cache_root = PROJECT_ROOT / "data" / "teacher_logits_cache"
    feature_cache_root.mkdir(parents=True, exist_ok=True)
    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HASH_KWS_FEATURE_CACHE_ROOT"] = str(feature_cache_root)
    os.environ["HASH_KWS_TEACHER_LOGITS_CACHE_ROOT"] = str(teacher_logits_cache_root)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DRIVE_CACHE_ACTIVE:", DRIVE_CACHE_ACTIVE)
print("Variants to train:", ENSEMBLE_VARIANT_NAMES)
print("Teacher variant:", TEACHER_VARIANT_NAME)
print("Runtime files written/skipped:", len(created_files), "/", len(skipped_files))


In [ ]:
import json
import shutil
import time
from copy import deepcopy
from dataclasses import replace
from pathlib import Path

import numpy as np
import torch
from torchaudio.datasets import SPEECHCOMMANDS

from hash_kws_lab.config import make_experiment
from hash_kws_lab.data import ensure_torchaudio_available, prepare_dataloaders
from hash_kws_lab.export import export_model_bundle
from hash_kws_lab.models import build_student_model, build_teacher_model, summarize_model
from hash_kws_lab.reporting import (
    add_note,
    initialize_run_state,
    record_export_artifacts,
    save_history,
    save_history_plots,
    save_json_artifact,
    save_metrics,
    save_model_summary,
    save_text_artifact,
    update_stage_state,
    write_run_summary,
)
from hash_kws_lab.trainer import evaluate, load_model_checkpoint, train_student, train_teacher
from hashednet95.hashednet95_recipes import with_drive_cache_paths
from hash_ensemble.ensemble_recipes import build_ensemble_recipe_book, describe_variant
from hash_ensemble import aggregation as agg
import export_hash_kws_firmware as firmware_exporter

ensure_torchaudio_available()
torch.manual_seed(13)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(13)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
SPEECHCOMMANDS_REQUIRED_FILES = ("validation_list.txt", "testing_list.txt")
SPEECHCOMMANDS_ARCHIVE = "speech_commands_v0.02.tar.gz"
SPEECHCOMMANDS_REQUIRED_DIRS = (
    "_background_noise_", "yes", "no", "up", "down", "left", "right",
    "on", "off", "stop", "go",
)

def speechcommands_extracted_dir(root: Path) -> Path:
    return root / "SpeechCommands" / "speech_commands_v0.02"

def speechcommands_is_complete(root: Path) -> bool:
    extracted = speechcommands_extracted_dir(root)
    return (
        extracted.is_dir()
        and all((extracted / name).is_file() for name in SPEECHCOMMANDS_REQUIRED_FILES)
        and all((extracted / name).is_dir() for name in SPEECHCOMMANDS_REQUIRED_DIRS)
    )

def reset_incomplete_speechcommands(root: Path) -> None:
    if speechcommands_is_complete(root):
        return
    extracted_parent = root / "SpeechCommands"
    if extracted_parent.exists():
        print("Removing incomplete Speech Commands extraction:", extracted_parent)
        shutil.rmtree(extracted_parent)

def remove_incomplete_speechcommands_archive(root: Path) -> None:
    archive_path = root / SPEECHCOMMANDS_ARCHIVE
    if archive_path.exists():
        print("Removing incomplete Speech Commands archive:", archive_path)
        archive_path.unlink()

def ensure_speechcommands_downloaded(root: Path) -> dict:
    root.mkdir(parents=True, exist_ok=True)
    reset_incomplete_speechcommands(root)
    if not speechcommands_is_complete(root):
        print("Downloading Speech Commands v0.02 into:", root)
        try:
            SPEECHCOMMANDS(root=str(root), download=True, subset="validation")
        except Exception:
            reset_incomplete_speechcommands(root)
            remove_incomplete_speechcommands_archive(root)
            SPEECHCOMMANDS(root=str(root), download=True, subset="validation")
    if not speechcommands_is_complete(root):
        raise RuntimeError("Speech Commands extraction incomplete")
    return {
        subset: len(SPEECHCOMMANDS(root=str(root), download=False, subset=subset))
        for subset in ("training", "validation", "testing")
    }

speechcommands_counts = ensure_speechcommands_downloaded(Path(os.environ["SPEECHCOMMANDS_DATA_ROOT"]))
print("Speech Commands split sizes:", speechcommands_counts)


In [ ]:
base = make_experiment(tag="hash_kws12_iterlab_v1", vocabulary_preset="kws12")
recipes = build_ensemble_recipe_book(base)
assert TEACHER_VARIANT_NAME in recipes, f"Unknown teacher variant: {TEACHER_VARIANT_NAME}"
assert all(name in recipes for name in ENSEMBLE_VARIANT_NAMES)

if DRIVE_CACHE_ACTIVE:
    recipes = {name: with_drive_cache_paths(recipe, DRIVE_CACHE_ROOT) for name, recipe in recipes.items()}

if SMOKE_MODE:
    shrunk = {}
    for name, recipe in recipes.items():
        shrunk[name] = replace(
            recipe,
            dataset=replace(recipe.dataset, train_limit=1024, val_limit=256, test_limit=256),
            train=replace(
                recipe.train,
                teacher_epochs=1,
                student_pretrain_epochs=1,
                student_epochs=1,
                student_polish_epochs=0,
            ),
        )
    recipes = shrunk

ensemble_root = PROJECT_ROOT / "code" / "training" / "hash_artifacts" / "hash_ensemble"
ensemble_root.mkdir(parents=True, exist_ok=True)

print("Variants:")
for name, recipe in recipes.items():
    d = describe_variant(recipe)
    print(f"  {name}: tag={d['tag']}  pw={d['pointwise_codebook_sizes']}  seed={d['seed']}")


## Phase 1 — Train teacher once (anchored to the chosen variant)


In [ ]:
teacher_recipe = recipes[TEACHER_VARIANT_NAME]
teacher_run_dir = initialize_run_state(
    PROJECT_ROOT, teacher_recipe, recipe_name=f"hash_ensemble.{TEACHER_VARIANT_NAME}.teacher"
)

t0 = time.perf_counter()
teacher_bundle = prepare_dataloaders(PROJECT_ROOT, teacher_recipe, device=device)
teacher_loaders = teacher_bundle["loaders"]
save_json_artifact(teacher_run_dir, "dataset_summary.json", teacher_bundle["summary"])
print("Data prepare seconds:", round(time.perf_counter() - t0, 1))
print("Dataset summary:", json.dumps(teacher_bundle["summary"], indent=2))

teacher_model = build_teacher_model(teacher_recipe)
teacher_summary = summarize_model(teacher_model, teacher_recipe)
save_json_artifact(teacher_run_dir, "teacher_model_inventory.json", teacher_summary)
print("Teacher params:", teacher_summary["trainable_parameters"])

teacher_result = train_teacher(teacher_model, loaders=teacher_loaders, experiment=teacher_recipe, device=device)
teacher_model.load_state_dict(teacher_result["best_state"], strict=True)
teacher_test_metrics = evaluate(
    teacher_model,
    teacher_loaders["test"],
    device=device,
    top_k=teacher_recipe.train.top_k,
    use_amp=teacher_recipe.train.use_amp,
    desc="ensemble | teacher | test",
)
save_metrics(teacher_run_dir, "teacher", teacher_test_metrics)

teacher_checkpoint_path = teacher_run_dir / "teacher_best.pt"
torch.save(
    {
        "experiment": teacher_recipe.to_dict(),
        "state_dict": {k: v.detach().cpu().clone() for k, v in teacher_model.state_dict().items()},
        "result": teacher_result,
    },
    teacher_checkpoint_path,
)
print("Teacher test metrics:", teacher_test_metrics)
print("Teacher checkpoint:", teacher_checkpoint_path)


## Phase 2 — Train each student variant (teacher reused, logits cached)


In [ ]:
from hash_kws_lab.config import experiment_from_dict

student_states: dict[str, dict] = {}

for variant_name in ENSEMBLE_VARIANT_NAMES:
    recipe = recipes[variant_name]
    print(f"\n=== Training student '{variant_name}' (tag={recipe.tag}) ===")
    run_dir = initialize_run_state(PROJECT_ROOT, recipe, recipe_name=f"hash_ensemble.{variant_name}")

    # Re-use the teacher's prepared loaders if dataset/feature configs match;
    # otherwise rebuild. For the ensemble track they always match.
    if variant_name == TEACHER_VARIANT_NAME:
        bundle = teacher_bundle
    else:
        bundle = prepare_dataloaders(PROJECT_ROOT, recipe, device=device)
    save_json_artifact(run_dir, "dataset_summary.json", bundle["summary"])

    student = build_student_model(recipe)
    summary = summarize_model(student, recipe)
    save_json_artifact(run_dir, "student_model_inventory.json", summary)
    print("Student params:", {
        "compact": summary["hash_compact_parameters"],
        "virtual": summary["virtual_dense_parameters"],
        "maccs": summary["maccs_rough"],
    })

    # Re-load teacher state (the same teacher for all students; checkpoint is fixed)
    teacher_for_student = build_teacher_model(recipe)
    load_model_checkpoint(teacher_for_student, teacher_checkpoint_path, device=device)

    student_result = train_student(
        student,
        loaders=bundle["loaders"],
        experiment=recipe,
        device=device,
        teacher=teacher_for_student,
    )
    student.load_state_dict(student_result["best_state"], strict=True)

    bundle_path = run_dir / "student_best.pt"
    torch.save(
        {
            "experiment": recipe.to_dict(),
            "state_dict": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()},
            "result": {key: value for key, value in student_result.items() if key != "best_state"},
            "test_metrics": student_result["test_metrics"],
        },
        bundle_path,
    )
    save_metrics(run_dir, "student", student_result["test_metrics"])
    save_history(run_dir, "student", student_result["history"])

    # Standard library bundle + per-variant firmware export
    export_metadata = export_model_bundle(student, experiment=recipe, stage_name="student")
    variant_firmware_dir = (
        PROJECT_ROOT / "code" / "firmware" / f"hash_kws_runtime_{variant_name}"
    )
    variant_firmware_dir.mkdir(parents=True, exist_ok=True)
    firmware_export = firmware_exporter.export_bundle_to_firmware(
        bundle_path=Path(export_metadata["bundle"]["path"]),
        output_dir=variant_firmware_dir,
        project_root=PROJECT_ROOT,
        device=device,
        calibration_split="validation",
        calibration_batches=8,
    )
    record_export_artifacts(run_dir, {
        "library_bundle": export_metadata,
        "firmware_export": firmware_export,
        "firmware_output_dir": str(variant_firmware_dir),
    })

    student_states[variant_name] = {
        "recipe": recipe,
        "run_dir": run_dir,
        "bundle_path": bundle_path,
        "summary": summary,
        "test_metrics": student_result["test_metrics"],
        "firmware_dir": variant_firmware_dir,
        "stage_summaries": student_result["stage_summaries"],
    }
    print(f"--- '{variant_name}' done. test={student_result['test_metrics']}")

print("\nAll students trained.")


## Phase 3 — Stack logits, evaluate ensemble, fit calibration / learned weights


In [ ]:
@torch.no_grad()
def collect_logits_and_labels(model, loader, device):
    model.eval()
    all_logits = []
    all_labels = []
    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) >= 2:
            features, targets = batch[0], batch[1]
        else:
            raise TypeError("Unexpected batch shape")
        features = features.to(device, non_blocking=True)
        logits = model(features).detach().cpu().numpy()
        all_logits.append(logits.astype(np.float64))
        all_labels.append(targets.detach().cpu().numpy().astype(np.int64))
    return np.concatenate(all_logits, axis=0), np.concatenate(all_labels, axis=0)

anchor_recipe = recipes[TEACHER_VARIANT_NAME]
label_names = anchor_recipe.all_labels

# Re-prepare (cached) loaders with shuffle off so all 3 models see the same order
per_variant_logits_test: dict[str, np.ndarray] = {}
per_variant_logits_val: dict[str, np.ndarray] = {}
test_labels_ref = None
val_labels_ref = None

for variant_name in ENSEMBLE_VARIANT_NAMES:
    recipe = student_states[variant_name]["recipe"]
    bundle_path = student_states[variant_name]["bundle_path"]
    bundle = teacher_bundle if variant_name == TEACHER_VARIANT_NAME else prepare_dataloaders(
        PROJECT_ROOT, recipe, device=device
    )
    # Build a fresh student and load its best state
    student = build_student_model(recipe).to(device)
    payload = torch.load(bundle_path, map_location=device)
    student.load_state_dict(payload["state_dict"], strict=True)

    test_logits, test_labels = collect_logits_and_labels(student, bundle["loaders"]["test"], device)
    val_logits, val_labels = collect_logits_and_labels(student, bundle["loaders"]["validation"], device)
    per_variant_logits_test[variant_name] = test_logits
    per_variant_logits_val[variant_name] = val_logits

    if test_labels_ref is None:
        test_labels_ref = test_labels
        val_labels_ref = val_labels
    else:
        if not np.array_equal(test_labels_ref, test_labels):
            raise RuntimeError("Test labels diverged between variants — loaders not deterministic?")
        if not np.array_equal(val_labels_ref, val_labels):
            raise RuntimeError("Val labels diverged between variants — loaders not deterministic?")

test_logits_stack = np.stack([per_variant_logits_test[name] for name in ENSEMBLE_VARIANT_NAMES], axis=0)
val_logits_stack = np.stack([per_variant_logits_val[name] for name in ENSEMBLE_VARIANT_NAMES], axis=0)
print("Test logits stack:", test_logits_stack.shape, "labels:", test_labels_ref.shape)
print("Val  logits stack:", val_logits_stack.shape,  "labels:", val_labels_ref.shape)

ensemble_eval = agg.evaluate_aggregators(
    test_logits=test_logits_stack,
    test_labels=test_labels_ref,
    val_logits=val_logits_stack,
    val_labels=val_labels_ref,
    label_names=label_names,
)

per_model = {}
for name in ENSEMBLE_VARIANT_NAMES:
    preds = per_variant_logits_test[name].argmax(axis=-1)
    top1 = float((preds == test_labels_ref).mean())
    topk_idx = np.argsort(per_variant_logits_test[name], axis=-1)[..., -3:]
    top3 = float((topk_idx == test_labels_ref.reshape(-1, 1)).any(axis=-1).mean())
    per_model[name] = {
        "top1": top1,
        "top3": top3,
        "compact_params": int(student_states[name]["summary"]["hash_compact_parameters"]),
        "virtual_params": int(student_states[name]["summary"]["virtual_dense_parameters"]),
        "maccs": int(student_states[name]["summary"]["maccs_rough"]),
    }

single_top1 = np.array([per_model[n]["top1"] for n in ENSEMBLE_VARIANT_NAMES])
ensemble_eval["dispersion"] = {
    "single_mean": float(single_top1.mean()),
    "single_std": float(single_top1.std(ddof=0)),
    "single_top1_per_variant": per_model,
}

results = {
    "teacher_variant": TEACHER_VARIANT_NAME,
    "teacher_test_metrics": teacher_test_metrics,
    "per_model": per_model,
    "aggregators_test": ensemble_eval["aggregators"],
    "oracle_top1": ensemble_eval["oracle_top1"],
    "pairwise_disagreement": ensemble_eval["pairwise_disagreement"],
    "per_class_disagreement_rate": ensemble_eval["per_class_disagreement_rate"],
    "dispersion": ensemble_eval["dispersion"],
    "labels": label_names,
}
results_path = ensemble_root / "ensemble_results.json"
results_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nWrote", results_path)
print(json.dumps({
    "per_model_top1": {n: per_model[n]["top1"] for n in ENSEMBLE_VARIANT_NAMES},
    "aggregators_top1": {k: v.get("top1") for k, v in ensemble_eval["aggregators"].items()},
    "oracle_top1": ensemble_eval["oracle_top1"],
}, indent=2))


## Phase 4 — Plots and aggregator_params.h


In [ ]:
import matplotlib.pyplot as plt

agg_top1 = {k: v["top1"] for k, v in ensemble_eval["aggregators"].items() if "top1" in v}
best_single = max(per_model.values(), key=lambda x: x["top1"])["top1"]

fig, ax = plt.subplots(figsize=(10, 4))
keys = list(agg_top1.keys())
vals = [agg_top1[k] for k in keys]
ax.axhline(best_single, color="grey", linestyle="--", label=f"best single = {best_single:.4f}")
ax.axhline(ensemble_eval["oracle_top1"], color="green", linestyle=":", label=f"oracle = {ensemble_eval['oracle_top1']:.4f}")
bars = ax.bar(keys, vals, color="steelblue")
for bar, value in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.001, f"{value:.4f}",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel("test top-1 accuracy")
ax.set_title("Hash KWS ensemble — aggregator comparison")
ax.set_ylim(min(vals) * 0.985, max(vals + [best_single, ensemble_eval["oracle_top1"]]) * 1.005)
ax.legend()
plt.xticks(rotation=20)
plot_path = ensemble_root / "aggregator_comparison.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.close()
print("Wrote", plot_path)

disagree = np.array(ensemble_eval["pairwise_disagreement"])
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(disagree, cmap="magma", vmin=0)
for i in range(disagree.shape[0]):
    for j in range(disagree.shape[1]):
        ax.text(j, i, f"{disagree[i, j]:.3f}", ha="center", va="center",
                color="white" if disagree[i, j] > disagree.max() * 0.5 else "black",
                fontsize=10)
ax.set_xticks(range(len(ENSEMBLE_VARIANT_NAMES)))
ax.set_yticks(range(len(ENSEMBLE_VARIANT_NAMES)))
ax.set_xticklabels(ENSEMBLE_VARIANT_NAMES)
ax.set_yticklabels(ENSEMBLE_VARIANT_NAMES)
ax.set_title("Pairwise disagreement rate")
plt.colorbar(im, ax=ax)
plot_path2 = ensemble_root / "pairwise_disagreement.png"
plt.savefig(plot_path2, bbox_inches="tight")
plt.close()
print("Wrote", plot_path2)


In [ ]:
# Generate aggregator_params.h with calibrated temperatures + learned weights
ts_block = ensemble_eval["aggregators"].get("temperature_scaled", {})
lw_block = ensemble_eval["aggregators"].get("learned_weights", {})
temps = ts_block.get("T", [1.0, 1.0, 1.0])
weights = lw_block.get("w", [1.0 / 3.0] * 3)

header_path = ensemble_root / "aggregator_params.h"
lines = [
    "// Auto-generated by code/training/hash_ensemble notebook.",
    "// One temperature per model + softmax-normalized 3-weight ensemble head.",
    "// Both fitted on validation split only — test metrics are not leaked.",
    "// Models order: " + ", ".join(ENSEMBLE_VARIANT_NAMES),
    "",
    "#ifndef HASH_KWS_AGGREGATOR_PARAMS_H_",
    "#define HASH_KWS_AGGREGATOR_PARAMS_H_",
    "",
    "#include <stddef.h>",
    "",
    "static const size_t kHashEnsembleNumModels = 3;",
    "static const size_t kHashEnsembleNumClasses = " + str(len(label_names)) + ";",
    "",
    "static const float kHashEnsembleTemperatures[kHashEnsembleNumModels] = {",
    ", ".join(f"{float(t):.8f}f" for t in temps),
    "};",
    "",
    "static const float kHashEnsembleLearnedWeights[kHashEnsembleNumModels] = {",
    ", ".join(f"{float(w):.8f}f" for w in weights),
    "};",
    "",
    "#endif  // HASH_KWS_AGGREGATOR_PARAMS_H_",
    "",
]
header_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", header_path)
print("Temperatures:", temps)
print("Learned weights:", weights)


## Phase 5 — Bundle everything for download


In [ ]:
stage_dir = ensemble_root / "stage"
if stage_dir.exists():
    shutil.rmtree(stage_dir)
stage_dir.mkdir(parents=True)

(stage_dir / "ensemble_results.json").write_bytes((ensemble_root / "ensemble_results.json").read_bytes())
(stage_dir / "aggregator_params.h").write_bytes((ensemble_root / "aggregator_params.h").read_bytes())
(stage_dir / "aggregator_comparison.png").write_bytes((ensemble_root / "aggregator_comparison.png").read_bytes())
(stage_dir / "pairwise_disagreement.png").write_bytes((ensemble_root / "pairwise_disagreement.png").read_bytes())

for variant_name in ENSEMBLE_VARIANT_NAMES:
    vdir = stage_dir / variant_name
    vdir.mkdir()
    shutil.copy2(student_states[variant_name]["bundle_path"], vdir / "student_best.pt")
    shutil.copytree(
        student_states[variant_name]["firmware_dir"],
        vdir / "firmware_export",
        dirs_exist_ok=True,
    )

archive_path = shutil.make_archive(str(ensemble_root / "hash_ensemble_bundle"), "zip", root_dir=stage_dir)
print("Bundle archive:", archive_path)
if DRIVE_CACHE_ACTIVE:
    drive_target = DRIVE_CACHE_ROOT / "runs" / Path(archive_path).name
    drive_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive_path, drive_target)
    print("Copied to Drive:", drive_target)
